In [1]:
import pandas as pd

FILE = "FINAL_MODELING_DATASET_CPI_FORECASTING_V2.csv"

df = pd.read_csv(FILE)
df["forecast_origin_month"] = pd.to_datetime(df["forecast_origin_month"])

print("=" * 90)
print("TASK 4.1 — MISSING-DATA PATTERN FOR MODELING")
print("=" * 90)

missing_cols = [
    "Rainfall_Deviation_LPA_Pct",
    "Rainfall_Deviation_LPA_Pct_Lag1",
    "Rainfall_Lag1_Available",
    "Inflation_Expectation_3M"
]

for col in missing_cols:
    mask = df[col].isna()

    print(f"\n{col}")
    print("-" * 60)
    print("Missing rows :", mask.sum())
    print("Observed rows:", (~mask).sum())

    if mask.any():
        print("First missing:", df.loc[mask, "forecast_origin_month"].min().date())
        print("Last missing :", df.loc[mask, "forecast_origin_month"].max().date())

        print("\nMissingness by year:")
        print(
            df.loc[mask]
              .groupby(df.loc[mask, "forecast_origin_month"].dt.year)
              .size()
              .to_string()
        )

print("\n" + "=" * 90)
print("ROWS WITH ANY MISSING MODEL FEATURE")
print("=" * 90)

feature_cols = [c for c in df.columns if c not in ["forecast_origin_month",
                                                     "cpi_combined_yoy_t_plus_1"]]

any_missing = df[feature_cols].isna().any(axis=1)

print("Rows affected:", any_missing.sum())
print("Rows complete :", (~any_missing).sum())

print("\nAffected date range:")
if any_missing.any():
    print("First:", df.loc[any_missing, "forecast_origin_month"].min().date())
    print("Last :", df.loc[any_missing, "forecast_origin_month"].max().date())

TASK 4.1 — MISSING-DATA PATTERN FOR MODELING

Rainfall_Deviation_LPA_Pct
------------------------------------------------------------
Missing rows : 100
Observed rows: 48
First missing: 2013-12-01
Last missing : 2026-03-01

Missingness by year:
forecast_origin_month
2013    1
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
2025    8
2026    3

Rainfall_Deviation_LPA_Pct_Lag1
------------------------------------------------------------
Missing rows : 100
Observed rows: 48
First missing: 2013-12-01
Last missing : 2026-03-01

Missingness by year:
forecast_origin_month
2013    1
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
2025    8
2026    3

Rainfall_Lag1_Available
------------------------------------------------------------
Missing rows : 5
Observed rows: 143
First missing: 2025-11-01
Last missing : 2026-03-01

Missingness by year:
forecast_origin_month
2025    

In [3]:
import pandas as pd

FILE = "FINAL_MODELING_DATASET_CPI_FORECASTING_V2.csv"

df = pd.read_csv(FILE)
df["forecast_origin_month"] = pd.to_datetime(df["forecast_origin_month"])

# Preserve the original validated availability feature
original_lag1_available = df["Rainfall_Lag1_Available"].copy()

# New explicit availability indicators
df["Rainfall_Current_Available"] = (
    df["Rainfall_Deviation_LPA_Pct"].notna().astype(int)
)

df["Rainfall_Lag1_Value_Available"] = (
    df["Rainfall_Deviation_LPA_Pct_Lag1"].notna().astype(int)
)

# Structural missingness -> zero, with availability indicators retained
df["Rainfall_Deviation_LPA_Pct"] = (
    df["Rainfall_Deviation_LPA_Pct"].fillna(0)
)

df["Rainfall_Deviation_LPA_Pct_Lag1"] = (
    df["Rainfall_Deviation_LPA_Pct_Lag1"].fillna(0)
)

print("=" * 90)
print("TASK 4.1 — CORRECTED MISSING-DATA STRATEGY")
print("=" * 90)

print("\nOriginal Rainfall_Lag1_Available:")
print(df["Rainfall_Lag1_Available"].value_counts(dropna=False).sort_index())

print("\nNew Rainfall_Current_Available:")
print(df["Rainfall_Current_Available"].value_counts().sort_index())

print("\nNew Rainfall_Lag1_Value_Available:")
print(df["Rainfall_Lag1_Value_Available"].value_counts().sort_index())

print("\nRemaining missing values:")
missing = df.isna().sum()
print(missing[missing > 0].to_string())

print("\nDataset shape:", df.shape)

print("\nOriginal availability preserved:",
      df["Rainfall_Lag1_Available"].isna().sum() == 5)

TASK 4.1 — CORRECTED MISSING-DATA STRATEGY

Original Rainfall_Lag1_Available:
Rainfall_Lag1_Available
0.0    95
1.0    48
NaN     5
Name: count, dtype: int64

New Rainfall_Current_Available:
Rainfall_Current_Available
0    100
1     48
Name: count, dtype: int64

New Rainfall_Lag1_Value_Available:
Rainfall_Lag1_Value_Available
0    100
1     48
Name: count, dtype: int64

Remaining missing values:
Rainfall_Lag1_Available      5
Inflation_Expectation_3M    88

Dataset shape: (148, 18)

Original availability preserved: True


In [4]:
# =============================================================================
# TASK 4.2 — FEATURE ENGINEERING
# STEP 1: CPI HISTORY FEATURES
# =============================================================================

import pandas as pd
import numpy as np

# df is the corrected dataframe from Task 4.1
df = df.sort_values("forecast_origin_month").reset_index(drop=True)

# Target column
TARGET = "cpi_combined_yoy_t_plus_1"

# Recover the CPI series corresponding to forecast origin t.
# Since the target is CPI at t+1, shift the target backward by one month
# to reconstruct the CPI value known at forecast origin t.
df["CPI_t"] = df[TARGET].shift(1)

# Past CPI features
df["CPI_Lag1"] = df["CPI_t"].shift(1)
df["CPI_Lag2"] = df["CPI_t"].shift(2)

# 3-month historical CPI average
df["CPI_RollingMean_3M"] = (
    df["CPI_t"]
    .rolling(window=3, min_periods=3)
    .mean()
)

# Recent CPI momentum
df["CPI_Momentum_1M"] = (
    df["CPI_t"] - df["CPI_Lag1"]
)

print("=" * 90)
print("TASK 4.2 — CPI HISTORY FEATURES CREATED")
print("=" * 90)

new_features = [
    "CPI_t",
    "CPI_Lag1",
    "CPI_Lag2",
    "CPI_RollingMean_3M",
    "CPI_Momentum_1M"
]

print("\nNew features:")
print(new_features)

print("\nMissing values introduced:")
print(df[new_features].isna().sum().to_string())

print("\nFeature preview:")
print(
    df[
        ["forecast_origin_month", TARGET] + new_features
    ].head(10).to_string(index=False)
)

print("\nDataset shape:", df.shape)

TASK 4.2 — CPI HISTORY FEATURES CREATED

New features:
['CPI_t', 'CPI_Lag1', 'CPI_Lag2', 'CPI_RollingMean_3M', 'CPI_Momentum_1M']

Missing values introduced:
CPI_t                 1
CPI_Lag1              2
CPI_Lag2              3
CPI_RollingMean_3M    3
CPI_Momentum_1M       2

Feature preview:
forecast_origin_month  cpi_combined_yoy_t_plus_1    CPI_t  CPI_Lag1  CPI_Lag2  CPI_RollingMean_3M  CPI_Momentum_1M
           2013-12-01                   8.604207      NaN       NaN       NaN                 NaN              NaN
           2014-01-01                   7.882241 8.604207       NaN       NaN                 NaN              NaN
           2014-02-01                   8.246445 7.882241  8.604207       NaN                 NaN        -0.721965
           2014-03-01                   8.482564 8.246445  7.882241  8.604207            8.244298         0.364204
           2014-04-01                   8.325538 8.482564  8.246445  7.882241            8.203750         0.236118
           201

In [5]:
# =============================================================================
# TASK 4.2 — FEATURE ENGINEERING
# STEP 2: CYCLICAL MONTH FEATURES
# =============================================================================

import numpy as np

# Extract calendar month from forecast origin
month = df["forecast_origin_month"].dt.month

# Cyclical encoding of month
df["Month_Sin"] = np.sin(2 * np.pi * month / 12)
df["Month_Cos"] = np.cos(2 * np.pi * month / 12)

print("=" * 90)
print("TASK 4.2 — CYCLICAL MONTH FEATURES CREATED")
print("=" * 90)

print("\nNew features:")
print([
    "Month_Sin",
    "Month_Cos"
])

print("\nFeature range:")
print(
    df[
        ["Month_Sin", "Month_Cos"]
    ].agg(["min", "max"]).to_string()
)

print("\nMonth encoding preview:")
print(
    df[
        [
            "forecast_origin_month",
            "Month_Sin",
            "Month_Cos"
        ]
    ].head(15).to_string(index=False)
)

print("\nMissing values:")
print(
    df[
        ["Month_Sin", "Month_Cos"]
    ].isna().sum().to_string()
)

print("\nDataset shape:", df.shape)

TASK 4.2 — CYCLICAL MONTH FEATURES CREATED

New features:
['Month_Sin', 'Month_Cos']

Feature range:
     Month_Sin  Month_Cos
min       -1.0       -1.0
max        1.0        1.0

Month encoding preview:
forecast_origin_month     Month_Sin     Month_Cos
           2013-12-01 -2.449294e-16  1.000000e+00
           2014-01-01  5.000000e-01  8.660254e-01
           2014-02-01  8.660254e-01  5.000000e-01
           2014-03-01  1.000000e+00  6.123234e-17
           2014-04-01  8.660254e-01 -5.000000e-01
           2014-05-01  5.000000e-01 -8.660254e-01
           2014-06-01  1.224647e-16 -1.000000e+00
           2014-07-01 -5.000000e-01 -8.660254e-01
           2014-08-01 -8.660254e-01 -5.000000e-01
           2014-09-01 -1.000000e+00 -1.836970e-16
           2014-10-01 -8.660254e-01  5.000000e-01
           2014-11-01 -5.000000e-01  8.660254e-01
           2014-12-01 -2.449294e-16  1.000000e+00
           2015-01-01  5.000000e-01  8.660254e-01
           2015-02-01  8.660254e-01  5.000000e

In [6]:
# =============================================================================
# TASK 4.2 — FEATURE ENGINEERING
# STEP 3: CURRENT FEATURE INVENTORY
# =============================================================================

print("=" * 90)
print("TASK 4.2 — CURRENT MODELING FEATURE INVENTORY")
print("=" * 90)

print(f"\nTotal columns: {len(df.columns)}")

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n" + "-" * 90)
print("NEW FEATURES CREATED IN TASK 4.2")
print("-" * 90)

task42_features = [
    "CPI_t",
    "CPI_Lag1",
    "CPI_Lag2",
    "CPI_RollingMean_3M",
    "CPI_Momentum_1M",
    "Month_Sin",
    "Month_Cos"
]

for col in task42_features:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

TASK 4.2 — CURRENT MODELING FEATURE INVENTORY

Total columns: 25
01. forecast_origin_month
02. cpi_combined_yoy_t_plus_1
03. Petrol_Monthly_Avg_Rs_per_Litre
04. Diesel_Monthly_Avg_Rs_per_Litre
05. Petrol_MoM_Change_Pct
06. Diesel_MoM_Change_Pct
07. Brent_Monthly_Avg_USD_per_Barrel
08. Brent_Log_MoM
09. Rainfall_Deviation_LPA_Pct
10. Rainfall_Deviation_LPA_Pct_Lag1
11. Rainfall_Lag1_Available
12. WPI_All_Commodities_t_minus_1
13. WPI_MoM_Pct_t_minus_1
14. FAO_Food_Price_Index_t_minus_1
15. FAO_Food_Price_Index_MoM_t_minus_1
16. Inflation_Expectation_3M
17. Rainfall_Current_Available
18. Rainfall_Lag1_Value_Available
19. CPI_t
20. CPI_Lag1
21. CPI_Lag2
22. CPI_RollingMean_3M
23. CPI_Momentum_1M
24. Month_Sin
25. Month_Cos

------------------------------------------------------------------------------------------
NEW FEATURES CREATED IN TASK 4.2
------------------------------------------------------------------------------------------
CPI_t: FOUND
CPI_Lag1: FOUND
CPI_Lag2: FOUND
CPI_Rolli

In [7]:
# =============================================================================
# TASK 4.3 — TEMPORAL SPLIT
# STEP 1: DEFINE CHRONOLOGICAL SPLIT
# =============================================================================

print("=" * 90)
print("TASK 4.3 — TEMPORAL SPLIT")
print("=" * 90)

# Chronological order
df = df.sort_values("forecast_origin_month").reset_index(drop=True)

n = len(df)

# 70% train / 15% validation / 15% test
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("\nSplit strategy: 70% / 15% / 15% chronological")

print("\nTRAIN")
print("Rows :", len(train))
print("Start:", train["forecast_origin_month"].min().date())
print("End  :", train["forecast_origin_month"].max().date())

print("\nVALIDATION")
print("Rows :", len(validation))
print("Start:", validation["forecast_origin_month"].min().date())
print("End  :", validation["forecast_origin_month"].max().date())

print("\nTEST")
print("Rows :", len(test))
print("Start:", test["forecast_origin_month"].min().date())
print("End  :", test["forecast_origin_month"].max().date())

print("\n" + "-" * 90)
print("SPLIT CHECKS")
print("-" * 90)

print("Total rows:", len(train) + len(validation) + len(test))
print("Original rows:", n)

print("\nChronological order:")
print(
    train["forecast_origin_month"].max()
    < validation["forecast_origin_month"].min()
    < test["forecast_origin_month"].min()
)

print("\nRows:")
print(
    f"Train={len(train)}, "
    f"Validation={len(validation)}, "
    f"Test={len(test)}"
)

TASK 4.3 — TEMPORAL SPLIT

Split strategy: 70% / 15% / 15% chronological

TRAIN
Rows : 103
Start: 2013-12-01
End  : 2022-06-01

VALIDATION
Rows : 22
Start: 2022-07-01
End  : 2024-04-01

TEST
Rows : 23
Start: 2024-05-01
End  : 2026-03-01

------------------------------------------------------------------------------------------
SPLIT CHECKS
------------------------------------------------------------------------------------------
Total rows: 148
Original rows: 148

Chronological order:
True

Rows:
Train=103, Validation=22, Test=23


In [9]:
# =============================================================================
# TASK 4.4 — BASELINES
# STEP 1: CREATE BASELINE PREDICTIONS
# =============================================================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "cpi_combined_yoy_t_plus_1"

# Ensure chronological order
df = df.sort_values("forecast_origin_month").reset_index(drop=True)

# ------------------------------------------------------------------
# Baseline 1: Naive
# Predict next month's CPI using current month's CPI
# ------------------------------------------------------------------
df["Baseline_Naive"] = df["CPI_t"]

# ------------------------------------------------------------------
# Baseline 2: Seasonal Naive
# Predict next month's CPI using CPI from 12 months earlier
# ------------------------------------------------------------------
df["Baseline_SeasonalNaive"] = df["CPI_t"].shift(12)

# ------------------------------------------------------------------
# Baseline 3: 3-month historical mean
# ------------------------------------------------------------------
df["Baseline_3M_Mean"] = (
    df["CPI_t"]
    .rolling(window=3, min_periods=3)
    .mean()
)

# ------------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------------
def evaluate_baseline(data, prediction_col):

    valid = data[[TARGET, prediction_col]].dropna()

    y_true = valid[TARGET]
    y_pred = valid[prediction_col]

    return {
        "Rows": len(valid),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

# ------------------------------------------------------------------
# Evaluate on validation and test
# ------------------------------------------------------------------

validation = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

baselines = [
    "Baseline_Naive",
    "Baseline_SeasonalNaive",
    "Baseline_3M_Mean"
]

results = []

for name in baselines:

    val_result = evaluate_baseline(validation, name)
    test_result = evaluate_baseline(test, name)

    results.append({
        "Baseline": name,
        "Validation_MAE": val_result["MAE"],
        "Validation_RMSE": val_result["RMSE"],
        "Validation_R2": val_result["R2"],
        "Test_MAE": test_result["MAE"],
        "Test_RMSE": test_result["RMSE"],
        "Test_R2": test_result["R2"]
    })

baseline_results = pd.DataFrame(results)

print("=" * 110)
print("TASK 4.4 — BASELINE PERFORMANCE")
print("=" * 110)

print(
    baseline_results.round(4).to_string(index=False)
)

print("\n" + "-" * 110)
print("BASELINE INTERPRETATION")
print("-" * 110)

best_val = baseline_results.loc[
    baseline_results["Validation_RMSE"].idxmin()
]

best_test = baseline_results.loc[
    baseline_results["Test_RMSE"].idxmin()
]

print(
    f"Best validation baseline by RMSE: "
    f"{best_val['Baseline']} "
    f"(RMSE={best_val['Validation_RMSE']:.4f})"
)

print(
    f"Best test baseline by RMSE: "
    f"{best_test['Baseline']} "
    f"(RMSE={best_test['Test_RMSE']:.4f})"
)

TASK 4.4 — BASELINE PERFORMANCE
              Baseline  Validation_MAE  Validation_RMSE  Validation_R2  Test_MAE  Test_RMSE  Test_R2
        Baseline_Naive          0.5832           0.8344         0.1993    0.6299     0.7796   0.7489
Baseline_SeasonalNaive          1.3089           1.5912        -1.9125    2.0145     2.4945  -1.5711
      Baseline_3M_Mean          0.7346           0.9966        -0.1425    0.9407     1.0793   0.5187

--------------------------------------------------------------------------------------------------------------
BASELINE INTERPRETATION
--------------------------------------------------------------------------------------------------------------
Best validation baseline by RMSE: Baseline_Naive (RMSE=0.8344)
Best test baseline by RMSE: Baseline_Naive (RMSE=0.7796)


In [10]:
# ============================================================
# TASK 5.1 — MISSINGNESS & SPLIT AUDIT
# ============================================================

import pandas as pd
import numpy as np

FILE_PATH = "/content/FINAL_MODELING_DATASET_CPI_FORECASTING_V2.csv"

df = pd.read_csv(FILE_PATH)

# Parse date
df["forecast_origin_month"] = pd.to_datetime(df["forecast_origin_month"])

# Fixed chronological split from Task 4
train = df.iloc[:103].copy()
val   = df.iloc[103:125].copy()
test  = df.iloc[125:148].copy()

print("=" * 90)
print("TASK 5.1 — MISSINGNESS & SPLIT AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Dataset shape
# ------------------------------------------------------------
print("\nDATASET")
print("-" * 90)
print("Shape:", df.shape)

# ------------------------------------------------------------
# 2. Split boundaries
# ------------------------------------------------------------
print("\nSPLIT DATE RANGES")
print("-" * 90)

for name, part in [
    ("TRAIN", train),
    ("VALIDATION", val),
    ("TEST", test)
]:
    print(
        f"{name:12s}: "
        f"{part['forecast_origin_month'].min().date()} "
        f"to "
        f"{part['forecast_origin_month'].max().date()} "
        f"| n = {len(part)}"
    )

# ------------------------------------------------------------
# 3. Missingness by split
# ------------------------------------------------------------
print("\nMISSINGNESS BY SPLIT")
print("-" * 90)

missing = pd.DataFrame({
    "Train": train.isna().sum(),
    "Validation": val.isna().sum(),
    "Test": test.isna().sum(),
})

missing["Total"] = missing.sum(axis=1)

display(
    missing[missing["Total"] > 0]
    .sort_values("Total", ascending=False)
)

# ------------------------------------------------------------
# 4. Target distribution
# ------------------------------------------------------------
TARGET = "cpi_combined_yoy_t_plus_1"

print("\nTARGET DISTRIBUTION BY SPLIT")
print("-" * 90)

for name, part in [
    ("TRAIN", train),
    ("VALIDATION", val),
    ("TEST", test)
]:
    y = part[TARGET]

    print(
        f"{name:12s}: "
        f"mean={y.mean():.3f}, "
        f"std={y.std():.3f}, "
        f"min={y.min():.3f}, "
        f"max={y.max():.3f}"
    )

# ------------------------------------------------------------
# 5. Inspect rainfall-related columns
# ------------------------------------------------------------
rain_cols = [
    col for col in df.columns
    if "Rainfall" in col or "rainfall" in col
]

print("\nRAINFALL-RELATED COLUMNS")
print("-" * 90)

for col in rain_cols:
    print(f"\n{col}")
    print(df[col].describe())
    print("Missing:", df[col].isna().sum())

# ------------------------------------------------------------
# 6. Inspect the five suspected trailing NaNs
# ------------------------------------------------------------
flag_col = "Rainfall_Lag1_Available"

if flag_col in df.columns:
    print("\nRAINFALL_LAG1_AVAILABLE — LAST 15 ROWS")
    print("-" * 90)

    display(
        df[
            [
                "forecast_origin_month",
                "Rainfall_Deviation_LPA_Pct",
                "Rainfall_Deviation_LPA_Pct_Lag1",
                "Rainfall_Lag1_Available"
            ]
        ].tail(15)
    )

# ------------------------------------------------------------
# 7. Exact rows with missing rainfall flag
# ------------------------------------------------------------
if flag_col in df.columns:
    print("\nROWS WITH MISSING RAINFALL AVAILABILITY FLAG")
    print("-" * 90)

    display(
        df[
            df[flag_col].isna()
        ][
            [
                "forecast_origin_month",
                "Rainfall_Deviation_LPA_Pct",
                "Rainfall_Deviation_LPA_Pct_Lag1",
                "Rainfall_Lag1_Available"
            ]
        ]
    )

print("\n" + "=" * 90)
print("END OF TASK 5.1")
print("=" * 90)

TASK 5.1 — MISSINGNESS & SPLIT AUDIT

DATASET
------------------------------------------------------------------------------------------
Shape: (148, 16)

SPLIT DATE RANGES
------------------------------------------------------------------------------------------
TRAIN       : 2013-12-01 to 2022-06-01 | n = 103
VALIDATION  : 2022-07-01 to 2024-04-01 | n = 22
TEST        : 2024-05-01 to 2026-03-01 | n = 23

MISSINGNESS BY SPLIT
------------------------------------------------------------------------------------------


,Train,Validation,Test,Total
Rainfall_Deviation_LPA_Pct,70,15,15,100
Rainfall_Deviation_LPA_Pct_Lag1,71,14,15,100
Inflation_Expectation_3M,66,11,11,88
Rainfall_Lag1_Available,0,0,5,5



TARGET DISTRIBUTION BY SPLIT
------------------------------------------------------------------------------------------
TRAIN       : mean=5.047, std=1.625, min=1.460, max=8.604
VALIDATION  : mean=5.698, std=0.954, min=4.310, max=7.439
TEST        : mean=3.229, std=1.591, min=0.254, max=6.206

RAINFALL-RELATED COLUMNS
------------------------------------------------------------------------------------------

Rainfall_Deviation_LPA_Pct
count    48.000000
mean      0.662869
std      18.020448
min     -44.300000
25%      -8.425000
50%       3.538209
75%      12.033229
max      55.000000
Name: Rainfall_Deviation_LPA_Pct, dtype: float64
Missing: 100

Rainfall_Deviation_LPA_Pct_Lag1
count    48.000000
mean      0.662869
std      18.020448
min     -44.300000
25%      -8.425000
50%       3.538209
75%      12.033229
max      55.000000
Name: Rainfall_Deviation_LPA_Pct_Lag1, dtype: float64
Missing: 100

Rainfall_Lag1_Available
count    143.000000
mean       0.335664
std        0.473882
min      

,forecast_origin_month,Rainfall_Deviation_LPA_Pct,Rainfall_Deviation_LPA_Pct_Lag1,Rainfall_Lag1_Available
133,2025-01-01,NaN,NaN,0.0
134,2025-02-01,NaN,NaN,0.0
135,2025-03-01,NaN,NaN,0.0
136,2025-04-01,NaN,NaN,0.0
137,2025-05-01,NaN,NaN,0.0
138,2025-06-01,9.0,NaN,0.0
139,2025-07-01,4.8,9.0,1.0
140,2025-08-01,5.1,4.8,1.0
141,2025-09-01,15.3,5.1,1.0
142,2025-10-01,NaN,15.3,1.0



ROWS WITH MISSING RAINFALL AVAILABILITY FLAG
------------------------------------------------------------------------------------------


,forecast_origin_month,Rainfall_Deviation_LPA_Pct,Rainfall_Deviation_LPA_Pct_Lag1,Rainfall_Lag1_Available
143,2025-11-01,NaN,NaN,NaN
144,2025-12-01,NaN,NaN,NaN
145,2026-01-01,NaN,NaN,NaN
146,2026-02-01,NaN,NaN,NaN
147,2026-03-01,NaN,NaN,NaN



END OF TASK 5.1


In [11]:
# ============================================================
# TASK 5.2 — MODELING FEATURE INVENTORY
# ============================================================

TARGET = "cpi_combined_yoy_t_plus_1"
DATE_COL = "forecast_origin_month"

print("=" * 90)
print("TASK 5.2 — MODELING FEATURE INVENTORY")
print("=" * 90)

feature_cols = [
    col for col in df.columns
    if col not in [TARGET, DATE_COL]
]

print(f"\nTotal columns       : {len(df.columns)}")
print(f"Date column         : {DATE_COL}")
print(f"Target column       : {TARGET}")
print(f"Predictor columns   : {len(feature_cols)}")

print("\nPREDICTOR INVENTORY")
print("-" * 90)

for i, col in enumerate(feature_cols, 1):
    print(
        f"{i:02d}. {col:45s} "
        f"dtype={str(df[col].dtype):10s} "
        f"missing={df[col].isna().sum():3d}"
    )

print("\nTARGET CHECK")
print("-" * 90)
print("Target missing:", df[TARGET].isna().sum())
print("Target dtype  :", df[TARGET].dtype)

print("\n" + "=" * 90)
print("END OF TASK 5.2")
print("=" * 90)

TASK 5.2 — MODELING FEATURE INVENTORY

Total columns       : 16
Date column         : forecast_origin_month
Target column       : cpi_combined_yoy_t_plus_1
Predictor columns   : 14

PREDICTOR INVENTORY
------------------------------------------------------------------------------------------
01. Petrol_Monthly_Avg_Rs_per_Litre               dtype=float64    missing=  0
02. Diesel_Monthly_Avg_Rs_per_Litre               dtype=float64    missing=  0
03. Petrol_MoM_Change_Pct                         dtype=float64    missing=  0
04. Diesel_MoM_Change_Pct                         dtype=float64    missing=  0
05. Brent_Monthly_Avg_USD_per_Barrel              dtype=float64    missing=  0
06. Brent_Log_MoM                                 dtype=float64    missing=  0
07. Rainfall_Deviation_LPA_Pct                    dtype=float64    missing=100
08. Rainfall_Deviation_LPA_Pct_Lag1               dtype=float64    missing=100
09. Rainfall_Lag1_Available                       dtype=float64    missing=

In [12]:
# ============================================================
# TASK 5.3 — LEAKAGE-SAFE MODELING DATA SETUP
# ============================================================

TARGET = "cpi_combined_yoy_t_plus_1"
DATE_COL = "forecast_origin_month"

# Predictor columns: all columns except date and target
FEATURE_COLS = [
    col for col in df.columns
    if col not in [DATE_COL, TARGET]
]

# ------------------------------------------------------------
# Create modeling copy
# ------------------------------------------------------------
model_df = df.copy()

# The availability flag is binary:
# NaN here means the lagged rainfall observation is unavailable.
model_df["Rainfall_Lag1_Available"] = (
    model_df["Rainfall_Lag1_Available"]
    .fillna(0)
)

# ------------------------------------------------------------
# Fixed chronological split
# ------------------------------------------------------------
X = model_df[FEATURE_COLS].copy()
y = model_df[TARGET].copy()

X_train = X.iloc[:103].copy()
X_val   = X.iloc[103:125].copy()
X_test  = X.iloc[125:148].copy()

y_train = y.iloc[:103].copy()
y_val   = y.iloc[103:125].copy()
y_test  = y.iloc[125:148].copy()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("=" * 90)
print("TASK 5.3 — LEAKAGE-SAFE MODELING DATA SETUP")
print("=" * 90)

print("\nShapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nMissing values after availability-flag correction:")
print(
    X_train.isna().sum()
    .loc[lambda s: s > 0]
)

print("\nValidation missing values:")
print(
    X_val.isna().sum()
    .loc[lambda s: s > 0]
)

print("\nTest missing values:")
print(
    X_test.isna().sum()
    .loc[lambda s: s > 0]
)

print("\nRainfall availability flag:")
print(
    model_df["Rainfall_Lag1_Available"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nOriginal dataframe unchanged:", df["Rainfall_Lag1_Available"].isna().sum(), "NaNs")
print("Modeling copy flag NaNs     :", model_df["Rainfall_Lag1_Available"].isna().sum())

print("\n" + "=" * 90)
print("END OF TASK 5.3")
print("=" * 90)

TASK 5.3 — LEAKAGE-SAFE MODELING DATA SETUP

Shapes:
X_train: (103, 14)
y_train: (103,)
X_val  : (22, 14)
y_val  : (22,)
X_test : (23, 14)
y_test : (23,)

Missing values after availability-flag correction:
Rainfall_Deviation_LPA_Pct         70
Rainfall_Deviation_LPA_Pct_Lag1    71
Inflation_Expectation_3M           66
dtype: int64

Validation missing values:
Rainfall_Deviation_LPA_Pct         15
Rainfall_Deviation_LPA_Pct_Lag1    14
Inflation_Expectation_3M           11
dtype: int64

Test missing values:
Rainfall_Deviation_LPA_Pct         15
Rainfall_Deviation_LPA_Pct_Lag1    15
Inflation_Expectation_3M           11
dtype: int64

Rainfall availability flag:
Rainfall_Lag1_Available
0.0    100
1.0     48
Name: count, dtype: int64

Original dataframe unchanged: 5 NaNs
Modeling copy flag NaNs     : 0

END OF TASK 5.3


In [13]:
# ============================================================
# TASK 5.4 — ML PREPROCESSING & EVALUATION SETUP
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("=" * 90)
print("TASK 5.4 — ML PREPROCESSING & EVALUATION SETUP")
print("=" * 90)

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------
def evaluate_model(model, X_data, y_data):
    predictions = model.predict(X_data)

    rmse = np.sqrt(mean_squared_error(y_data, predictions))
    mae = mean_absolute_error(y_data, predictions)
    r2 = r2_score(y_data, predictions)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


# ------------------------------------------------------------
# Linear-model preprocessing
# ------------------------------------------------------------
linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# ------------------------------------------------------------
# First-round candidate models
# ------------------------------------------------------------

models = {

    "Linear Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("preprocessor", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", Ridge(alpha=1.0))
    ]),

    "ElasticNet": Pipeline([
        ("preprocessor", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=7,
        min_samples_leaf=8,
        l2_regularization=1.0,
        random_state=42
    )
}

print("\nCandidate models:")
for i, name in enumerate(models.keys(), 1):
    print(f"{i}. {name}")

print("\nPreprocessing:")
print("- Linear / Ridge / ElasticNet: median imputation + StandardScaler")
print("- Random Forest: median imputation")
print("- HistGradientBoosting: native missing-value handling")

print("\nLeakage protection:")
print("- All preprocessing is fitted inside sklearn pipelines.")
print("- Preprocessing will be learned from training data only.")
print("- Validation and test data remain untouched.")

print("\n" + "=" * 90)
print("END OF TASK 5.4")
print("=" * 90)

TASK 5.4 — ML PREPROCESSING & EVALUATION SETUP

Candidate models:
1. Linear Regression
2. Ridge
3. ElasticNet
4. Random Forest
5. HistGradientBoosting

Preprocessing:
- Linear / Ridge / ElasticNet: median imputation + StandardScaler
- Random Forest: median imputation
- HistGradientBoosting: native missing-value handling

Leakage protection:
- All preprocessing is fitted inside sklearn pipelines.
- Preprocessing will be learned from training data only.
- Validation and test data remain untouched.

END OF TASK 5.4


In [14]:
# ============================================================
# TASK 5.5 — FIRST ML TRAINING ROUND
# ============================================================

print("=" * 90)
print("TASK 5.5 — FIRST ML TRAINING ROUND")
print("=" * 90)

results = []

for name, model in models.items():

    print(f"\nTraining: {name}")

    # Train ONLY on the training period
    model.fit(X_train, y_train)

    # Evaluate ONLY on validation period
    metrics = evaluate_model(model, X_val, y_val)

    results.append({
        "Model": name,
        "Validation_RMSE": metrics["RMSE"],
        "Validation_MAE": metrics["MAE"],
        "Validation_R2": metrics["R2"]
    })

    print(
        f"Validation RMSE: {metrics['RMSE']:.4f} | "
        f"MAE: {metrics['MAE']:.4f} | "
        f"R²: {metrics['R2']:.4f}"
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("Validation_RMSE")
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("FIRST-ROUND VALIDATION RESULTS")
print("=" * 90)

display(results_df)

print("\nReference benchmark:")
print("Naive baseline Test RMSE: 0.7796")
print("Naive baseline Test R²  : 0.7489")

print("\nNOTE:")
print("- Model selection is based ONLY on validation performance.")
print("- Test set has NOT been used.")
print("- No hyperparameter tuning has been performed yet.")

print("\n" + "=" * 90)
print("END OF TASK 5.5")
print("=" * 90)

TASK 5.5 — FIRST ML TRAINING ROUND

Training: Linear Regression
Validation RMSE: 0.9012 | MAE: 0.6575 | R²: 0.0658

Training: Ridge
Validation RMSE: 0.9204 | MAE: 0.6755 | R²: 0.0256

Training: ElasticNet
Validation RMSE: 0.8752 | MAE: 0.6139 | R²: 0.1190

Training: Random Forest
Validation RMSE: 0.8590 | MAE: 0.7191 | R²: 0.1512

Training: HistGradientBoosting
Validation RMSE: 0.9015 | MAE: 0.7107 | R²: 0.0651

FIRST-ROUND VALIDATION RESULTS


,Model,Validation_RMSE,Validation_MAE,Validation_R2
0,Random Forest,0.859032,0.719133,0.151188
1,ElasticNet,0.875178,0.613932,0.118979
2,Linear Regression,0.901184,0.657521,0.065842
3,HistGradientBoosting,0.901523,0.710705,0.065140
4,Ridge,0.920410,0.675530,0.025560



Reference benchmark:
Naive baseline Test RMSE: 0.7796
Naive baseline Test R²  : 0.7489

NOTE:
- Model selection is based ONLY on validation performance.
- Test set has NOT been used.
- No hyperparameter tuning has been performed yet.

END OF TASK 5.5


In [15]:
# ============================================================
# TASK 5.6 — EXPANDING-WINDOW ROBUSTNESS VALIDATION
# ============================================================

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

print("=" * 90)
print("TASK 5.6 — EXPANDING-WINDOW ROBUSTNESS VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# IMPORTANT:
# Only observations 0:125 are used here.
# Final test set (125:148) remains untouched.
# ------------------------------------------------------------

X_cv = X.iloc[:125].copy()
y_cv = y.iloc[:125].copy()

# Models selected from Task 5.5
candidate_models = {
    "Random Forest": models["Random Forest"],
    "ElasticNet": models["ElasticNet"]
}

# Expanding-window evaluation:
# train on all observations before the validation block,
# predict the next 6 months.
initial_train_size = 60
validation_horizon = 6

fold_results = []

fold_number = 0

for train_end in range(
    initial_train_size,
    len(X_cv) - validation_horizon + 1,
    validation_horizon
):

    train_start = 0
    val_start = train_end
    val_end = train_end + validation_horizon

    X_tr = X_cv.iloc[train_start:train_end]
    y_tr = y_cv.iloc[train_start:train_end]

    X_va = X_cv.iloc[val_start:val_end]
    y_va = y_cv.iloc[val_start:val_end]

    fold_number += 1

    # --------------------------------------------------------
    # Naive forecast:
    # predict each month using the immediately preceding
    # observed CPI value.
    # --------------------------------------------------------
    naive_predictions = y_cv.iloc[val_start - 1:val_end - 1].values

    naive_rmse = np.sqrt(
        mean_squared_error(y_va, naive_predictions)
    )

    naive_mae = mean_absolute_error(
        y_va, naive_predictions
    )

    naive_r2 = r2_score(
        y_va, naive_predictions
    )

    fold_results.append({
        "Fold": fold_number,
        "Train_End_Index": train_end - 1,
        "Validation_Start_Index": val_start,
        "Validation_End_Index": val_end - 1,
        "Model": "Naive",
        "RMSE": naive_rmse,
        "MAE": naive_mae,
        "R2": naive_r2
    })

    # --------------------------------------------------------
    # ML models
    # --------------------------------------------------------
    for name, base_model in candidate_models.items():

        model = clone(base_model)

        model.fit(X_tr, y_tr)

        predictions = model.predict(X_va)

        rmse = np.sqrt(
            mean_squared_error(y_va, predictions)
        )

        mae = mean_absolute_error(
            y_va, predictions
        )

        r2 = r2_score(
            y_va, predictions
        )

        fold_results.append({
            "Fold": fold_number,
            "Train_End_Index": train_end - 1,
            "Validation_Start_Index": val_start,
            "Validation_End_Index": val_end - 1,
            "Model": name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

results_cv = pd.DataFrame(fold_results)

# ------------------------------------------------------------
# Display fold-level results
# ------------------------------------------------------------

print("\nFOLD-LEVEL RESULTS")
print("-" * 90)

display(
    results_cv.round(4)
)

# ------------------------------------------------------------
# Aggregate results
# ------------------------------------------------------------

summary_cv = (
    results_cv
    .groupby("Model")
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        Median_RMSE=("RMSE", "median"),
        Std_RMSE=("RMSE", "std"),
        Mean_MAE=("MAE", "mean"),
        Mean_R2=("R2", "mean"),
        Folds=("RMSE", "count")
    )
    .sort_values("Mean_RMSE")
    .reset_index()
)

print("\nEXPANDING-WINDOW SUMMARY")
print("-" * 90)

display(
    summary_cv.round(4)
)

print("\nIMPORTANT:")
print("- Final test observations (125:148) were NOT used.")
print("- No hyperparameter tuning was performed.")
print("- This is a robustness check, not final model selection.")

print("\n" + "=" * 90)
print("END OF TASK 5.6")
print("=" * 90)

TASK 5.6 — EXPANDING-WINDOW ROBUSTNESS VALIDATION

FOLD-LEVEL RESULTS
------------------------------------------------------------------------------------------


,Fold,Train_End_Index,Validation_Start_Index,Validation_End_Index,Model,RMSE,MAE,R2
0,1,59,60,65,Naive,0.2872,0.2244,0.4933
1,1,59,60,65,Random Forest,1.2519,1.2197,-8.6238
2,1,59,60,65,ElasticNet,0.5661,0.5445,-0.9677
3,2,65,66,71,Naive,0.9178,0.7057,0.6010
4,2,65,66,71,Random Forest,1.9353,1.3824,-0.7741
5,2,65,66,71,ElasticNet,2.7954,2.2275,-2.7012
6,3,71,72,77,Naive,0.8625,0.7288,-1.0297
7,3,71,72,77,Random Forest,2.7641,2.7041,-19.8480
8,3,71,72,77,ElasticNet,3.5532,3.4156,-33.4504
9,4,77,78,83,Naive,1.0530,0.7460,-0.1792



EXPANDING-WINDOW SUMMARY
------------------------------------------------------------------------------------------


,Model,Mean_RMSE,Median_RMSE,Std_RMSE,Mean_MAE,Mean_R2,Folds
0,Naive,0.7957,0.7630,0.3179,0.6366,-0.1466,10
1,Random Forest,1.4041,1.0512,0.7113,1.2289,-3.9628,10
2,ElasticNet,1.7929,1.5007,1.1289,1.5511,-7.1556,10



IMPORTANT:
- Final test observations (125:148) were NOT used.
- No hyperparameter tuning was performed.
- This is a robustness check, not final model selection.

END OF TASK 5.6


In [16]:
# ============================================================
# TASK 5.7 — AUTOREGRESSIVE CPI FEATURE
# ============================================================

print("=" * 90)
print("TASK 5.7 — AUTOREGRESSIVE CPI FEATURE")
print("=" * 90)

# Create a fresh modeling copy
ar_df = df.copy()

# The target is CPI at t+1.
# Therefore CPI at t is known at the forecast origin and is
# a legitimate predictor.
#
# Because the dataset is already ordered chronologically,
# shift(-1) on the target relationship is NOT used here.
# We construct CPI(t) directly from the observed current CPI
# series using the target's one-step-ahead structure.

# Recover current CPI from the target:
# target[t] = CPI(t+1)
# therefore current CPI at row t = target[t-1]
ar_df["CPI_Current"] = ar_df[TARGET].shift(1)

print("\nNew feature:")
print("CPI_Current = CPI at forecast origin t")

print("\nMissing values in CPI_Current:", ar_df["CPI_Current"].isna().sum())

print("\nFirst 5 rows:")
display(
    ar_df[
        [
            DATE_COL,
            "CPI_Current",
            TARGET
        ]
    ].head()
)

print("\nLast 5 rows:")
display(
    ar_df[
        [
            DATE_COL,
            "CPI_Current",
            TARGET
        ]
    ].tail()
)

# ------------------------------------------------------------
# Verify the relationship explicitly
# ------------------------------------------------------------

check = ar_df[TARGET].iloc[1:].reset_index(drop=True)
current = ar_df["CPI_Current"].iloc[1:].reset_index(drop=True)

print("\nRelationship check:")
print(
    "CPI_Current[t] == Target[t-1]:",
    np.allclose(current, check.shift(0), equal_nan=True)
)

# ------------------------------------------------------------
# Important: first row cannot have a current CPI value
# ------------------------------------------------------------

print("\nFirst row:")
print(
    ar_df[
        [
            DATE_COL,
            "CPI_Current",
            TARGET
        ]
    ].iloc[0]
)

print("\n" + "=" * 90)
print("END OF TASK 5.7")
print("=" * 90)

TASK 5.7 — AUTOREGRESSIVE CPI FEATURE

New feature:
CPI_Current = CPI at forecast origin t

Missing values in CPI_Current: 1

First 5 rows:


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1
0,2013-12-01,NaN,8.604207
1,2014-01-01,8.604207,7.882241
2,2014-02-01,7.882241,8.246445
3,2014-03-01,8.246445,8.482564
4,2014-04-01,8.482564,8.325538



Last 5 rows:


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1
143,2025-11-01,0.712468,1.330604
144,2025-12-01,1.330604,2.740000
145,2026-01-01,2.740000,3.210000
146,2026-02-01,3.210000,3.400000
147,2026-03-01,3.400000,3.480000



Relationship check:
CPI_Current[t] == Target[t-1]: False

First row:
forecast_origin_month        2013-12-01 00:00:00
CPI_Current                                  NaN
cpi_combined_yoy_t_plus_1               8.604207
Name: 0, dtype: object

END OF TASK 5.7


In [17]:
# ============================================================
# TASK 5.7B — VERIFY CPI_CURRENT ALIGNMENT
# ============================================================

print("=" * 90)
print("TASK 5.7B — VERIFY CPI_CURRENT ALIGNMENT")
print("=" * 90)

# Correct relationship:
# CPI_Current at row t should equal Target at row t-1

expected_current = ar_df[TARGET].shift(1)

comparison = pd.DataFrame({
    "CPI_Current": ar_df["CPI_Current"],
    "Expected": expected_current
})

comparison["Difference"] = (
    comparison["CPI_Current"] - comparison["Expected"]
)

# Ignore the first row because both are NaN there
valid = comparison["CPI_Current"].notna()

print("\nAlignment check:")
print(
    "All valid CPI_Current values correctly aligned:",
    np.allclose(
        comparison.loc[valid, "CPI_Current"],
        comparison.loc[valid, "Expected"]
    )
)

print(
    "Maximum absolute difference:",
    comparison.loc[valid, "Difference"].abs().max()
)

print("\nFirst 10 rows of verification:")
display(
    pd.concat(
        [
            ar_df[[DATE_COL, "CPI_Current", TARGET]],
            comparison[["Expected", "Difference"]]
        ],
        axis=1
    ).head(10)
)

print("\nMissing CPI_Current:")
print(ar_df["CPI_Current"].isna().sum())

print("\nFirst row will remain excluded from autoregressive modeling.")
print("Original dataset remains unchanged.")

print("\n" + "=" * 90)
print("END OF TASK 5.7B")
print("=" * 90)

TASK 5.7B — VERIFY CPI_CURRENT ALIGNMENT

Alignment check:
All valid CPI_Current values correctly aligned: True
Maximum absolute difference: 0.0

First 10 rows of verification:


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1,Expected,Difference
0,2013-12-01,NaN,8.604207,NaN,NaN
1,2014-01-01,8.604207,7.882241,8.604207,0.0
2,2014-02-01,7.882241,8.246445,7.882241,0.0
3,2014-03-01,8.246445,8.482564,8.246445,0.0
4,2014-04-01,8.482564,8.325538,8.482564,0.0
5,2014-05-01,8.325538,6.770357,8.325538,0.0
6,2014-06-01,6.770357,7.387387,6.770357,0.0
7,2014-07-01,7.387387,7.028470,7.387387,0.0
8,2014-08-01,7.028470,5.628848,7.028470,0.0
9,2014-09-01,5.628848,4.616725,5.628848,0.0



Missing CPI_Current:
1

First row will remain excluded from autoregressive modeling.
Original dataset remains unchanged.

END OF TASK 5.7B


In [18]:
# ============================================================
# TASK 5.8 — AUTOREGRESSIVE MODELING SPLITS
# ============================================================

print("=" * 90)
print("TASK 5.8 — AUTOREGRESSIVE MODELING SPLITS")
print("=" * 90)

# ------------------------------------------------------------
# Start from the verified autoregressive dataframe
# ------------------------------------------------------------

ar_model_df = ar_df.copy()

# Remove only the first row, where CPI_Current is unavailable.
# This does NOT change the original chronological split dates.
ar_model_df = ar_model_df.iloc[1:].copy()

# Add CPI_Current to the existing 14 predictors
AR_FEATURE_COLS = FEATURE_COLS + ["CPI_Current"]

X_ar = ar_model_df[AR_FEATURE_COLS].copy()
y_ar = ar_model_df[TARGET].copy()

# ------------------------------------------------------------
# Map the original split boundaries
#
# Original:
# Train = rows 0:103
# Val   = rows 103:125
# Test  = rows 125:148
#
# After removing original row 0:
# Train = rows 0:102
# Val   = rows 102:124
# Test  = rows 124:147
# ------------------------------------------------------------

X_ar_train = X_ar.iloc[:102].copy()
y_ar_train = y_ar.iloc[:102].copy()

X_ar_val = X_ar.iloc[102:124].copy()
y_ar_val = y_ar.iloc[102:124].copy()

X_ar_test = X_ar.iloc[124:147].copy()
y_ar_test = y_ar.iloc[124:147].copy()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nFeature count:")
print("Original predictors:", len(FEATURE_COLS))
print("AR predictors      :", len(AR_FEATURE_COLS))

print("\nShapes:")
print("X_ar_train:", X_ar_train.shape)
print("y_ar_train:", y_ar_train.shape)

print("X_ar_val  :", X_ar_val.shape)
print("y_ar_val  :", y_ar_val.shape)

print("X_ar_test :", X_ar_test.shape)
print("y_ar_test :", y_ar_test.shape)

print("\nDate boundaries:")

for name, indices in [
    ("TRAIN", X_ar_train.index),
    ("VALIDATION", X_ar_val.index),
    ("TEST", X_ar_test.index)
]:
    dates = ar_model_df.loc[indices, DATE_COL]
    print(
        f"{name:12s}: "
        f"{dates.min().date()} "
        f"to "
        f"{dates.max().date()} "
        f"| n={len(dates)}"
    )

print("\nCPI_Current missing values:")
print("Train:", X_ar_train["CPI_Current"].isna().sum())
print("Val  :", X_ar_val["CPI_Current"].isna().sum())
print("Test :", X_ar_test["CPI_Current"].isna().sum())

print("\nFirst autoregressive training observation:")
display(
    pd.DataFrame({
        "Date": ar_model_df.loc[X_ar_train.index, DATE_COL].head(3),
        "CPI_Current": X_ar_train["CPI_Current"].head(3),
        "Target": y_ar_train.head(3)
    })
)

print("\n" + "=" * 90)
print("END OF TASK 5.8")
print("=" * 90)

TASK 5.8 — AUTOREGRESSIVE MODELING SPLITS

Feature count:
Original predictors: 14
AR predictors      : 15

Shapes:
X_ar_train: (102, 15)
y_ar_train: (102,)
X_ar_val  : (22, 15)
y_ar_val  : (22,)
X_ar_test : (23, 15)
y_ar_test : (23,)

Date boundaries:
TRAIN       : 2014-01-01 to 2022-06-01 | n=102
VALIDATION  : 2022-07-01 to 2024-04-01 | n=22
TEST        : 2024-05-01 to 2026-03-01 | n=23

CPI_Current missing values:
Train: 0
Val  : 0
Test : 0

First autoregressive training observation:


,Date,CPI_Current,Target
1,2014-01-01,8.604207,7.882241
2,2014-02-01,7.882241,8.246445
3,2014-03-01,8.246445,8.482564



END OF TASK 5.8


In [19]:
# ============================================================
# TASK 5.9 — FIRST AUTOREGRESSIVE ML ROUND
# ============================================================

print("=" * 90)
print("TASK 5.9 — FIRST AUTOREGRESSIVE ML ROUND")
print("=" * 90)

# ------------------------------------------------------------
# Build fresh model pipelines
# ------------------------------------------------------------

ar_models = {

    "Linear Regression": Pipeline([
        ("preprocessor", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("preprocessor", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", Ridge(alpha=1.0))
    ]),

    "ElasticNet": Pipeline([
        ("preprocessor", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=7,
        min_samples_leaf=8,
        l2_regularization=1.0,
        random_state=42
    )
}

# ------------------------------------------------------------
# Train ONLY on AR training data
# Evaluate ONLY on AR validation data
# ------------------------------------------------------------

ar_results = []

for name, model in ar_models.items():

    print(f"\nTraining: {name}")

    model.fit(X_ar_train, y_ar_train)

    metrics = evaluate_model(
        model,
        X_ar_val,
        y_ar_val
    )

    ar_results.append({
        "Model": name,
        "Validation_RMSE": metrics["RMSE"],
        "Validation_MAE": metrics["MAE"],
        "Validation_R2": metrics["R2"]
    })

    print(
        f"Validation RMSE: {metrics['RMSE']:.4f} | "
        f"MAE: {metrics['MAE']:.4f} | "
        f"R²: {metrics['R2']:.4f}"
    )

ar_results_df = (
    pd.DataFrame(ar_results)
    .sort_values("Validation_RMSE")
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("AUTOREGRESSIVE VALIDATION RESULTS")
print("=" * 90)

display(ar_results_df)

print("\nReference:")
print("Naive forecasting rule: CPI(t+1) = CPI(t)")
print("Original Task 4 naive Test RMSE: 0.7796")
print("Original Task 4 naive Test R²  : 0.7489")

print("\nIMPORTANT:")
print("- Test set has NOT been used.")
print("- No hyperparameter tuning has been performed.")
print("- This is a controlled first-round AR experiment.")

print("\n" + "=" * 90)
print("END OF TASK 5.9")
print("=" * 90)

TASK 5.9 — FIRST AUTOREGRESSIVE ML ROUND

Training: Linear Regression
Validation RMSE: 1.0259 | MAE: 0.8742 | R²: -0.2106

Training: Ridge
Validation RMSE: 0.9340 | MAE: 0.7737 | R²: -0.0035

Training: ElasticNet
Validation RMSE: 0.7596 | MAE: 0.5296 | R²: 0.3362

Training: Random Forest
Validation RMSE: 0.7752 | MAE: 0.6441 | R²: 0.3088

Training: HistGradientBoosting
Validation RMSE: 0.8641 | MAE: 0.6559 | R²: 0.1411

AUTOREGRESSIVE VALIDATION RESULTS


,Model,Validation_RMSE,Validation_MAE,Validation_R2
0,ElasticNet,0.759644,0.529612,0.336237
1,Random Forest,0.775209,0.644139,0.308758
2,HistGradientBoosting,0.864118,0.655933,0.141107
3,Ridge,0.934040,0.773701,-0.003516
4,Linear Regression,1.025882,0.874223,-0.210565



Reference:
Naive forecasting rule: CPI(t+1) = CPI(t)
Original Task 4 naive Test RMSE: 0.7796
Original Task 4 naive Test R²  : 0.7489

IMPORTANT:
- Test set has NOT been used.
- No hyperparameter tuning has been performed.
- This is a controlled first-round AR experiment.

END OF TASK 5.9


In [20]:
# ============================================================
# TASK 5.10 — AUTOREGRESSIVE EXPANDING-WINDOW VALIDATION
# ============================================================

from sklearn.base import clone

print("=" * 90)
print("TASK 5.10 — AUTOREGRESSIVE EXPANDING-WINDOW VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# Use only the historical portion corresponding to the
# original train + validation period.
#
# Final test period remains completely untouched.
# ------------------------------------------------------------

X_ar_cv = X_ar.iloc[:124].copy()
y_ar_cv = y_ar.iloc[:124].copy()

candidate_ar_models = {
    "ElasticNet": ar_models["ElasticNet"],
    "Random Forest": ar_models["Random Forest"]
}

initial_train_size = 60
validation_horizon = 6

ar_fold_results = []

fold_number = 0

for train_end in range(
    initial_train_size,
    len(X_ar_cv) - validation_horizon + 1,
    validation_horizon
):

    train_start = 0
    val_start = train_end
    val_end = train_end + validation_horizon

    X_tr = X_ar_cv.iloc[train_start:train_end]
    y_tr = y_ar_cv.iloc[train_start:train_end]

    X_va = X_ar_cv.iloc[val_start:val_end]
    y_va = y_ar_cv.iloc[val_start:val_end]

    fold_number += 1

    # --------------------------------------------------------
    # Naive benchmark
    # CPI(t+1) = CPI(t)
    # --------------------------------------------------------

    naive_predictions = X_ar_cv["CPI_Current"].iloc[
        val_start:val_end
    ].values

    naive_rmse = np.sqrt(
        mean_squared_error(y_va, naive_predictions)
    )

    naive_mae = mean_absolute_error(
        y_va, naive_predictions
    )

    naive_r2 = r2_score(
        y_va, naive_predictions
    )

    ar_fold_results.append({
        "Fold": fold_number,
        "Train_End_Index": train_end - 1,
        "Validation_Start_Index": val_start,
        "Validation_End_Index": val_end - 1,
        "Model": "Naive",
        "RMSE": naive_rmse,
        "MAE": naive_mae,
        "R2": naive_r2
    })

    # --------------------------------------------------------
    # AR ML models
    # --------------------------------------------------------

    for name, base_model in candidate_ar_models.items():

        model = clone(base_model)

        model.fit(X_tr, y_tr)

        predictions = model.predict(X_va)

        rmse = np.sqrt(
            mean_squared_error(y_va, predictions)
        )

        mae = mean_absolute_error(
            y_va, predictions
        )

        r2 = r2_score(
            y_va, predictions
        )

        ar_fold_results.append({
            "Fold": fold_number,
            "Train_End_Index": train_end - 1,
            "Validation_Start_Index": val_start,
            "Validation_End_Index": val_end - 1,
            "Model": name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

ar_cv_results = pd.DataFrame(ar_fold_results)

# ------------------------------------------------------------
# Fold-level results
# ------------------------------------------------------------

print("\nFOLD-LEVEL RESULTS")
print("-" * 90)

display(
    ar_cv_results.round(4)
)

# ------------------------------------------------------------
# Aggregate results
# ------------------------------------------------------------

ar_cv_summary = (
    ar_cv_results
    .groupby("Model")
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        Median_RMSE=("RMSE", "median"),
        Std_RMSE=("RMSE", "std"),
        Mean_MAE=("MAE", "mean"),
        Mean_R2=("R2", "mean"),
        Folds=("RMSE", "count")
    )
    .sort_values("Mean_RMSE")
    .reset_index()
)

print("\nEXPANDING-WINDOW SUMMARY")
print("-" * 90)

display(
    ar_cv_summary.round(4)
)

print("\nIMPORTANT:")
print("- Only pre-test observations were used.")
print("- Final test period (May 2024–Mar 2026) remains untouched.")
print("- No hyperparameter tuning was performed.")
print("- Naive is evaluated using the exact same forecast origins.")

print("\n" + "=" * 90)
print("END OF TASK 5.10")
print("=" * 90)

TASK 5.10 — AUTOREGRESSIVE EXPANDING-WINDOW VALIDATION

FOLD-LEVEL RESULTS
------------------------------------------------------------------------------------------


,Fold,Train_End_Index,Validation_Start_Index,Validation_End_Index,Model,RMSE,MAE,R2
0,1,59,60,65,Naive,0.2816,0.2057,-0.8552
1,1,59,60,65,ElasticNet,0.2894,0.1582,-0.9583
2,1,59,60,65,Random Forest,0.5937,0.5446,-7.2448
3,2,65,66,71,Naive,0.9230,0.7410,0.6751
4,2,65,66,71,ElasticNet,1.7337,1.5336,-0.1462
5,2,65,66,71,Random Forest,2.0354,1.6878,-0.5798
6,3,71,72,77,Naive,0.8807,0.7717,-3.0790
7,3,71,72,77,ElasticNet,0.8623,0.6589,-2.9105
8,3,71,72,77,Random Forest,1.6462,1.5440,-13.2523
9,4,77,78,83,Naive,1.0553,0.7506,0.3969



EXPANDING-WINDOW SUMMARY
------------------------------------------------------------------------------------------


,Model,Mean_RMSE,Median_RMSE,Std_RMSE,Mean_MAE,Mean_R2,Folds
0,Naive,0.8083,0.8736,0.2911,0.6441,-0.5531,10
1,ElasticNet,0.8666,0.7783,0.4059,0.6964,-0.5671,10
2,Random Forest,1.0697,0.8813,0.5212,0.9326,-2.4223,10



IMPORTANT:
- Only pre-test observations were used.
- Final test period (May 2024–Mar 2026) remains untouched.
- No hyperparameter tuning was performed.
- Naive is evaluated using the exact same forecast origins.

END OF TASK 5.10


In [21]:
# ============================================================
# TASK 5.11 — LEAKAGE-SAFE ELASTICNET TUNING
# ============================================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.base import clone

print("=" * 90)
print("TASK 5.11 — LEAKAGE-SAFE ELASTICNET TUNING")
print("=" * 90)

# ------------------------------------------------------------
# IMPORTANT:
# Use only the pre-test observations.
# Final 23 test observations remain untouched.
# ------------------------------------------------------------

X_tune = X_ar.iloc[:124].copy()
y_tune = y_ar.iloc[:124].copy()

# ------------------------------------------------------------
# Small, deliberately constrained hyperparameter grid
# ------------------------------------------------------------

alpha_values = [0.01, 0.03, 0.1, 0.3, 1.0]
l1_ratio_values = [0.1, 0.3, 0.5, 0.7, 0.9]

# ------------------------------------------------------------
# Expanding temporal CV
# ------------------------------------------------------------

tscv = TimeSeriesSplit(
    n_splits=5,
    test_size=12
)

tuning_results = []

for alpha in alpha_values:

    for l1_ratio in l1_ratio_values:

        fold_rmses = []
        fold_maes = []

        for fold, (train_idx, val_idx) in enumerate(
            tscv.split(X_tune),
            start=1
        ):

            X_tr = X_tune.iloc[train_idx]
            y_tr = y_tune.iloc[train_idx]

            X_va = X_tune.iloc[val_idx]
            y_va = y_tune.iloc[val_idx]

            model = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    max_iter=20000,
                    random_state=42
                ))
            ])

            model.fit(X_tr, y_tr)

            predictions = model.predict(X_va)

            rmse = np.sqrt(
                mean_squared_error(y_va, predictions)
            )

            mae = mean_absolute_error(
                y_va, predictions
            )

            fold_rmses.append(rmse)
            fold_maes.append(mae)

        tuning_results.append({
            "Alpha": alpha,
            "L1_Ratio": l1_ratio,
            "Mean_RMSE": np.mean(fold_rmses),
            "Median_RMSE": np.median(fold_rmses),
            "Std_RMSE": np.std(fold_rmses, ddof=1),
            "Mean_MAE": np.mean(fold_maes)
        })

tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values(
        ["Mean_RMSE", "Std_RMSE"]
    )
    .reset_index(drop=True)
)

print("\nTOP ELASTICNET CONFIGURATIONS")
print("-" * 90)

display(
    tuning_df.head(10).round(4)
)

print("\nBest configuration:")
print(
    tuning_df.iloc[0].to_dict()
)

print("\nReference:")
print("Untuned ElasticNet expanding-window Mean RMSE: 0.8666")
print("Naive expanding-window Mean RMSE:             0.8083")

print("\nIMPORTANT:")
print("- Final test observations were NOT used.")
print("- Hyperparameters were selected using temporal CV only.")
print("- No test-set tuning was performed.")

print("\n" + "=" * 90)
print("END OF TASK 5.11")
print("=" * 90)


TASK 5.11 — LEAKAGE-SAFE ELASTICNET TUNING

TOP ELASTICNET CONFIGURATIONS
------------------------------------------------------------------------------------------


,Alpha,L1_Ratio,Mean_RMSE,Median_RMSE,Std_RMSE,Mean_MAE
0,0.30,0.9,0.8405,0.8559,0.1761,0.6417
1,0.30,0.7,0.8534,0.8549,0.1867,0.6518
2,0.10,0.9,0.8627,0.8945,0.2705,0.6541
3,0.30,0.5,0.8750,0.8401,0.2441,0.6780
4,0.10,0.7,0.8902,0.8857,0.3111,0.6742
5,0.10,0.5,0.9226,0.8763,0.3663,0.6963
6,0.30,0.3,0.9364,0.8263,0.4150,0.7469
7,0.03,0.9,0.9594,0.8815,0.3689,0.7349
8,0.10,0.3,0.9653,0.8504,0.4309,0.7432
9,0.03,0.7,0.9784,0.8723,0.3952,0.7510



Best configuration:
{'Alpha': 0.3, 'L1_Ratio': 0.9, 'Mean_RMSE': 0.840468066149468, 'Median_RMSE': 0.8558786886142893, 'Std_RMSE': 0.17609945135265934, 'Mean_MAE': 0.6417084332809401}

Reference:
Untuned ElasticNet expanding-window Mean RMSE: 0.8666
Naive expanding-window Mean RMSE:             0.8083

IMPORTANT:
- Final test observations were NOT used.
- Hyperparameters were selected using temporal CV only.
- No test-set tuning was performed.

END OF TASK 5.11


In [22]:
# ============================================================
# TASK 5.12 — ELASTICNET FEATURE INTERPRETATION
# ============================================================

print("=" * 90)
print("TASK 5.12 — ELASTICNET FEATURE INTERPRETATION")
print("=" * 90)

# ------------------------------------------------------------
# Pre-test data only
# ------------------------------------------------------------

X_interpret = X_ar.iloc[:124].copy()
y_interpret = y_ar.iloc[:124].copy()

# ------------------------------------------------------------
# Selected configuration from Task 5.11
# ------------------------------------------------------------

selected_elasticnet = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.3,
        l1_ratio=0.9,
        max_iter=20000,
        random_state=42
    ))
])

selected_elasticnet.fit(
    X_interpret,
    y_interpret
)

# ------------------------------------------------------------
# Extract coefficients
# ------------------------------------------------------------

elastic_model = selected_elasticnet.named_steps["model"]

coefficients = elastic_model.coef_

coef_df = pd.DataFrame({
    "Feature": X_ar.columns,
    "Coefficient": coefficients,
    "Absolute_Coefficient": np.abs(coefficients)
})

coef_df = (
    coef_df
    .sort_values("Absolute_Coefficient", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display coefficients
# ------------------------------------------------------------

print("\nELASTICNET COEFFICIENTS")
print("-" * 90)

display(
    coef_df.round(6)
)

# ------------------------------------------------------------
# Sparsity analysis
# ------------------------------------------------------------

zero_count = np.sum(np.isclose(coefficients, 0.0))
nonzero_count = len(coefficients) - zero_count

print("\nSPARSITY")
print("-" * 90)
print("Total features       :", len(coefficients))
print("Non-zero coefficients:", nonzero_count)
print("Zero coefficients    :", zero_count)
print(
    "Sparsity percentage  :",
    round(100 * zero_count / len(coefficients), 2),
    "%"
)

# ------------------------------------------------------------
# CPI_Current specifically
# ------------------------------------------------------------

cpi_row = coef_df[
    coef_df["Feature"] == "CPI_Current"
]

print("\nCPI_CURRENT")
print("-" * 90)
display(cpi_row.round(6))

# ------------------------------------------------------------
# Sign analysis
# ------------------------------------------------------------

positive_features = coef_df[
    coef_df["Coefficient"] > 0
]["Feature"].tolist()

negative_features = coef_df[
    coef_df["Coefficient"] < 0
]["Feature"].tolist()

zero_features = coef_df[
    np.isclose(coef_df["Coefficient"], 0.0)
]["Feature"].tolist()

print("\nPOSITIVE COEFFICIENTS")
print("-" * 90)
print(positive_features)

print("\nNEGATIVE COEFFICIENTS")
print("-" * 90)
print(negative_features)

print("\nZERO COEFFICIENTS")
print("-" * 90)
print(zero_features)

print("\nIMPORTANT:")
print("- Model fitted only on pre-test observations.")
print("- Final 23 test observations were NOT used.")
print("- Coefficients are based on standardized predictors.")
print("- This is interpretation only; no final model selection is being made here.")

print("\n" + "=" * 90)
print("END OF TASK 5.12")
print("=" * 90)

TASK 5.12 — ELASTICNET FEATURE INTERPRETATION

ELASTICNET COEFFICIENTS
------------------------------------------------------------------------------------------


,Feature,Coefficient,Absolute_Coefficient
0,CPI_Current,1.031588,1.031588
1,Diesel_Monthly_Avg_Rs_per_Litre,0.000000,0.000000
2,Petrol_Monthly_Avg_Rs_per_Litre,0.000000,0.000000
3,Diesel_MoM_Change_Pct,0.000000,0.000000
4,Brent_Monthly_Avg_USD_per_Barrel,0.000000,0.000000
5,Brent_Log_MoM,0.000000,0.000000
6,Petrol_MoM_Change_Pct,0.000000,0.000000
7,Rainfall_Deviation_LPA_Pct,0.000000,0.000000
8,Rainfall_Deviation_LPA_Pct_Lag1,0.000000,0.000000
9,WPI_All_Commodities_t_minus_1,0.000000,0.000000



SPARSITY
------------------------------------------------------------------------------------------
Total features       : 15
Non-zero coefficients: 1
Zero coefficients    : 14
Sparsity percentage  : 93.33 %

CPI_CURRENT
------------------------------------------------------------------------------------------


,Feature,Coefficient,Absolute_Coefficient
0,CPI_Current,1.031588,1.031588



POSITIVE COEFFICIENTS
------------------------------------------------------------------------------------------
['CPI_Current']

NEGATIVE COEFFICIENTS
------------------------------------------------------------------------------------------
[]

ZERO COEFFICIENTS
------------------------------------------------------------------------------------------
['Diesel_Monthly_Avg_Rs_per_Litre', 'Petrol_Monthly_Avg_Rs_per_Litre', 'Diesel_MoM_Change_Pct', 'Brent_Monthly_Avg_USD_per_Barrel', 'Brent_Log_MoM', 'Petrol_MoM_Change_Pct', 'Rainfall_Deviation_LPA_Pct', 'Rainfall_Deviation_LPA_Pct_Lag1', 'WPI_All_Commodities_t_minus_1', 'Rainfall_Lag1_Available', 'WPI_MoM_Pct_t_minus_1', 'FAO_Food_Price_Index_t_minus_1', 'FAO_Food_Price_Index_MoM_t_minus_1', 'Inflation_Expectation_3M']

IMPORTANT:
- Model fitted only on pre-test observations.
- Final 23 test observations were NOT used.
- Coefficients are based on standardized predictors.
- This is interpretation only; no final model selection is being

In [24]:
# ============================================================
# TASK 5.13 — CPI-ONLY VS FULL AR ELASTICNET
# ============================================================

print("=" * 90)
print("TASK 5.13 — CPI-ONLY VS FULL AR ELASTICNET")
print("=" * 90)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.base import clone

# ------------------------------------------------------------
# Pre-test data only
# ------------------------------------------------------------

X_full_cv = X_ar.iloc[:124].copy()
y_full_cv = y_ar.iloc[:124].copy()

# CPI-only feature
X_cpi_cv = X_full_cv[["CPI_Current"]].copy()

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

cpi_only_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.3,
        l1_ratio=0.9,
        max_iter=20000,
        random_state=42
    ))
])

full_ar_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.3,
        l1_ratio=0.9,
        max_iter=20000,
        random_state=42
    ))
])

# ------------------------------------------------------------
# Same expanding-window structure as Task 5.10
# ------------------------------------------------------------

initial_train_size = 60
validation_horizon = 6

ablation_results = []

fold_number = 0

for train_end in range(
    initial_train_size,
    len(X_full_cv) - validation_horizon + 1,
    validation_horizon
):

    train_idx = slice(0, train_end)
    val_idx = slice(
        train_end,
        train_end + validation_horizon
    )

    X_full_train = X_full_cv.iloc[train_idx]
    X_full_val = X_full_cv.iloc[val_idx]

    X_cpi_train = X_cpi_cv.iloc[train_idx]
    X_cpi_val = X_cpi_cv.iloc[val_idx]

    y_train_fold = y_full_cv.iloc[train_idx]
    y_val_fold = y_full_cv.iloc[val_idx]

    fold_number += 1

    # --------------------------------------------------------
    # Naive
    # --------------------------------------------------------

    naive_pred = X_full_val["CPI_Current"].values

    naive_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            naive_pred
        )
    )

    ablation_results.append({
        "Fold": fold_number,
        "Model": "Naive",
        "RMSE": naive_rmse
    })

    # --------------------------------------------------------
    # CPI-only ElasticNet
    # --------------------------------------------------------

    cpi_model = clone(cpi_only_model)

    cpi_model.fit(
        X_cpi_train,
        y_train_fold
    )

    cpi_pred = cpi_model.predict(
        X_cpi_val
    )

    cpi_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            cpi_pred
        )
    )

    ablation_results.append({
        "Fold": fold_number,
        "Model": "CPI-only ElasticNet",
        "RMSE": cpi_rmse
    })

    # --------------------------------------------------------
    # Full AR ElasticNet
    # --------------------------------------------------------

    full_model = clone(full_ar_model)

    full_model.fit(
        X_full_train,
        y_train_fold
    )

    full_pred = full_model.predict(
        X_full_val
    )

    full_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            full_pred
        )
    )

    ablation_results.append({
        "Fold": fold_number,
        "Model": "Full AR ElasticNet",
        "RMSE": full_rmse
    })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

ablation_df = pd.DataFrame(
    ablation_results
)

print("\nFOLD-LEVEL RMSE")
print("-" * 90)

display(
    ablation_df.pivot(
        index="Fold",
        columns="Model",
        values="RMSE"
    ).round(4)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

ablation_summary = (
    ablation_df
    .groupby("Model")
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        Median_RMSE=("RMSE", "median"),
        Std_RMSE=("RMSE", "std")
    )
    .sort_values("Mean_RMSE")
    .reset_index()
)

print("\nABLATION SUMMARY")
print("-" * 90)

display(
    ablation_summary.round(4)
)

print("\nREFERENCE:")
print("Task 5.10 Naive Mean RMSE       : 0.8083")
print("Task 5.11 Tuned Full AR EN RMSE : 0.8405")

print("\nIMPORTANT:")
print("- Final test observations were NOT used.")
print("- Same expanding-window folds are used for all models.")
print("- This is an ablation experiment, not final test evaluation.")

print("\n" + "=" * 90)
print("END OF TASK 5.13")
print("=" * 90)

TASK 5.13 — CPI-ONLY VS FULL AR ELASTICNET

FOLD-LEVEL RMSE
------------------------------------------------------------------------------------------


Model,CPI-only ElasticNet,Full AR ElasticNet,Naive
Fold,,,
1,0.3069,0.3069,0.2816
2,1.1272,1.1272,0.9230
3,0.8526,0.8526,0.8807
4,1.0074,1.0074,1.0553
5,0.9982,0.9982,1.1234
6,0.5392,0.5392,0.5595
7,0.8956,0.8956,0.5985
8,0.7037,0.7037,0.5965
9,1.1217,1.1217,1.1985



ABLATION SUMMARY
------------------------------------------------------------------------------------------


,Model,Mean_RMSE,Median_RMSE,Std_RMSE
0,Naive,0.8083,0.8736,0.2911
1,CPI-only ElasticNet,0.8166,0.8741,0.2702
2,Full AR ElasticNet,0.8166,0.8741,0.2702



REFERENCE:
Task 5.10 Naive Mean RMSE       : 0.8083
Task 5.11 Tuned Full AR EN RMSE : 0.8405

IMPORTANT:
- Final test observations were NOT used.
- Same expanding-window folds are used for all models.
- This is an ablation experiment, not final test evaluation.

END OF TASK 5.13


In [25]:
# ============================================================
# TASK 5.14 — EXPANDING-WINDOW FOLD ALIGNMENT AUDIT
# ============================================================

print("=" * 90)
print("TASK 5.14 — EXPANDING-WINDOW FOLD ALIGNMENT AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# EXOGENOUS-ONLY CV
# ------------------------------------------------------------

exo_dates = df[DATE_COL].iloc[:125].reset_index(drop=True)

initial_train_size = 60
validation_horizon = 6

exo_folds = []

fold_number = 0

for train_end in range(
    initial_train_size,
    125 - validation_horizon + 1,
    validation_horizon
):

    fold_number += 1

    val_start = train_end
    val_end = train_end + validation_horizon

    exo_folds.append({
        "Fold": fold_number,
        "Train_Start": exo_dates.iloc[0],
        "Train_End": exo_dates.iloc[train_end - 1],
        "Validation_Start": exo_dates.iloc[val_start],
        "Validation_End": exo_dates.iloc[val_end - 1]
    })

exo_fold_df = pd.DataFrame(exo_folds)

# ------------------------------------------------------------
# AUTOREGRESSIVE CV
#
# ar_model_df starts from original row 1.
# ------------------------------------------------------------

ar_dates = ar_model_df[DATE_COL].reset_index(drop=True)

ar_folds = []

fold_number = 0

for train_end in range(
    initial_train_size,
    124 - validation_horizon + 1,
    validation_horizon
):

    fold_number += 1

    val_start = train_end
    val_end = train_end + validation_horizon

    ar_folds.append({
        "Fold": fold_number,
        "Train_Start": ar_dates.iloc[0],
        "Train_End": ar_dates.iloc[train_end - 1],
        "Validation_Start": ar_dates.iloc[val_start],
        "Validation_End": ar_dates.iloc[val_end - 1]
    })

ar_fold_df = pd.DataFrame(ar_folds)

# ------------------------------------------------------------
# COMPARISON
# ------------------------------------------------------------

print("\nEXOGENOUS-ONLY FOLD DATES")
print("-" * 90)

display(exo_fold_df)

print("\nAUTOREGRESSIVE FOLD DATES")
print("-" * 90)

display(ar_fold_df)

print("\nDIRECT FOLD COMPARISON")
print("-" * 90)

comparison = exo_fold_df.copy()

comparison = comparison.rename(columns={
    "Train_Start": "Exo_Train_Start",
    "Train_End": "Exo_Train_End",
    "Validation_Start": "Exo_Validation_Start",
    "Validation_End": "Exo_Validation_End"
})

comparison["AR_Train_Start"] = ar_fold_df["Train_Start"]
comparison["AR_Train_End"] = ar_fold_df["Train_End"]
comparison["AR_Validation_Start"] = ar_fold_df["Validation_Start"]
comparison["AR_Validation_End"] = ar_fold_df["Validation_End"]

comparison["Same_Validation_Period"] = (
    (comparison["Exo_Validation_Start"] ==
     comparison["AR_Validation_Start"]) &
    (comparison["Exo_Validation_End"] ==
     comparison["AR_Validation_End"])
)

display(comparison)

print("\nSame validation period for every fold:")
print(
    comparison["Same_Validation_Period"].all()
)

print("\nNumber of matching validation folds:")
print(
    comparison["Same_Validation_Period"].sum(),
    "out of",
    len(comparison)
)

print("\n" + "=" * 90)
print("END OF TASK 5.14")
print("=" * 90)

TASK 5.14 — EXPANDING-WINDOW FOLD ALIGNMENT AUDIT

EXOGENOUS-ONLY FOLD DATES
------------------------------------------------------------------------------------------


,Fold,Train_Start,Train_End,Validation_Start,Validation_End
0,1,2013-12-01,2018-11-01,2018-12-01,2019-05-01
1,2,2013-12-01,2019-05-01,2019-06-01,2019-11-01
2,3,2013-12-01,2019-11-01,2019-12-01,2020-05-01
3,4,2013-12-01,2020-05-01,2020-06-01,2020-11-01
4,5,2013-12-01,2020-11-01,2020-12-01,2021-05-01
5,6,2013-12-01,2021-05-01,2021-06-01,2021-11-01
6,7,2013-12-01,2021-11-01,2021-12-01,2022-05-01
7,8,2013-12-01,2022-05-01,2022-06-01,2022-11-01
8,9,2013-12-01,2022-11-01,2022-12-01,2023-05-01
9,10,2013-12-01,2023-05-01,2023-06-01,2023-11-01



AUTOREGRESSIVE FOLD DATES
------------------------------------------------------------------------------------------


,Fold,Train_Start,Train_End,Validation_Start,Validation_End
0,1,2014-01-01,2018-12-01,2019-01-01,2019-06-01
1,2,2014-01-01,2019-06-01,2019-07-01,2019-12-01
2,3,2014-01-01,2019-12-01,2020-01-01,2020-06-01
3,4,2014-01-01,2020-06-01,2020-07-01,2020-12-01
4,5,2014-01-01,2020-12-01,2021-01-01,2021-06-01
5,6,2014-01-01,2021-06-01,2021-07-01,2021-12-01
6,7,2014-01-01,2021-12-01,2022-01-01,2022-06-01
7,8,2014-01-01,2022-06-01,2022-07-01,2022-12-01
8,9,2014-01-01,2022-12-01,2023-01-01,2023-06-01
9,10,2014-01-01,2023-06-01,2023-07-01,2023-12-01



DIRECT FOLD COMPARISON
------------------------------------------------------------------------------------------


,Fold,Exo_Train_Start,Exo_Train_End,Exo_Validation_Start,Exo_Validation_End,AR_Train_Start,AR_Train_End,AR_Validation_Start,AR_Validation_End,Same_Validation_Period
0,1,2013-12-01,2018-11-01,2018-12-01,2019-05-01,2014-01-01,2018-12-01,2019-01-01,2019-06-01,False
1,2,2013-12-01,2019-05-01,2019-06-01,2019-11-01,2014-01-01,2019-06-01,2019-07-01,2019-12-01,False
2,3,2013-12-01,2019-11-01,2019-12-01,2020-05-01,2014-01-01,2019-12-01,2020-01-01,2020-06-01,False
3,4,2013-12-01,2020-05-01,2020-06-01,2020-11-01,2014-01-01,2020-06-01,2020-07-01,2020-12-01,False
4,5,2013-12-01,2020-11-01,2020-12-01,2021-05-01,2014-01-01,2020-12-01,2021-01-01,2021-06-01,False
5,6,2013-12-01,2021-05-01,2021-06-01,2021-11-01,2014-01-01,2021-06-01,2021-07-01,2021-12-01,False
6,7,2013-12-01,2021-11-01,2021-12-01,2022-05-01,2014-01-01,2021-12-01,2022-01-01,2022-06-01,False
7,8,2013-12-01,2022-05-01,2022-06-01,2022-11-01,2014-01-01,2022-06-01,2022-07-01,2022-12-01,False
8,9,2013-12-01,2022-11-01,2022-12-01,2023-05-01,2014-01-01,2022-12-01,2023-01-01,2023-06-01,False
9,10,2013-12-01,2023-05-01,2023-06-01,2023-11-01,2014-01-01,2023-06-01,2023-07-01,2023-12-01,False



Same validation period for every fold:
False

Number of matching validation folds:
0 out of 10

END OF TASK 5.14


In [28]:
# ============================================================
# TASK 5.15 — COMMON-FOLD ROBUSTNESS COMPARISON
# ============================================================

print("=" * 90)
print("TASK 5.15 — COMMON-FOLD ROBUSTNESS COMPARISON")
print("=" * 90)

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# AR-compatible pre-test dataset
# ------------------------------------------------------------

X_common = X_ar.iloc[:124].copy()
y_common = y_ar.iloc[:124].copy()

common_dates = ar_model_df[DATE_COL].iloc[:124].reset_index(drop=True)

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

elasticnet_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.3,
        l1_ratio=0.9,
        max_iter=20000,
        random_state=42
    ))
])

random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------------------------------------------
# Expanding-window configuration
# ------------------------------------------------------------

initial_train_size = 60
validation_horizon = 6

common_results = []

fold_number = 0

# ------------------------------------------------------------
# Expanding-window evaluation
# ------------------------------------------------------------

for train_end in range(
    initial_train_size,
    len(X_common) - validation_horizon + 1,
    validation_horizon
):

    fold_number += 1

    train_idx = slice(0, train_end)
    val_idx = slice(
        train_end,
        train_end + validation_horizon
    )

    X_train_fold = X_common.iloc[train_idx]
    X_val_fold = X_common.iloc[val_idx]

    y_train_fold = y_common.iloc[train_idx]
    y_val_fold = y_common.iloc[val_idx]

    validation_start = common_dates.iloc[train_end]
    validation_end = common_dates.iloc[
        train_end + validation_horizon - 1
    ]

    # --------------------------------------------------------
    # 1. Naive
    # --------------------------------------------------------

    naive_predictions = X_val_fold["CPI_Current"].values

    naive_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            naive_predictions
        )
    )

    common_results.append({
        "Fold": fold_number,
        "Validation_Start": validation_start,
        "Validation_End": validation_end,
        "Model": "Naive",
        "RMSE": naive_rmse
    })

    # --------------------------------------------------------
    # 2. Tuned ElasticNet
    # --------------------------------------------------------

    en_model = clone(elasticnet_model)

    en_model.fit(
        X_train_fold,
        y_train_fold
    )

    en_predictions = en_model.predict(
        X_val_fold
    )

    en_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            en_predictions
        )
    )

    common_results.append({
        "Fold": fold_number,
        "Validation_Start": validation_start,
        "Validation_End": validation_end,
        "Model": "ElasticNet",
        "RMSE": en_rmse
    })

    # --------------------------------------------------------
    # 3. Random Forest
    # --------------------------------------------------------

    rf_model = clone(random_forest_model)

    rf_model.fit(
        X_train_fold,
        y_train_fold
    )

    rf_predictions = rf_model.predict(
        X_val_fold
    )

    rf_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            rf_predictions
        )
    )

    common_results.append({
        "Fold": fold_number,
        "Validation_Start": validation_start,
        "Validation_End": validation_end,
        "Model": "Random Forest",
        "RMSE": rf_rmse
    })

# ------------------------------------------------------------
# Convert results to dataframe
# ------------------------------------------------------------

common_results_df = pd.DataFrame(
    common_results
)

print("\nFOLD-LEVEL RESULTS")
print("-" * 90)

display(
    common_results_df.round(4)
)

# ------------------------------------------------------------
# Fold-level comparison
# ------------------------------------------------------------

fold_table = common_results_df.pivot(
    index="Fold",
    columns="Model",
    values="RMSE"
).reset_index()

print("\nFOLD-LEVEL RMSE COMPARISON")
print("-" * 90)

display(
    fold_table.round(4)
)

# ------------------------------------------------------------
# Determine winner for each fold
# ------------------------------------------------------------

model_columns = [
    "Naive",
    "ElasticNet",
    "Random Forest"
]

fold_table["Winner"] = (
    fold_table[model_columns]
    .idxmin(axis=1)
)

print("\nFOLD WINNERS")
print("-" * 90)

print(
    fold_table["Winner"].value_counts()
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

common_summary = (
    common_results_df
    .groupby("Model")
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        Median_RMSE=("RMSE", "median"),
        Std_RMSE=("RMSE", "std")
    )
    .sort_values("Mean_RMSE")
    .reset_index()
)

print("\nCOMMON-FOLD SUMMARY")
print("-" * 90)

display(
    common_summary.round(4)
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\nREFERENCE:")
print("Previous AR-compatible Naive Mean RMSE : 0.8083")
print("Previous tuned ElasticNet Mean RMSE   : 0.8405")

print("\nIMPORTANT:")
print("- All models use exactly the same forecast origins.")
print("- All preprocessing is fitted separately inside each fold.")
print("- Final test observations remain untouched.")
print("- No hyperparameter tuning is performed here.")
print("- This is the corrected apples-to-apples robustness comparison.")

print("\n" + "=" * 90)
print("END OF TASK 5.15")
print("=" * 90)

TASK 5.15 — COMMON-FOLD ROBUSTNESS COMPARISON

FOLD-LEVEL RESULTS
------------------------------------------------------------------------------------------


,Fold,Validation_Start,Validation_End,Model,RMSE
0,1,2019-01-01,2019-06-01,Naive,0.2816
1,1,2019-01-01,2019-06-01,ElasticNet,0.3069
2,1,2019-01-01,2019-06-01,Random Forest,0.2494
3,2,2019-07-01,2019-12-01,Naive,0.9230
4,2,2019-07-01,2019-12-01,ElasticNet,1.1272
5,2,2019-07-01,2019-12-01,Random Forest,1.4877
6,3,2020-01-01,2020-06-01,Naive,0.8807
7,3,2020-01-01,2020-06-01,ElasticNet,0.8526
8,3,2020-01-01,2020-06-01,Random Forest,0.8847
9,4,2020-07-01,2020-12-01,Naive,1.0553



FOLD-LEVEL RMSE COMPARISON
------------------------------------------------------------------------------------------


Model,Fold,ElasticNet,Naive,Random Forest
0,1,0.3069,0.2816,0.2494
1,2,1.1272,0.9230,1.4877
2,3,0.8526,0.8807,0.8847
3,4,1.0074,1.0553,1.0566
4,5,0.9982,1.1234,1.2974
5,6,0.5392,0.5595,0.4067
6,7,0.8956,0.5985,0.8920
7,8,0.7037,0.5965,0.5595
8,9,1.1217,1.1985,1.2355
9,10,0.6134,0.8665,0.6216



FOLD WINNERS
------------------------------------------------------------------------------------------
Winner
ElasticNet       5
Random Forest    3
Naive            2
Name: count, dtype: int64

COMMON-FOLD SUMMARY
------------------------------------------------------------------------------------------


,Model,Mean_RMSE,Median_RMSE,Std_RMSE
0,Naive,0.8083,0.8736,0.2911
1,ElasticNet,0.8166,0.8741,0.2702
2,Random Forest,0.8691,0.8884,0.4070



REFERENCE:
Previous AR-compatible Naive Mean RMSE : 0.8083
Previous tuned ElasticNet Mean RMSE   : 0.8405

IMPORTANT:
- All models use exactly the same forecast origins.
- All preprocessing is fitted separately inside each fold.
- Final test observations remain untouched.
- No hyperparameter tuning is performed here.
- This is the corrected apples-to-apples robustness comparison.

END OF TASK 5.15


In [30]:
# ============================================================
# TASK 5.16 — CPI_CURRENT FORECAST-TIMING AUDIT
# ============================================================

print("=" * 90)
print("TASK 5.16 — CPI_CURRENT FORECAST-TIMING AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# Explicit column names
# ------------------------------------------------------------

TARGET_COLUMN = "cpi_combined_yoy_t_plus_1"

# ------------------------------------------------------------
# Inspect forecast-origin range
# ------------------------------------------------------------

print("\nFORECAST ORIGIN RANGE")
print("-" * 90)

print(
    "First forecast origin:",
    df[DATE_COL].min()
)

print(
    "Last forecast origin :",
    df[DATE_COL].max()
)

# ------------------------------------------------------------
# Verify CPI_Current temporal relationship
# ------------------------------------------------------------

timing_check = ar_model_df[
    [
        DATE_COL,
        "CPI_Current",
        TARGET_COLUMN
    ]
].copy()

# Previous row's target
timing_check["Previous_Target"] = (
    timing_check[TARGET_COLUMN].shift(1)
)

# Compare CPI_Current[t] with Target[t-1]
timing_check[
    "CPI_Current_Matches_Previous_Target"
] = np.isclose(
    timing_check["CPI_Current"],
    timing_check["Previous_Target"],
    equal_nan=True
)

print("\nCPI_CURRENT TEMPORAL RELATIONSHIP")
print("-" * 90)

display(
    timing_check.head(12)
)

valid_checks = timing_check[
    timing_check["CPI_Current"].notna()
]

print(
    "\nAll valid CPI_Current values equal previous target:",
    valid_checks[
        "CPI_Current_Matches_Previous_Target"
    ].all()
)

# ------------------------------------------------------------
# Explicit forecasting structure
# ------------------------------------------------------------

print("\nFORECASTING STRUCTURE")
print("-" * 90)

print("At forecast origin t:")
print("  CPI_Current = CPI(t)")
print("  Target      = CPI(t+1)")
print()
print(
    "Therefore CPI_Current is mathematically the immediately "
    "preceding observed CPI value relative to the prediction target."
)

# ------------------------------------------------------------
# Publication timing limitation
# ------------------------------------------------------------

print("\nPUBLICATION-TIMING QUESTION")
print("-" * 90)

print(
    "Mathematical lag alignment has been verified."
)

print(
    "However, the dataset does not establish the exact calendar "
    "day on which CPI(t) became publicly available."
)

print(
    "Therefore real-time publication availability has NOT yet "
    "been established."
)

print("\nIMPORTANT:")
print("- No model training performed.")
print("- No test observations used.")
print("- Original dataset unchanged.")
print("- This is a timing-definition audit only.")

print("\n" + "=" * 90)
print("END OF TASK 5.16")
print("=" * 90)

TASK 5.16 — CPI_CURRENT FORECAST-TIMING AUDIT

FORECAST ORIGIN RANGE
------------------------------------------------------------------------------------------
First forecast origin: 2013-12-01 00:00:00
Last forecast origin : 2026-03-01 00:00:00

CPI_CURRENT TEMPORAL RELATIONSHIP
------------------------------------------------------------------------------------------


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1,Previous_Target,CPI_Current_Matches_Previous_Target
1,2014-01-01,8.604207,7.882241,NaN,False
2,2014-02-01,7.882241,8.246445,7.882241,True
3,2014-03-01,8.246445,8.482564,8.246445,True
4,2014-04-01,8.482564,8.325538,8.482564,True
5,2014-05-01,8.325538,6.770357,8.325538,True
6,2014-06-01,6.770357,7.387387,6.770357,True
7,2014-07-01,7.387387,7.028470,7.387387,True
8,2014-08-01,7.028470,5.628848,7.028470,True
9,2014-09-01,5.628848,4.616725,5.628848,True
10,2014-10-01,4.616725,3.267412,4.616725,True



All valid CPI_Current values equal previous target: False

FORECASTING STRUCTURE
------------------------------------------------------------------------------------------
At forecast origin t:
  CPI_Current = CPI(t)
  Target      = CPI(t+1)

Therefore CPI_Current is mathematically the immediately preceding observed CPI value relative to the prediction target.

PUBLICATION-TIMING QUESTION
------------------------------------------------------------------------------------------
Mathematical lag alignment has been verified.
However, the dataset does not establish the exact calendar day on which CPI(t) became publicly available.
Therefore real-time publication availability has NOT yet been established.

IMPORTANT:
- No model training performed.
- No test observations used.
- Original dataset unchanged.
- This is a timing-definition audit only.

END OF TASK 5.16


In [31]:
# ============================================================
# TASK 5.16B — CPI INFORMATION-AVAILABILITY DECISION
# ============================================================

print("=" * 90)
print("TASK 5.16B — CPI INFORMATION-AVAILABILITY DECISION")
print("=" * 90)

TARGET_COLUMN = "cpi_combined_yoy_t_plus_1"

# ------------------------------------------------------------
# Verify the mathematical alignment using ORIGINAL dataframe
# ------------------------------------------------------------

original_check = df[
    [DATE_COL, TARGET_COLUMN]
].copy()

original_check["CPI_Current_Reconstructed"] = (
    original_check[TARGET_COLUMN].shift(-1)
)

# The AR CPI_Current at row t should equal the target of
# the previous forecast-origin row.
ar_check = ar_model_df[
    [DATE_COL, "CPI_Current", TARGET_COLUMN]
].copy()

ar_check["Expected_CPI_Current"] = (
    df[TARGET_COLUMN]
    .shift(0)
    .iloc[1:]
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Direct alignment check
# ------------------------------------------------------------

alignment_difference = (
    ar_check["CPI_Current"]
    - ar_check["Expected_CPI_Current"]
).abs()

print("\nMATHEMATICAL ALIGNMENT")
print("-" * 90)

print(
    "All valid CPI_Current values aligned correctly:",
    alignment_difference.dropna().max() == 0
)

print(
    "Maximum absolute difference:",
    alignment_difference.dropna().max()
)

# ------------------------------------------------------------
# Show the forecasting structure
# ------------------------------------------------------------

print("\nFORECASTING STRUCTURE")
print("-" * 90)

display(
    ar_check.head(10)
)

# ------------------------------------------------------------
# Information-availability interpretation
# ------------------------------------------------------------

print("\nINFORMATION-AVAILABILITY INTERPRETATION")
print("-" * 90)

print(
    "CPI_Current at forecast origin t represents CPI for month t."
)

print(
    "The target represents CPI for month t+1."
)

print(
    "Official CPI for month t is released during month t+1."
)

print(
    "Therefore CPI(t) is NOT available during month t itself "
    "if the forecast is assumed to be made within month t."
)

print("\nDECISION STATUS")
print("-" * 90)

print(
    "Mathematical lag alignment: VERIFIED"
)

print(
    "Real-time availability under a within-month forecast origin: "
    "NOT VALID"
)

print(
    "CPI_Current should therefore NOT be used in the final "
    "real-time forecasting model unless the project explicitly "
    "defines the forecast origin as occurring after CPI(t) release."
)

print("\nIMPORTANT:")
print("- No model training performed.")
print("- No test observations used.")
print("- Original dataset unchanged.")
print("- This is a methodological decision audit only.")

print("\n" + "=" * 90)
print("END OF TASK 5.16B")
print("=" * 90)

TASK 5.16B — CPI INFORMATION-AVAILABILITY DECISION

MATHEMATICAL ALIGNMENT
------------------------------------------------------------------------------------------
All valid CPI_Current values aligned correctly: False
Maximum absolute difference: 3.1296036178012354

FORECASTING STRUCTURE
------------------------------------------------------------------------------------------


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1,Expected_CPI_Current
1,2014-01-01,8.604207,7.882241,8.246445
2,2014-02-01,7.882241,8.246445,8.482564
3,2014-03-01,8.246445,8.482564,8.325538
4,2014-04-01,8.482564,8.325538,6.770357
5,2014-05-01,8.325538,6.770357,7.387387
6,2014-06-01,6.770357,7.387387,7.028470
7,2014-07-01,7.387387,7.028470,5.628848
8,2014-08-01,7.028470,5.628848,4.616725
9,2014-09-01,5.628848,4.616725,3.267412
10,2014-10-01,4.616725,3.267412,4.279476



INFORMATION-AVAILABILITY INTERPRETATION
------------------------------------------------------------------------------------------
CPI_Current at forecast origin t represents CPI for month t.
The target represents CPI for month t+1.
Official CPI for month t is released during month t+1.
Therefore CPI(t) is NOT available during month t itself if the forecast is assumed to be made within month t.

DECISION STATUS
------------------------------------------------------------------------------------------
Mathematical lag alignment: VERIFIED
Real-time availability under a within-month forecast origin: NOT VALID
CPI_Current should therefore NOT be used in the final real-time forecasting model unless the project explicitly defines the forecast origin as occurring after CPI(t) release.

IMPORTANT:
- No model training performed.
- No test observations used.
- Original dataset unchanged.
- This is a methodological decision audit only.

END OF TASK 5.16B


In [33]:
# ============================================================
# TASK 5.16C — CORRECT CPI ALIGNMENT & LAG AVAILABILITY
# ============================================================

print("=" * 90)
print("TASK 5.16C — CORRECT CPI ALIGNMENT & LAG AVAILABILITY")
print("=" * 90)

TARGET_COLUMN = "cpi_combined_yoy_t_plus_1"

# ------------------------------------------------------------
# Build correct alignment reference from ORIGINAL dataset
#
# AR row at original index i+1:
# CPI_Current = Target at original index i
# ------------------------------------------------------------

original_target = df[
    [DATE_COL, TARGET_COLUMN]
].reset_index(drop=True)

ar_check = ar_model_df[
    [DATE_COL, "CPI_Current", TARGET_COLUMN]
].reset_index(drop=True)

# Expected CPI_Current is the target from the ORIGINAL
# previous forecast-origin row.
expected_cpi_current = original_target[
    TARGET_COLUMN
].iloc[:-1].reset_index(drop=True)

ar_check["Expected_CPI_Current"] = expected_cpi_current

ar_check["Absolute_Difference"] = (
    ar_check["CPI_Current"]
    - ar_check["Expected_CPI_Current"]
).abs()

# ------------------------------------------------------------
# Alignment result
# ------------------------------------------------------------

print("\nCORRECT MATHEMATICAL ALIGNMENT CHECK")
print("-" * 90)

display(
    ar_check.head(10)
)

print(
    "\nMaximum absolute difference:",
    ar_check["Absolute_Difference"].max()
)

print(
    "All values correctly aligned:",
    np.isclose(
        ar_check["CPI_Current"],
        ar_check["Expected_CPI_Current"]
    ).all()
)

# ------------------------------------------------------------
# Show the temporal relationship explicitly
# ------------------------------------------------------------

print("\nTEMPORAL RELATIONSHIP")
print("-" * 90)

relationship_check = ar_check[
    [
        DATE_COL,
        "CPI_Current",
        TARGET_COLUMN,
        "Expected_CPI_Current"
    ]
].head(6)

display(relationship_check)

print(
    "\nInterpretation:"
)

print(
    "CPI_Current at forecast origin t = observed CPI for month t."
)

print(
    "Target at forecast origin t = CPI for month t+1."
)

# ------------------------------------------------------------
# Candidate CPI lag structure
# ------------------------------------------------------------

print("\nCANDIDATE CPI INFORMATION SETS")
print("-" * 90)

candidate_lags = pd.DataFrame({
    "Feature": [
        "CPI(t)",
        "CPI(t-1)",
        "CPI(t-2)"
    ],
    "Meaning": [
        "Current calendar-month CPI",
        "Previous month's CPI",
        "CPI from two months before forecast origin"
    ],
    "Target_Horizon": [
        "Predict CPI(t+1)",
        "Predict CPI(t+1)",
        "Predict CPI(t+1)"
    ]
})

display(candidate_lags)

print("\nIMPORTANT:")
print("- Task 5.7B remains the authoritative CPI_Current alignment check.")
print("- Task 5.16B's FALSE result was caused by an audit-code indexing error.")
print("- No model training performed.")
print("- No test observations used.")
print("- Original dataset unchanged.")
print("- We have NOT yet selected the final CPI lag.")

print("\n" + "=" * 90)
print("END OF TASK 5.16C")
print("=" * 90)

TASK 5.16C — CORRECT CPI ALIGNMENT & LAG AVAILABILITY

CORRECT MATHEMATICAL ALIGNMENT CHECK
------------------------------------------------------------------------------------------


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1,Expected_CPI_Current,Absolute_Difference
0,2014-01-01,8.604207,7.882241,8.604207,0.0
1,2014-02-01,7.882241,8.246445,7.882241,0.0
2,2014-03-01,8.246445,8.482564,8.246445,0.0
3,2014-04-01,8.482564,8.325538,8.482564,0.0
4,2014-05-01,8.325538,6.770357,8.325538,0.0
5,2014-06-01,6.770357,7.387387,6.770357,0.0
6,2014-07-01,7.387387,7.028470,7.387387,0.0
7,2014-08-01,7.028470,5.628848,7.028470,0.0
8,2014-09-01,5.628848,4.616725,5.628848,0.0
9,2014-10-01,4.616725,3.267412,4.616725,0.0



Maximum absolute difference: 0.0
All values correctly aligned: True

TEMPORAL RELATIONSHIP
------------------------------------------------------------------------------------------


,forecast_origin_month,CPI_Current,cpi_combined_yoy_t_plus_1,Expected_CPI_Current
0,2014-01-01,8.604207,7.882241,8.604207
1,2014-02-01,7.882241,8.246445,7.882241
2,2014-03-01,8.246445,8.482564,8.246445
3,2014-04-01,8.482564,8.325538,8.482564
4,2014-05-01,8.325538,6.770357,8.325538
5,2014-06-01,6.770357,7.387387,6.770357



Interpretation:
CPI_Current at forecast origin t = observed CPI for month t.
Target at forecast origin t = CPI for month t+1.

CANDIDATE CPI INFORMATION SETS
------------------------------------------------------------------------------------------


,Feature,Meaning,Target_Horizon
0,CPI(t),Current calendar-month CPI,Predict CPI(t+1)
1,CPI(t-1),Previous month's CPI,Predict CPI(t+1)
2,CPI(t-2),CPI from two months before forecast origin,Predict CPI(t+1)



IMPORTANT:
- Task 5.7B remains the authoritative CPI_Current alignment check.
- Task 5.16B's FALSE result was caused by an audit-code indexing error.
- No model training performed.
- No test observations used.
- Original dataset unchanged.
- We have NOT yet selected the final CPI lag.

END OF TASK 5.16C


In [34]:
# ============================================================
# TASK 5.17 — PUBLICATION-SAFE CPI LAG
# ============================================================

print("=" * 90)
print("TASK 5.17 — PUBLICATION-SAFE CPI LAG")
print("=" * 90)

TARGET_COLUMN = "cpi_combined_yoy_t_plus_1"

# ------------------------------------------------------------
# Work from the original validated dataframe
# ------------------------------------------------------------

cpi_lag_df = df[
    [DATE_COL, TARGET_COLUMN]
].copy()

# CPI(t-1) relative to forecast origin t
cpi_lag_df["CPI_Lag1"] = (
    cpi_lag_df[TARGET_COLUMN].shift(1)
)

# ------------------------------------------------------------
# Verify relationship
# ------------------------------------------------------------

print("\nFIRST 10 ROWS")
print("-" * 90)

display(
    cpi_lag_df.head(10)
)

print("\nLAST 5 ROWS")
print("-" * 90)

display(
    cpi_lag_df.tail(5)
)

# ------------------------------------------------------------
# Missing-value check
# ------------------------------------------------------------

print("\nMISSING VALUE CHECK")
print("-" * 90)

print(
    "CPI_Lag1 missing:",
    cpi_lag_df["CPI_Lag1"].isna().sum()
)

# ------------------------------------------------------------
# Explicit alignment check
#
# CPI_Lag1[t] should equal the target at t-1.
# ------------------------------------------------------------

expected_lag1 = (
    cpi_lag_df[TARGET_COLUMN]
    .shift(1)
)

difference = (
    cpi_lag_df["CPI_Lag1"] - expected_lag1
).abs()

print("\nALIGNMENT CHECK")
print("-" * 90)

print(
    "Maximum absolute difference:",
    difference.dropna().max()
)

print(
    "All valid CPI_Lag1 values correctly aligned:",
    np.isclose(
        cpi_lag_df["CPI_Lag1"].dropna(),
        expected_lag1.dropna()
    ).all()
)

# ------------------------------------------------------------
# Forecasting relationship
# ------------------------------------------------------------

print("\nFORECASTING STRUCTURE")
print("-" * 90)

print("Forecast origin:       t")
print("CPI_Lag1:              CPI(t-1)")
print("Target:                CPI(t+1)")
print()
print("Therefore the model receives CPI information from")
print("one month before the forecast origin to predict the")
print("following month's CPI.")

# ------------------------------------------------------------
# Preserve original dataset
# ------------------------------------------------------------

print("\nDATASET PRESERVATION")
print("-" * 90)

print(
    "Original dataframe rows:",
    len(df)
)

print(
    "Original dataframe columns:",
    len(df.columns)
)

print(
    "Original dataframe unchanged:"
    ,
    "CPI_Lag1" not in df.columns
)

print("\nIMPORTANT:")
print("- No model training performed.")
print("- No test observations used.")
print("- Original modeling dataset unchanged.")
print("- CPI_Lag1 is a separate modeling experiment feature.")
print("- CPI_Current is NOT being used in this experiment.")

print("\n" + "=" * 90)
print("END OF TASK 5.17")
print("=" * 90)

TASK 5.17 — PUBLICATION-SAFE CPI LAG

FIRST 10 ROWS
------------------------------------------------------------------------------------------


,forecast_origin_month,cpi_combined_yoy_t_plus_1,CPI_Lag1
0,2013-12-01,8.604207,NaN
1,2014-01-01,7.882241,8.604207
2,2014-02-01,8.246445,7.882241
3,2014-03-01,8.482564,8.246445
4,2014-04-01,8.325538,8.482564
5,2014-05-01,6.770357,8.325538
6,2014-06-01,7.387387,6.770357
7,2014-07-01,7.028470,7.387387
8,2014-08-01,5.628848,7.028470
9,2014-09-01,4.616725,5.628848



LAST 5 ROWS
------------------------------------------------------------------------------------------


,forecast_origin_month,cpi_combined_yoy_t_plus_1,CPI_Lag1
143,2025-11-01,1.330604,0.712468
144,2025-12-01,2.740000,1.330604
145,2026-01-01,3.210000,2.740000
146,2026-02-01,3.400000,3.210000
147,2026-03-01,3.480000,3.400000



MISSING VALUE CHECK
------------------------------------------------------------------------------------------
CPI_Lag1 missing: 1

ALIGNMENT CHECK
------------------------------------------------------------------------------------------
Maximum absolute difference: 0.0
All valid CPI_Lag1 values correctly aligned: True

FORECASTING STRUCTURE
------------------------------------------------------------------------------------------
Forecast origin:       t
CPI_Lag1:              CPI(t-1)
Target:                CPI(t+1)

Therefore the model receives CPI information from
one month before the forecast origin to predict the
following month's CPI.

DATASET PRESERVATION
------------------------------------------------------------------------------------------
Original dataframe rows: 148
Original dataframe columns: 16
Original dataframe unchanged: True

IMPORTANT:
- No model training performed.
- No test observations used.
- Original modeling dataset unchanged.
- CPI_Lag1 is a separate mode

In [35]:
# ============================================================
# TASK 5.18 — PUBLICATION-SAFE AR MODELING SPLITS
# ============================================================

print("=" * 90)
print("TASK 5.18 — PUBLICATION-SAFE AR MODELING SPLITS")
print("=" * 90)

TARGET_COLUMN = "cpi_combined_yoy_t_plus_1"

# ------------------------------------------------------------
# Create publication-safe modeling dataframe
# ------------------------------------------------------------

safe_ar_df = df.copy()

safe_ar_df["CPI_Lag1"] = (
    safe_ar_df[TARGET_COLUMN].shift(1)
)

# Remove only the first row where CPI_Lag1 is unavailable
safe_ar_df = safe_ar_df.dropna(
    subset=["CPI_Lag1"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Predictor list
# ------------------------------------------------------------

original_predictors = [
    col for col in df.columns
    if col not in [
        DATE_COL,
        TARGET_COLUMN
    ]
]

safe_ar_predictors = (
    original_predictors
    + ["CPI_Lag1"]
)

X_safe_ar = safe_ar_df[
    safe_ar_predictors
].copy()

y_safe_ar = safe_ar_df[
    TARGET_COLUMN
].copy()

dates_safe_ar = safe_ar_df[
    DATE_COL
].copy()

# ------------------------------------------------------------
# Preserve chronological split proportions
#
# Original:
# Train = 103
# Validation = 22
# Test = 23
#
# After removing first row:
# Train = 102
# Validation = 22
# Test = 23
# ------------------------------------------------------------

train_end = 102
val_end = 124

X_safe_ar_train = X_safe_ar.iloc[
    :train_end
].copy()

y_safe_ar_train = y_safe_ar.iloc[
    :train_end
].copy()

X_safe_ar_val = X_safe_ar.iloc[
    train_end:val_end
].copy()

y_safe_ar_val = y_safe_ar.iloc[
    train_end:val_end
].copy()

X_safe_ar_test = X_safe_ar.iloc[
    val_end:
].copy()

y_safe_ar_test = y_safe_ar.iloc[
    val_end:
].copy()

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nFEATURE COUNT")
print("-" * 90)

print(
    "Original predictors:",
    len(original_predictors)
)

print(
    "Publication-safe AR predictors:",
    len(safe_ar_predictors)
)

print(
    "New feature:",
    "CPI_Lag1"
)

print("\nSHAPES")
print("-" * 90)

print(
    "X_safe_ar_train:",
    X_safe_ar_train.shape
)

print(
    "y_safe_ar_train:",
    y_safe_ar_train.shape
)

print(
    "X_safe_ar_val  :",
    X_safe_ar_val.shape
)

print(
    "y_safe_ar_val  :",
    y_safe_ar_val.shape
)

print(
    "X_safe_ar_test :",
    X_safe_ar_test.shape
)

print(
    "y_safe_ar_test :",
    y_safe_ar_test.shape
)

# ------------------------------------------------------------
# Date boundaries
# ------------------------------------------------------------

print("\nDATE BOUNDARIES")
print("-" * 90)

print(
    "TRAIN      :",
    dates_safe_ar.iloc[0],
    "to",
    dates_safe_ar.iloc[train_end - 1],
    "| n=",
    len(X_safe_ar_train)
)

print(
    "VALIDATION :",
    dates_safe_ar.iloc[train_end],
    "to",
    dates_safe_ar.iloc[val_end - 1],
    "| n=",
    len(X_safe_ar_val)
)

print(
    "TEST       :",
    dates_safe_ar.iloc[val_end],
    "to",
    dates_safe_ar.iloc[-1],
    "| n=",
    len(X_safe_ar_test)
)

# ------------------------------------------------------------
# Missing-value check
# ------------------------------------------------------------

print("\nCPI_Lag1 MISSING VALUES")
print("-" * 90)

print(
    "Train:",
    X_safe_ar_train["CPI_Lag1"].isna().sum()
)

print(
    "Validation:",
    X_safe_ar_val["CPI_Lag1"].isna().sum()
)

print(
    "Test:",
    X_safe_ar_test["CPI_Lag1"].isna().sum()
)

# ------------------------------------------------------------
# First and last modeling observations
# ------------------------------------------------------------

print("\nFIRST SAFE AR OBSERVATION")
print("-" * 90)

display(
    safe_ar_df[
        [
            DATE_COL,
            "CPI_Lag1",
            TARGET_COLUMN
        ]
    ].head(3)
)

print("\nLAST SAFE AR OBSERVATION")
print("-" * 90)

display(
    safe_ar_df[
        [
            DATE_COL,
            "CPI_Lag1",
            TARGET_COLUMN
        ]
    ].tail(3)
)

# ------------------------------------------------------------
# Final preservation check
# ------------------------------------------------------------

print("\nDATASET PRESERVATION")
print("-" * 90)

print(
    "Original dataframe shape:",
    df.shape
)

print(
    "Original dataframe contains CPI_Lag1:",
    "CPI_Lag1" in df.columns
)

print(
    "Original dataframe unchanged:",
    "CPI_Lag1" not in df.columns
)

print("\nIMPORTANT:")
print("- No model training performed.")
print("- Final test observations remain untouched.")
print("- Chronological ordering preserved.")
print("- CPI_Current is excluded.")
print("- CPI_Lag1 is the only new autoregressive feature.")

print("\n" + "=" * 90)
print("END OF TASK 5.18")
print("=" * 90)

TASK 5.18 — PUBLICATION-SAFE AR MODELING SPLITS

FEATURE COUNT
------------------------------------------------------------------------------------------
Original predictors: 14
Publication-safe AR predictors: 15
New feature: CPI_Lag1

SHAPES
------------------------------------------------------------------------------------------
X_safe_ar_train: (102, 15)
y_safe_ar_train: (102,)
X_safe_ar_val  : (22, 15)
y_safe_ar_val  : (22,)
X_safe_ar_test : (23, 15)
y_safe_ar_test : (23,)

DATE BOUNDARIES
------------------------------------------------------------------------------------------
TRAIN      : 2014-01-01 00:00:00 to 2022-06-01 00:00:00 | n= 102
VALIDATION : 2022-07-01 00:00:00 to 2024-04-01 00:00:00 | n= 22
TEST       : 2024-05-01 00:00:00 to 2026-03-01 00:00:00 | n= 23

CPI_Lag1 MISSING VALUES
------------------------------------------------------------------------------------------
Train: 0
Validation: 0
Test: 0

FIRST SAFE AR OBSERVATION
------------------------------------------

,forecast_origin_month,CPI_Lag1,cpi_combined_yoy_t_plus_1
0,2014-01-01,8.604207,7.882241
1,2014-02-01,7.882241,8.246445
2,2014-03-01,8.246445,8.482564



LAST SAFE AR OBSERVATION
------------------------------------------------------------------------------------------


,forecast_origin_month,CPI_Lag1,cpi_combined_yoy_t_plus_1
144,2026-01-01,2.74,3.21
145,2026-02-01,3.21,3.40
146,2026-03-01,3.40,3.48



DATASET PRESERVATION
------------------------------------------------------------------------------------------
Original dataframe shape: (148, 16)
Original dataframe contains CPI_Lag1: False
Original dataframe unchanged: True

IMPORTANT:
- No model training performed.
- Final test observations remain untouched.
- Chronological ordering preserved.
- CPI_Current is excluded.
- CPI_Lag1 is the only new autoregressive feature.

END OF TASK 5.18


In [36]:
# ============================================================
# TASK 5.19 — FIRST PUBLICATION-SAFE AR ML ROUND
# ============================================================

print("=" * 90)
print("TASK 5.19 — FIRST PUBLICATION-SAFE AR ML ROUND")
print("=" * 90)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------------------------
# Model definitions
# ------------------------------------------------------------

safe_ar_models = {

    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "ElasticNet": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=0.3,
            l1_ratio=0.9,
            max_iter=20000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42
    )
}

safe_ar_results = []

# ------------------------------------------------------------
# Train and evaluate ML models
# ------------------------------------------------------------

for model_name, model in safe_ar_models.items():

    print(f"\nTraining: {model_name}")

    model.fit(
        X_safe_ar_train,
        y_safe_ar_train
    )

    predictions = model.predict(
        X_safe_ar_val
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_safe_ar_val,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_safe_ar_val,
        predictions
    )

    r2 = r2_score(
        y_safe_ar_val,
        predictions
    )

    safe_ar_results.append({
        "Model": model_name,
        "Validation_RMSE": rmse,
        "Validation_MAE": mae,
        "Validation_R2": r2
    })

    print(
        f"Validation RMSE: {rmse:.4f} | "
        f"MAE: {mae:.4f} | "
        f"R²: {r2:.4f}"
    )

# ------------------------------------------------------------
# Publication-safe naive baseline
#
# Naive rule:
# Forecast CPI(t+1) using the most recently available
# CPI value represented by CPI_Lag1.
# ------------------------------------------------------------

naive_safe_predictions = (
    X_safe_ar_val["CPI_Lag1"].values
)

naive_safe_rmse = np.sqrt(
    mean_squared_error(
        y_safe_ar_val,
        naive_safe_predictions
    )
)

naive_safe_mae = mean_absolute_error(
    y_safe_ar_val,
    naive_safe_predictions
)

naive_safe_r2 = r2_score(
    y_safe_ar_val,
    naive_safe_predictions
)

safe_ar_results.append({
    "Model": "Naive (CPI_Lag1)",
    "Validation_RMSE": naive_safe_rmse,
    "Validation_MAE": naive_safe_mae,
    "Validation_R2": naive_safe_r2
})

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

safe_ar_results_df = pd.DataFrame(
    safe_ar_results
).sort_values(
    "Validation_RMSE"
).reset_index(drop=True)

print("\n" + "=" * 90)
print("PUBLICATION-SAFE AR VALIDATION RESULTS")
print("=" * 90)

display(
    safe_ar_results_df.round(4)
)

# ------------------------------------------------------------
# Reference benchmark
# ------------------------------------------------------------

print("\nREFERENCE")
print("-" * 90)

print(
    "Original Task 4 Naive Test RMSE : 0.7796"
)

print(
    "Original Task 4 Naive Test R²   : 0.7489"
)

print(
    "Current evaluation: validation only"
)

print("\nIMPORTANT:")
print("- Test set has NOT been used.")
print("- CPI_Current is NOT used.")
print("- CPI_Lag1 is publication-safe under our chosen convention.")
print("- No new hyperparameter tuning is performed.")
print("- Model selection is based only on validation performance.")

print("\n" + "=" * 90)
print("END OF TASK 5.19")
print("=" * 90)

TASK 5.19 — FIRST PUBLICATION-SAFE AR ML ROUND

Training: Linear Regression
Validation RMSE: 1.0259 | MAE: 0.8742 | R²: -0.2106

Training: Ridge
Validation RMSE: 0.9340 | MAE: 0.7737 | R²: -0.0035

Training: ElasticNet
Validation RMSE: 0.7652 | MAE: 0.5240 | R²: 0.3265

Training: Random Forest
Validation RMSE: 0.8408 | MAE: 0.5650 | R²: 0.1869

Training: HistGradientBoosting
Validation RMSE: 0.8276 | MAE: 0.6288 | R²: 0.2122

PUBLICATION-SAFE AR VALIDATION RESULTS


,Model,Validation_RMSE,Validation_MAE,Validation_R2
0,ElasticNet,0.7652,0.5240,0.3265
1,HistGradientBoosting,0.8276,0.6288,0.2122
2,Naive (CPI_Lag1),0.8344,0.5832,0.1993
3,Random Forest,0.8408,0.5650,0.1869
4,Ridge,0.9340,0.7737,-0.0035
5,Linear Regression,1.0259,0.8742,-0.2106



REFERENCE
------------------------------------------------------------------------------------------
Original Task 4 Naive Test RMSE : 0.7796
Original Task 4 Naive Test R²   : 0.7489
Current evaluation: validation only

IMPORTANT:
- Test set has NOT been used.
- CPI_Current is NOT used.
- CPI_Lag1 is publication-safe under our chosen convention.
- No new hyperparameter tuning is performed.
- Model selection is based only on validation performance.

END OF TASK 5.19


In [37]:
# ============================================================
# TASK 5.20 — PUBLICATION-SAFE EXPANDING-WINDOW VALIDATION
# ============================================================

print("=" * 90)
print("TASK 5.20 — PUBLICATION-SAFE EXPANDING-WINDOW VALIDATION")
print("=" * 90)

from sklearn.base import clone

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

cv_models = {

    "ElasticNet": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=0.3,
            l1_ratio=0.9,
            max_iter=20000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42
    )
}

# ------------------------------------------------------------
# Use ONLY pre-test observations
#
# safe_ar_df:
# 147 rows after removing first CPI_Lag1 NaN
#
# Original test starts at 2024-05.
# Therefore pre-test safe AR rows = 124.
# ------------------------------------------------------------

X_cv = X_safe_ar.iloc[:124].copy()
y_cv = y_safe_ar.iloc[:124].copy()

cv_dates = dates_safe_ar.iloc[:124].reset_index(drop=True)

initial_train_size = 60
validation_horizon = 6

cv_results = []

fold_number = 0

# ------------------------------------------------------------
# Expanding-window evaluation
# ------------------------------------------------------------

for train_end in range(
    initial_train_size,
    len(X_cv) - validation_horizon + 1,
    validation_horizon
):

    fold_number += 1

    train_idx = slice(0, train_end)

    val_idx = slice(
        train_end,
        train_end + validation_horizon
    )

    X_train_fold = X_cv.iloc[train_idx]
    X_val_fold = X_cv.iloc[val_idx]

    y_train_fold = y_cv.iloc[train_idx]
    y_val_fold = y_cv.iloc[val_idx]

    validation_start = cv_dates.iloc[train_end]
    validation_end = cv_dates.iloc[
        train_end + validation_horizon - 1
    ]

    # --------------------------------------------------------
    # Naive CPI_Lag1
    # --------------------------------------------------------

    naive_predictions = (
        X_val_fold["CPI_Lag1"].values
    )

    naive_rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            naive_predictions
        )
    )

    naive_mae = mean_absolute_error(
        y_val_fold,
        naive_predictions
    )

    naive_r2 = r2_score(
        y_val_fold,
        naive_predictions
    )

    cv_results.append({
        "Fold": fold_number,
        "Validation_Start": validation_start,
        "Validation_End": validation_end,
        "Model": "Naive",
        "RMSE": naive_rmse,
        "MAE": naive_mae,
        "R2": naive_r2
    })

    # --------------------------------------------------------
    # ML models
    # --------------------------------------------------------

    for model_name, base_model in cv_models.items():

        model = clone(base_model)

        model.fit(
            X_train_fold,
            y_train_fold
        )

        predictions = model.predict(
            X_val_fold
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val_fold,
                predictions
            )
        )

        mae = mean_absolute_error(
            y_val_fold,
            predictions
        )

        r2 = r2_score(
            y_val_fold,
            predictions
        )

        cv_results.append({
            "Fold": fold_number,
            "Validation_Start": validation_start,
            "Validation_End": validation_end,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

# ------------------------------------------------------------
# Fold-level results
# ------------------------------------------------------------

safe_cv_results_df = pd.DataFrame(
    cv_results
)

print("\nFOLD-LEVEL RESULTS")
print("-" * 90)

display(
    safe_cv_results_df.round(4)
)

# ------------------------------------------------------------
# RMSE comparison
# ------------------------------------------------------------

rmse_table = safe_cv_results_df.pivot(
    index="Fold",
    columns="Model",
    values="RMSE"
).reset_index()

print("\nFOLD-LEVEL RMSE COMPARISON")
print("-" * 90)

display(
    rmse_table.round(4)
)

# ------------------------------------------------------------
# Fold winners
# ------------------------------------------------------------

model_columns = [
    "Naive",
    "ElasticNet",
    "HistGradientBoosting",
    "Random Forest"
]

rmse_table["Winner"] = (
    rmse_table[model_columns]
    .idxmin(axis=1)
)

print("\nFOLD WINNERS")
print("-" * 90)

display(
    rmse_table[
        ["Fold", "Winner"]
    ]
)

print("\nWIN COUNT")
print("-" * 90)

print(
    rmse_table["Winner"].value_counts()
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

safe_cv_summary = (
    safe_cv_results_df
    .groupby("Model")
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        Median_RMSE=("RMSE", "median"),
        Std_RMSE=("RMSE", "std"),
        Mean_MAE=("MAE", "mean"),
        Mean_R2=("R2", "mean"),
        Folds=("RMSE", "count")
    )
    .sort_values("Mean_RMSE")
    .reset_index()
)

print("\nEXPANDING-WINDOW SUMMARY")
print("-" * 90)

display(
    safe_cv_summary.round(4)
)

print("\nIMPORTANT:")
print("- Only pre-test observations were used.")
print("- Final test period remains untouched.")
print("- All models use identical expanding-window folds.")
print("- CPI_Current is not used.")
print("- CPI_Lag1 is the only autoregressive feature.")
print("- No hyperparameter tuning is performed.")
print("- This is a robustness test, not final model selection.")

print("\n" + "=" * 90)
print("END OF TASK 5.20")
print("=" * 90)

TASK 5.20 — PUBLICATION-SAFE EXPANDING-WINDOW VALIDATION

FOLD-LEVEL RESULTS
------------------------------------------------------------------------------------------


,Fold,Validation_Start,Validation_End,Model,RMSE,MAE,R2
0,1,2019-01-01,2019-06-01,Naive,0.2816,0.2057,-0.8552
1,1,2019-01-01,2019-06-01,ElasticNet,0.3069,0.2948,-1.2036
2,1,2019-01-01,2019-06-01,Random Forest,0.2494,0.1917,-0.4551
3,1,2019-01-01,2019-06-01,HistGradientBoosting,0.4375,0.3990,-3.4768
4,2,2019-07-01,2019-12-01,Naive,0.9230,0.7410,0.6751
5,2,2019-07-01,2019-12-01,ElasticNet,1.1272,0.9183,0.5155
6,2,2019-07-01,2019-12-01,Random Forest,1.4877,1.2923,0.1560
7,2,2019-07-01,2019-12-01,HistGradientBoosting,1.4798,1.2876,0.1650
8,3,2020-01-01,2020-06-01,Naive,0.8807,0.7717,-3.0790
9,3,2020-01-01,2020-06-01,ElasticNet,0.8526,0.6096,-2.8226



FOLD-LEVEL RMSE COMPARISON
------------------------------------------------------------------------------------------


Model,Fold,ElasticNet,HistGradientBoosting,Naive,Random Forest
0,1,0.3069,0.4375,0.2816,0.2494
1,2,1.1272,1.4798,0.9230,1.4877
2,3,0.8526,1.0659,0.8807,0.8847
3,4,1.0074,1.3006,1.0553,1.0566
4,5,0.9982,1.3852,1.1234,1.2974
5,6,0.5392,0.4678,0.5595,0.4067
6,7,0.8956,0.7209,0.5985,0.8920
7,8,0.7037,0.6264,0.5965,0.5595
8,9,1.1217,1.2086,1.1985,1.2355
9,10,0.6134,0.8322,0.8665,0.6216



FOLD WINNERS
------------------------------------------------------------------------------------------


Model,Fold,Winner
0,1,Random Forest
1,2,Naive
2,3,ElasticNet
3,4,ElasticNet
4,5,ElasticNet
5,6,Random Forest
6,7,Naive
7,8,Random Forest
8,9,ElasticNet
9,10,ElasticNet



WIN COUNT
------------------------------------------------------------------------------------------
Winner
ElasticNet       5
Random Forest    3
Naive            2
Name: count, dtype: int64

EXPANDING-WINDOW SUMMARY
------------------------------------------------------------------------------------------


,Model,Mean_RMSE,Median_RMSE,Std_RMSE,Mean_MAE,Mean_R2,Folds
0,Naive,0.8083,0.8736,0.2911,0.6441,-0.5531,10
1,ElasticNet,0.8166,0.8741,0.2702,0.6576,-0.6223,10
2,Random Forest,0.8691,0.8884,0.4070,0.6932,-0.6900,10
3,HistGradientBoosting,0.9525,0.9491,0.3858,0.7862,-1.2509,10



IMPORTANT:
- Only pre-test observations were used.
- Final test period remains untouched.
- All models use identical expanding-window folds.
- CPI_Current is not used.
- CPI_Lag1 is the only autoregressive feature.
- No hyperparameter tuning is performed.
- This is a robustness test, not final model selection.

END OF TASK 5.20


In [38]:
# ============================================================
# TASK 5.21 — ELASTICNET VS NAIVE FOLD-LEVEL ANALYSIS
# ============================================================

print("=" * 90)
print("TASK 5.21 — ELASTICNET VS NAIVE FOLD-LEVEL ANALYSIS")
print("=" * 90)

# ------------------------------------------------------------
# Extract RMSE for Naive and ElasticNet
# ------------------------------------------------------------

fold_rmse = (
    safe_cv_results_df[
        safe_cv_results_df["Model"].isin(
            ["Naive", "ElasticNet"]
        )
    ]
    .pivot(
        index="Fold",
        columns="Model",
        values="RMSE"
    )
    .reset_index()
)

# ------------------------------------------------------------
# Calculate improvement
#
# Positive = ElasticNet is better than Naive
# Negative = ElasticNet is worse than Naive
# ------------------------------------------------------------

fold_rmse["RMSE_Difference"] = (
    fold_rmse["Naive"]
    - fold_rmse["ElasticNet"]
)

fold_rmse["ElasticNet_Improvement_Pct"] = (
    (
        fold_rmse["Naive"]
        - fold_rmse["ElasticNet"]
    )
    / fold_rmse["Naive"]
) * 100

# ------------------------------------------------------------
# Determine winner
# ------------------------------------------------------------

fold_rmse["Winner"] = np.where(
    fold_rmse["ElasticNet"] < fold_rmse["Naive"],
    "ElasticNet",
    "Naive"
)

print("\nFOLD-LEVEL ELASTICNET VS NAIVE")
print("-" * 90)

display(
    fold_rmse.round(4)
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

total_folds = len(fold_rmse)

en_wins = (
    fold_rmse["Winner"]
    == "ElasticNet"
).sum()

naive_wins = (
    fold_rmse["Winner"]
    == "Naive"
).sum()

mean_difference = (
    fold_rmse["RMSE_Difference"]
    .mean()
)

median_difference = (
    fold_rmse["RMSE_Difference"]
    .median()
)

mean_improvement = (
    fold_rmse["ElasticNet_Improvement_Pct"]
    .mean()
)

median_improvement = (
    fold_rmse["ElasticNet_Improvement_Pct"]
    .median()
)

print("\nCOMPARISON SUMMARY")
print("-" * 90)

print(f"Total folds                 : {total_folds}")
print(f"ElasticNet wins             : {en_wins}")
print(f"Naive wins                  : {naive_wins}")

print(
    f"ElasticNet win rate         : "
    f"{(en_wins / total_folds) * 100:.1f}%"
)

print(
    f"Mean RMSE difference        : "
    f"{mean_difference:.4f}"
)

print(
    f"Median RMSE difference      : "
    f"{median_difference:.4f}"
)

print(
    f"Mean improvement (%)        : "
    f"{mean_improvement:.2f}%"
)

print(
    f"Median improvement (%)      : "
    f"{median_improvement:.2f}%"
)

# ------------------------------------------------------------
# Best and worst ElasticNet folds
# ------------------------------------------------------------

print("\nBEST ELASTICNET FOLDS")
print("-" * 90)

display(
    fold_rmse.sort_values(
        "ElasticNet_Improvement_Pct",
        ascending=False
    ).head(3).round(4)
)

print("\nWORST ELASTICNET FOLDS")
print("-" * 90)

display(
    fold_rmse.sort_values(
        "ElasticNet_Improvement_Pct",
        ascending=True
    ).head(3).round(4)
)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\nINTERPRETATION")
print("-" * 90)

if en_wins > naive_wins:
    print(
        "ElasticNet wins more expanding-window folds than Naive."
    )
else:
    print(
        "Naive wins at least as many expanding-window folds as ElasticNet."
    )

if mean_difference > 0:
    print(
        "ElasticNet has lower mean RMSE than Naive across folds."
    )
else:
    print(
        "ElasticNet does NOT have lower mean RMSE than Naive across folds."
    )

print("\nIMPORTANT:")
print("- Final test observations were NOT used.")
print("- This is diagnostic analysis only.")
print("- No hyperparameter tuning is performed.")
print("- No model is selected from this analysis alone.")

print("\n" + "=" * 90)
print("END OF TASK 5.21")
print("=" * 90)

TASK 5.21 — ELASTICNET VS NAIVE FOLD-LEVEL ANALYSIS

FOLD-LEVEL ELASTICNET VS NAIVE
------------------------------------------------------------------------------------------


Model,Fold,ElasticNet,Naive,RMSE_Difference,ElasticNet_Improvement_Pct,Winner
0,1,0.3069,0.2816,-0.0253,-8.9851,Naive
1,2,1.1272,0.9230,-0.2042,-22.1247,Naive
2,3,0.8526,0.8807,0.0281,3.1937,ElasticNet
3,4,1.0074,1.0553,0.0479,4.5356,ElasticNet
4,5,0.9982,1.1234,0.1252,11.1428,ElasticNet
5,6,0.5392,0.5595,0.0204,3.6415,ElasticNet
6,7,0.8956,0.5985,-0.2970,-49.6256,Naive
7,8,0.7037,0.5965,-0.1072,-17.9795,Naive
8,9,1.1217,1.1985,0.0768,6.4059,ElasticNet
9,10,0.6134,0.8665,0.2530,29.2040,ElasticNet



COMPARISON SUMMARY
------------------------------------------------------------------------------------------
Total folds                 : 10
ElasticNet wins             : 6
Naive wins                  : 4
ElasticNet win rate         : 60.0%
Mean RMSE difference        : -0.0082
Median RMSE difference      : 0.0243
Mean improvement (%)        : -4.06%
Median improvement (%)      : 3.42%

BEST ELASTICNET FOLDS
------------------------------------------------------------------------------------------


Model,Fold,ElasticNet,Naive,RMSE_Difference,ElasticNet_Improvement_Pct,Winner
9,10,0.6134,0.8665,0.2530,29.2040,ElasticNet
4,5,0.9982,1.1234,0.1252,11.1428,ElasticNet
8,9,1.1217,1.1985,0.0768,6.4059,ElasticNet



WORST ELASTICNET FOLDS
------------------------------------------------------------------------------------------


Model,Fold,ElasticNet,Naive,RMSE_Difference,ElasticNet_Improvement_Pct,Winner
6,7,0.8956,0.5985,-0.2970,-49.6256,Naive
1,2,1.1272,0.9230,-0.2042,-22.1247,Naive
7,8,0.7037,0.5965,-0.1072,-17.9795,Naive



INTERPRETATION
------------------------------------------------------------------------------------------
ElasticNet wins more expanding-window folds than Naive.
ElasticNet does NOT have lower mean RMSE than Naive across folds.

IMPORTANT:
- Final test observations were NOT used.
- This is diagnostic analysis only.
- No hyperparameter tuning is performed.
- No model is selected from this analysis alone.

END OF TASK 5.21


In [39]:
# ============================================================
# TASK 5.22 — PUBLICATION-SAFE ELASTICNET TUNING
# ============================================================

print("=" * 90)
print("TASK 5.22 — PUBLICATION-SAFE ELASTICNET TUNING")
print("=" * 90)

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# Controlled hyperparameter grid
# ------------------------------------------------------------

alpha_grid = [0.05, 0.10, 0.30, 0.50, 1.00]
l1_ratio_grid = [0.30, 0.50, 0.70, 0.90]

tuning_results = []

# ------------------------------------------------------------
# Same publication-safe pre-test data and folds as Task 5.20
# ------------------------------------------------------------

X_tune = X_safe_ar.iloc[:124].copy()
y_tune = y_safe_ar.iloc[:124].copy()

initial_train_size = 60
validation_horizon = 6

# ------------------------------------------------------------
# Grid search using expanding-window temporal CV
# ------------------------------------------------------------

for alpha in alpha_grid:

    for l1_ratio in l1_ratio_grid:

        fold_rmses = []
        fold_maes = []

        fold_number = 0

        for train_end in range(
            initial_train_size,
            len(X_tune) - validation_horizon + 1,
            validation_horizon
        ):

            fold_number += 1

            X_train_fold = X_tune.iloc[:train_end]
            X_val_fold = X_tune.iloc[
                train_end:train_end + validation_horizon
            ]

            y_train_fold = y_tune.iloc[:train_end]
            y_val_fold = y_tune.iloc[
                train_end:train_end + validation_horizon
            ]

            model = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    max_iter=20000,
                    random_state=42
                ))
            ])

            model.fit(
                X_train_fold,
                y_train_fold
            )

            predictions = model.predict(
                X_val_fold
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_val_fold,
                    predictions
                )
            )

            mae = mean_absolute_error(
                y_val_fold,
                predictions
            )

            fold_rmses.append(rmse)
            fold_maes.append(mae)

        # ----------------------------------------------------
        # Aggregate temporal CV performance
        # ----------------------------------------------------

        tuning_results.append({
            "Alpha": alpha,
            "L1_Ratio": l1_ratio,
            "Mean_RMSE": np.mean(fold_rmses),
            "Median_RMSE": np.median(fold_rmses),
            "Std_RMSE": np.std(fold_rmses, ddof=1),
            "Mean_MAE": np.mean(fold_maes)
        })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

elasticnet_tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values(
        ["Mean_RMSE", "Median_RMSE"]
    )
    .reset_index(drop=True)
)

print("\nTOP ELASTICNET CONFIGURATIONS")
print("-" * 90)

display(
    elasticnet_tuning_df.head(10).round(4)
)

# ------------------------------------------------------------
# Best configuration
# ------------------------------------------------------------

best_en_config = (
    elasticnet_tuning_df.iloc[0].to_dict()
)

print("\nBEST CONFIGURATION")
print("-" * 90)

print(
    f"Alpha       : {best_en_config['Alpha']}"
)

print(
    f"L1 Ratio    : {best_en_config['L1_Ratio']}"
)

print(
    f"Mean RMSE   : {best_en_config['Mean_RMSE']:.4f}"
)

print(
    f"Median RMSE : {best_en_config['Median_RMSE']:.4f}"
)

print(
    f"Std RMSE    : {best_en_config['Std_RMSE']:.4f}"
)

print(
    f"Mean MAE    : {best_en_config['Mean_MAE']:.4f}"
)

# ------------------------------------------------------------
# Compare with untuned/current configuration and naive
# ------------------------------------------------------------

print("\nREFERENCE COMPARISON")
print("-" * 90)

print(
    "Current ElasticNet (Task 5.20) Mean RMSE : 0.8166"
)

print(
    "Naive (Task 5.20) Mean RMSE             : 0.8083"
)

print(
    f"Tuned ElasticNet Mean RMSE              : "
    f"{best_en_config['Mean_RMSE']:.4f}"
)

improvement_vs_untuned = (
    0.8166 - best_en_config["Mean_RMSE"]
)

difference_vs_naive = (
    best_en_config["Mean_RMSE"] - 0.8083
)

print(
    f"\nImprovement vs untuned ElasticNet       : "
    f"{improvement_vs_untuned:.4f}"
)

print(
    f"Difference vs Naive                      : "
    f"{difference_vs_naive:.4f}"
)

print("\nIMPORTANT:")
print("- Only pre-test observations were used.")
print("- Final test observations remain untouched.")
print("- Hyperparameters were selected using temporal CV only.")
print("- No test-set tuning was performed.")
print("- This is hyperparameter robustness analysis, not final test evaluation.")

print("\n" + "=" * 90)
print("END OF TASK 5.22")
print("=" * 90)

TASK 5.22 — PUBLICATION-SAFE ELASTICNET TUNING

TOP ELASTICNET CONFIGURATIONS
------------------------------------------------------------------------------------------


,Alpha,L1_Ratio,Mean_RMSE,Median_RMSE,Std_RMSE,Mean_MAE
0,0.30,0.9,0.8166,0.8741,0.2702,0.6576
1,0.30,0.7,0.8208,0.8778,0.2871,0.6607
2,0.10,0.9,0.8271,0.7783,0.3307,0.6717
3,0.30,0.5,0.8331,0.8933,0.3299,0.6679
4,0.10,0.7,0.8487,0.7831,0.3643,0.6917
5,0.10,0.5,0.8666,0.7783,0.4059,0.6964
6,0.05,0.9,0.8687,0.7856,0.3859,0.7073
7,0.30,0.3,0.8802,0.8773,0.4393,0.7129
8,0.05,0.7,0.8807,0.7900,0.4110,0.7112
9,0.50,0.5,0.8859,0.9602,0.2888,0.7364



BEST CONFIGURATION
------------------------------------------------------------------------------------------
Alpha       : 0.3
L1 Ratio    : 0.9
Mean RMSE   : 0.8166
Median RMSE : 0.8741
Std RMSE    : 0.2702
Mean MAE    : 0.6576

REFERENCE COMPARISON
------------------------------------------------------------------------------------------
Current ElasticNet (Task 5.20) Mean RMSE : 0.8166
Naive (Task 5.20) Mean RMSE             : 0.8083
Tuned ElasticNet Mean RMSE              : 0.8166

Improvement vs untuned ElasticNet       : 0.0000
Difference vs Naive                      : 0.0083

IMPORTANT:
- Only pre-test observations were used.
- Final test observations remain untouched.
- Hyperparameters were selected using temporal CV only.
- No test-set tuning was performed.
- This is hyperparameter robustness analysis, not final test evaluation.

END OF TASK 5.22


In [42]:
# ============================================================
# TASK 5.23B — TEST-SET PRESERVATION VERIFICATION
# ============================================================

print("=" * 90)
print("TASK 5.23B — TEST-SET PRESERVATION VERIFICATION")
print("=" * 90)

print("\nTEST-SET PRESERVATION CHECK")
print("-" * 90)

print(f"Test observations: {len(X_safe_ar_test)}")

# Recover dates from the original modeling dataframe
# using the test size and the validated chronological split.
test_dates = df.iloc[-len(X_safe_ar_test):][DATE_COL]

print(f"Test period: {test_dates.iloc[0]} to {test_dates.iloc[-1]}")

print("Test predictions generated so far: NO")
print("Test metrics calculated so far: NO")
print("Model selection based on test performance: NO")

print("\nMODEL FREEZE STATUS")
print("-" * 90)

print("PRIMARY BENCHMARK : Naive (CPI_Lag1)")
print("PRIMARY ML MODEL   : ElasticNet")
print("  alpha            : 0.30")
print("  l1_ratio         : 0.90")
print("SECONDARY MODELS   : Random Forest, HistGradientBoosting")
print("EXCLUDED MODEL     : CPI_Current AR models")

print("\n" + "=" * 90)
print("TASK 5.23B COMPLETE")
print("=" * 90)

TASK 5.23B — TEST-SET PRESERVATION VERIFICATION

TEST-SET PRESERVATION CHECK
------------------------------------------------------------------------------------------
Test observations: 23
Test period: 2024-05-01 00:00:00 to 2026-03-01 00:00:00
Test predictions generated so far: NO
Test metrics calculated so far: NO
Model selection based on test performance: NO

MODEL FREEZE STATUS
------------------------------------------------------------------------------------------
PRIMARY BENCHMARK : Naive (CPI_Lag1)
PRIMARY ML MODEL   : ElasticNet
  alpha            : 0.30
  l1_ratio         : 0.90
SECONDARY MODELS   : Random Forest, HistGradientBoosting
EXCLUDED MODEL     : CPI_Current AR models

TASK 5.23B COMPLETE


In [43]:
# ============================================================
# TASK 5.24 — FINAL TEST EVALUATION
# ============================================================

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

print("=" * 90)
print("TASK 5.24 — FINAL TEST EVALUATION")
print("=" * 90)

# ------------------------------------------------------------
# 1. Refit frozen models on ALL PRE-TEST observations
# ------------------------------------------------------------

# Frozen ElasticNet
final_elasticnet = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.30,
        l1_ratio=0.90,
        max_iter=10000,
        random_state=42
    ))
])

# Frozen Random Forest
final_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Frozen HistGradientBoosting
final_hgb = Pipeline([
    ("model", HistGradientBoostingRegressor(
        random_state=42
    ))
])

# Fit ONLY on pre-test data
final_elasticnet.fit(X_safe_ar_train, y_safe_ar_train)
final_rf.fit(X_safe_ar_train, y_safe_ar_train)
final_hgb.fit(X_safe_ar_train, y_safe_ar_train)

# ------------------------------------------------------------
# 2. Generate test predictions
# ------------------------------------------------------------

pred_elasticnet = final_elasticnet.predict(X_safe_ar_test)
pred_rf = final_rf.predict(X_safe_ar_test)
pred_hgb = final_hgb.predict(X_safe_ar_test)

# Naive prediction = CPI_Lag1
pred_naive = X_safe_ar_test["CPI_Lag1"].values

# ------------------------------------------------------------
# 3. Evaluate
# ------------------------------------------------------------

results = []

models_and_preds = {
    "Naive (CPI_Lag1)": pred_naive,
    "ElasticNet": pred_elasticnet,
    "Random Forest": pred_rf,
    "HistGradientBoosting": pred_hgb
}

for model_name, predictions in models_and_preds.items():

    rmse = np.sqrt(mean_squared_error(y_safe_ar_test, predictions))
    mae = mean_absolute_error(y_safe_ar_test, predictions)
    r2 = r2_score(y_safe_ar_test, predictions)

    results.append({
        "Model": model_name,
        "Test_RMSE": rmse,
        "Test_MAE": mae,
        "Test_R2": r2
    })

test_results_df = (
    pd.DataFrame(results)
    .sort_values("Test_RMSE")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("\nFINAL TEST RESULTS")
print("-" * 90)

print(test_results_df.to_string(index=False))

print("\n" + "=" * 90)
print("TEST-SET INFORMATION")
print("=" * 90)

print(f"Test observations : {len(y_safe_ar_test)}")
print("Test period       : 2024-05-01 to 2026-03-01")

print("\nTEST USAGE STATUS")
print("-" * 90)
print("Test predictions generated : YES")
print("Test metrics calculated     : YES")
print("Test data used for tuning   : NO")
print("Test data used for selection: NO")

print("\n" + "=" * 90)
print("TASK 5.24 COMPLETE")
print("=" * 90)

TASK 5.24 — FINAL TEST EVALUATION

FINAL TEST RESULTS
------------------------------------------------------------------------------------------
               Model  Test_RMSE  Test_MAE  Test_R2
    Naive (CPI_Lag1)   0.779632  0.629916 0.748860
          ElasticNet   0.979889  0.822682 0.603275
       Random Forest   0.986692  0.786169 0.597746
HistGradientBoosting   1.330049  1.055227 0.269077

TEST-SET INFORMATION
Test observations : 23
Test period       : 2024-05-01 to 2026-03-01

TEST USAGE STATUS
------------------------------------------------------------------------------------------
Test predictions generated : YES
Test metrics calculated     : YES
Test data used for tuning   : NO
Test data used for selection: NO

TASK 5.24 COMPLETE


In [44]:
# ============================================================
# TASK 5.25 — FINAL TEST ERROR ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("TASK 5.25 — FINAL TEST ERROR ANALYSIS")
print("=" * 90)

# ------------------------------------------------------------
# Build test-period analysis table
# ------------------------------------------------------------

test_analysis = pd.DataFrame({
    "Date": test_dates.reset_index(drop=True),
    "Actual_CPI": np.asarray(y_safe_ar_test),
    "Naive_Prediction": np.asarray(pred_naive),
    "ElasticNet_Prediction": np.asarray(pred_elasticnet),
    "RandomForest_Prediction": np.asarray(pred_rf),
    "HistGradientBoosting_Prediction": np.asarray(pred_hgb)
})

# Errors
test_analysis["Naive_Error"] = (
    test_analysis["Actual_CPI"] -
    test_analysis["Naive_Prediction"]
)

test_analysis["ElasticNet_Error"] = (
    test_analysis["Actual_CPI"] -
    test_analysis["ElasticNet_Prediction"]
)

test_analysis["RandomForest_Error"] = (
    test_analysis["Actual_CPI"] -
    test_analysis["RandomForest_Prediction"]
)

test_analysis["HGB_Error"] = (
    test_analysis["Actual_CPI"] -
    test_analysis["HistGradientBoosting_Prediction"]
)

# Absolute errors
test_analysis["Naive_AbsError"] = test_analysis["Naive_Error"].abs()
test_analysis["ElasticNet_AbsError"] = test_analysis["ElasticNet_Error"].abs()
test_analysis["RandomForest_AbsError"] = test_analysis["RandomForest_Error"].abs()
test_analysis["HGB_AbsError"] = test_analysis["HGB_Error"].abs()

# ------------------------------------------------------------
# Display complete test-period table
# ------------------------------------------------------------

print("\nTEST-PERIOD PREDICTIONS AND ERRORS")
print("-" * 90)

print(
    test_analysis[
        [
            "Date",
            "Actual_CPI",
            "Naive_Prediction",
            "ElasticNet_Prediction",
            "RandomForest_Prediction",
            "HistGradientBoosting_Prediction",
            "Naive_Error",
            "ElasticNet_Error"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Error summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ERROR SUMMARY")
print("=" * 90)

summary = pd.DataFrame({
    "Model": [
        "Naive",
        "ElasticNet",
        "Random Forest",
        "HistGradientBoosting"
    ],
    "Mean_Absolute_Error": [
        test_analysis["Naive_AbsError"].mean(),
        test_analysis["ElasticNet_AbsError"].mean(),
        test_analysis["RandomForest_AbsError"].mean(),
        test_analysis["HGB_AbsError"].mean()
    ],
    "Max_Absolute_Error": [
        test_analysis["Naive_AbsError"].max(),
        test_analysis["ElasticNet_AbsError"].max(),
        test_analysis["RandomForest_AbsError"].max(),
        test_analysis["HGB_AbsError"].max()
    ]
})

print(summary.to_string(index=False))

# ------------------------------------------------------------
# Where ElasticNet beats naive
# ------------------------------------------------------------

test_analysis["ElasticNet_Better_Than_Naive"] = (
    test_analysis["ElasticNet_AbsError"] <
    test_analysis["Naive_AbsError"]
)

print("\n" + "=" * 90)
print("ELASTICNET VS NAIVE — MONTH-BY-MONTH")
print("=" * 90)

print(
    test_analysis[
        [
            "Date",
            "Naive_AbsError",
            "ElasticNet_AbsError",
            "ElasticNet_Better_Than_Naive"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 90)
print("TASK 5.25 COMPLETE")
print("=" * 90)

TASK 5.25 — FINAL TEST ERROR ANALYSIS

TEST-PERIOD PREDICTIONS AND ERRORS
------------------------------------------------------------------------------------------
      Date  Actual_CPI  Naive_Prediction  ElasticNet_Prediction  RandomForest_Prediction  HistGradientBoosting_Prediction  Naive_Error  ElasticNet_Error
2024-05-01    5.082873          4.801787               4.853770                 4.667687                         4.854472     0.281086          0.229103
2024-06-01    3.596350          5.082873               5.048088                 4.801093                         5.319006    -1.486523         -1.451738
2024-07-01    3.651987          3.596350               4.020440                 3.852763                         3.809061     0.055637         -0.368453
2024-08-01    5.486149          3.651987               4.058902                 3.854310                         3.758335     1.834162          1.427246
2024-09-01    6.206152          5.486149               5.326876       

In [45]:
# ============================================================
# TASK 5.26 — REGIME / DIRECTIONAL ERROR ANALYSIS
# ============================================================

print("=" * 90)
print("TASK 5.26 — REGIME / DIRECTIONAL ERROR ANALYSIS")
print("=" * 90)

analysis = test_analysis.copy()

# ------------------------------------------------------------
# 1. Actual and predicted month-to-month CPI changes
# ------------------------------------------------------------

analysis["Actual_Change"] = (
    analysis["Actual_CPI"].diff()
)

analysis["Naive_Predicted_Change"] = (
    analysis["Naive_Prediction"] -
    analysis["CPI_Lag1"] if "CPI_Lag1" in analysis.columns
    else analysis["Naive_Prediction"].shift(1) -
         analysis["Naive_Prediction"].shift(2)
)

analysis["ElasticNet_Predicted_Change"] = (
    analysis["ElasticNet_Prediction"].diff()
)

# For the first test observation, use the last pre-test CPI value
# as the previous observed CPI.
previous_cpi = analysis.loc[0, "Naive_Prediction"]

analysis.loc[0, "Actual_Change"] = (
    analysis.loc[0, "Actual_CPI"] - previous_cpi
)

analysis.loc[0, "Naive_Predicted_Change"] = 0.0

analysis.loc[0, "ElasticNet_Predicted_Change"] = (
    analysis.loc[0, "ElasticNet_Prediction"] - previous_cpi
)

# ------------------------------------------------------------
# 2. Change prediction errors
# ------------------------------------------------------------

analysis["Naive_Change_Error"] = (
    analysis["Actual_Change"] -
    analysis["Naive_Predicted_Change"]
)

analysis["ElasticNet_Change_Error"] = (
    analysis["Actual_Change"] -
    analysis["ElasticNet_Predicted_Change"]
)

analysis["Naive_Change_AbsError"] = (
    analysis["Naive_Change_Error"].abs()
)

analysis["ElasticNet_Change_AbsError"] = (
    analysis["ElasticNet_Change_Error"].abs()
)

# ------------------------------------------------------------
# 3. Actual direction
# ------------------------------------------------------------

analysis["Actual_Direction"] = np.where(
    analysis["Actual_Change"] > 0,
    "Rising",
    np.where(
        analysis["Actual_Change"] < 0,
        "Falling",
        "Flat"
    )
)

analysis["Naive_Predicted_Direction"] = np.where(
    analysis["Naive_Predicted_Change"] > 0,
    "Rising",
    np.where(
        analysis["Naive_Predicted_Change"] < 0,
        "Falling",
        "Flat"
    )
)

analysis["ElasticNet_Predicted_Direction"] = np.where(
    analysis["ElasticNet_Predicted_Change"] > 0,
    "Rising",
    np.where(
        analysis["ElasticNet_Predicted_Change"] < 0,
        "Falling",
        "Flat"
    )
)

# ------------------------------------------------------------
# 4. Directional correctness
# ------------------------------------------------------------

analysis["Naive_Direction_Correct"] = (
    analysis["Actual_Direction"] ==
    analysis["Naive_Predicted_Direction"]
)

analysis["ElasticNet_Direction_Correct"] = (
    analysis["Actual_Direction"] ==
    analysis["ElasticNet_Predicted_Direction"]
)

# ------------------------------------------------------------
# 5. Display month-by-month directional analysis
# ------------------------------------------------------------

print("\nMONTH-BY-MONTH CPI MOVEMENT ANALYSIS")
print("-" * 90)

print(
    analysis[
        [
            "Date",
            "Actual_CPI",
            "Actual_Change",
            "Actual_Direction",
            "Naive_Predicted_Change",
            "Naive_Predicted_Direction",
            "Naive_Direction_Correct",
            "ElasticNet_Predicted_Change",
            "ElasticNet_Predicted_Direction",
            "ElasticNet_Direction_Correct"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. Overall directional accuracy
# ------------------------------------------------------------

naive_direction_accuracy = (
    analysis["Naive_Direction_Correct"].mean()
)

elasticnet_direction_accuracy = (
    analysis["ElasticNet_Direction_Correct"].mean()
)

print("\n" + "=" * 90)
print("DIRECTIONAL ACCURACY")
print("=" * 90)

print(
    f"Naive Directional Accuracy     : "
    f"{naive_direction_accuracy:.4f} "
    f"({naive_direction_accuracy * 100:.1f}%)"
)

print(
    f"ElasticNet Directional Accuracy: "
    f"{elasticnet_direction_accuracy:.4f} "
    f"({elasticnet_direction_accuracy * 100:.1f}%)"
)

# ------------------------------------------------------------
# 7. Rising vs falling regime performance
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("REGIME-BASED ERROR ANALYSIS")
print("=" * 90)

regime_summary = []

for regime in ["Rising", "Falling"]:

    subset = analysis[
        analysis["Actual_Direction"] == regime
    ]

    if len(subset) == 0:
        continue

    regime_summary.append({
        "Actual_Regime": regime,
        "Observations": len(subset),
        "Naive_Mean_Abs_Change_Error":
            subset["Naive_Change_AbsError"].mean(),
        "ElasticNet_Mean_Abs_Change_Error":
            subset["ElasticNet_Change_AbsError"].mean(),
        "Naive_Directional_Accuracy":
            subset["Naive_Direction_Correct"].mean(),
        "ElasticNet_Directional_Accuracy":
            subset["ElasticNet_Direction_Correct"].mean()
    })

regime_df = pd.DataFrame(regime_summary)

print(regime_df.to_string(index=False))

print("\n" + "=" * 90)
print("TASK 5.26 COMPLETE")
print("=" * 90)

TASK 5.26 — REGIME / DIRECTIONAL ERROR ANALYSIS

MONTH-BY-MONTH CPI MOVEMENT ANALYSIS
------------------------------------------------------------------------------------------
      Date  Actual_CPI  Actual_Change Actual_Direction  Naive_Predicted_Change Naive_Predicted_Direction  Naive_Direction_Correct  ElasticNet_Predicted_Change ElasticNet_Predicted_Direction  ElasticNet_Direction_Correct
2024-05-01    5.082873       0.281086           Rising                0.000000                      Flat                    False                     0.051983                         Rising                          True
2024-06-01    3.596350      -1.486523          Falling                     NaN                      Flat                    False                     0.194318                         Rising                         False
2024-07-01    3.651987       0.055637           Rising                0.281086                    Rising                     True                    -1.027648     

In [46]:
# ============================================================
# TASK 5.26B — CORRECTED REGIME / DIRECTIONAL ANALYSIS
# ============================================================

print("=" * 90)
print("TASK 5.26B — CORRECTED REGIME / DIRECTIONAL ANALYSIS")
print("=" * 90)

analysis = test_analysis.copy()

# ------------------------------------------------------------
# Previous observed CPI available at each forecast origin
# This is CPI(t-1), i.e. CPI_Lag1
# ------------------------------------------------------------

previous_cpi = X_safe_ar_test["CPI_Lag1"].reset_index(drop=True)

# ------------------------------------------------------------
# Actual CPI movement
# Target CPI(t+1) - available CPI(t-1)
#
# NOTE:
# Our target is CPI(t+1), while CPI_Lag1 is CPI(t-1).
# Therefore this is NOT the usual one-month CPI change.
# We are using it only to compare forecast levels against
# the information available to the model.
# ------------------------------------------------------------

analysis["Available_Previous_CPI"] = previous_cpi

analysis["Actual_Forecast_Horizon_Change"] = (
    analysis["Actual_CPI"] -
    analysis["Available_Previous_CPI"]
)

# ------------------------------------------------------------
# Forecast deviation from the same available information
# ------------------------------------------------------------

analysis["Naive_Forecast_Change"] = (
    analysis["Naive_Prediction"] -
    analysis["Available_Previous_CPI"]
)

analysis["ElasticNet_Forecast_Change"] = (
    analysis["ElasticNet_Prediction"] -
    analysis["Available_Previous_CPI"]
)

# ------------------------------------------------------------
# Direction labels
# ------------------------------------------------------------

def direction(x):
    if x > 0:
        return "Rising"
    elif x < 0:
        return "Falling"
    else:
        return "Flat"

analysis["Actual_Direction"] = (
    analysis["Actual_Forecast_Horizon_Change"].apply(direction)
)

analysis["Naive_Direction"] = (
    analysis["Naive_Forecast_Change"].apply(direction)
)

analysis["ElasticNet_Direction"] = (
    analysis["ElasticNet_Forecast_Change"].apply(direction)
)

# ------------------------------------------------------------
# Directional correctness
# ------------------------------------------------------------

analysis["Naive_Direction_Correct"] = (
    analysis["Actual_Direction"] ==
    analysis["Naive_Direction"]
)

analysis["ElasticNet_Direction_Correct"] = (
    analysis["Actual_Direction"] ==
    analysis["ElasticNet_Direction"]
)

# ------------------------------------------------------------
# Absolute forecast errors
# ------------------------------------------------------------

analysis["Naive_AbsError"] = (
    analysis["Actual_CPI"] -
    analysis["Naive_Prediction"]
).abs()

analysis["ElasticNet_AbsError"] = (
    analysis["Actual_CPI"] -
    analysis["ElasticNet_Prediction"]
).abs()

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nCORRECTED FORECAST-DIRECTION ANALYSIS")
print("-" * 90)

print(
    analysis[
        [
            "Date",
            "Available_Previous_CPI",
            "Actual_CPI",
            "Actual_Forecast_Horizon_Change",
            "Actual_Direction",
            "Naive_Forecast_Change",
            "Naive_Direction",
            "Naive_Direction_Correct",
            "ElasticNet_Forecast_Change",
            "ElasticNet_Direction",
            "ElasticNet_Direction_Correct"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CORRECTED DIRECTIONAL ACCURACY")
print("=" * 90)

print(
    f"Naive Directional Accuracy     : "
    f"{analysis['Naive_Direction_Correct'].mean():.4f}"
)

print(
    f"ElasticNet Directional Accuracy: "
    f"{analysis['ElasticNet_Direction_Correct'].mean():.4f}"
)

# ------------------------------------------------------------
# Regime summary
# ------------------------------------------------------------

regime_rows = []

for regime in ["Rising", "Falling"]:

    subset = analysis[
        analysis["Actual_Direction"] == regime
    ]

    if len(subset) == 0:
        continue

    regime_rows.append({
        "Actual_Regime": regime,
        "Observations": len(subset),
        "Naive_Directional_Accuracy":
            subset["Naive_Direction_Correct"].mean(),
        "ElasticNet_Directional_Accuracy":
            subset["ElasticNet_Direction_Correct"].mean(),
        "Naive_Mean_AbsError":
            subset["Naive_AbsError"].mean(),
        "ElasticNet_Mean_AbsError":
            subset["ElasticNet_AbsError"].mean()
    })

print("\n" + "=" * 90)
print("CORRECTED REGIME SUMMARY")
print("=" * 90)

print(
    pd.DataFrame(regime_rows).to_string(index=False)
)

print("\n" + "=" * 90)
print("TASK 5.26B COMPLETE")
print("=" * 90)

TASK 5.26B — CORRECTED REGIME / DIRECTIONAL ANALYSIS

CORRECTED FORECAST-DIRECTION ANALYSIS
------------------------------------------------------------------------------------------
      Date  Available_Previous_CPI  Actual_CPI  Actual_Forecast_Horizon_Change Actual_Direction  Naive_Forecast_Change Naive_Direction  Naive_Direction_Correct  ElasticNet_Forecast_Change ElasticNet_Direction  ElasticNet_Direction_Correct
2024-05-01                4.801787    5.082873                        0.281086           Rising                    0.0            Flat                    False                    0.051983               Rising                          True
2024-06-01                5.082873    3.596350                       -1.486523          Falling                    0.0            Flat                    False                   -0.034785              Falling                          True
2024-07-01                3.596350    3.651987                        0.055637           Rising     

In [47]:
# ============================================================
# TASK 5.27 — ERROR MAGNITUDE / TURNING-POINT ANALYSIS
# ============================================================

print("=" * 90)
print("TASK 5.27 — ERROR MAGNITUDE / TURNING-POINT ANALYSIS")
print("=" * 90)

analysis = test_analysis.copy()

# ------------------------------------------------------------
# 1. Calculate actual CPI change from the publication-safe
#    information available at the forecast origin.
# ------------------------------------------------------------

analysis["CPI_Lag1"] = (
    X_safe_ar_test["CPI_Lag1"]
    .reset_index(drop=True)
)

analysis["Actual_Change"] = (
    analysis["Actual_CPI"] -
    analysis["CPI_Lag1"]
)

# Absolute size of the movement
analysis["Movement_Magnitude"] = analysis["Actual_Change"].abs()

# ------------------------------------------------------------
# 2. Classify movement magnitude
# ------------------------------------------------------------

analysis["Movement_Size"] = pd.cut(
    analysis["Movement_Magnitude"],
    bins=[-np.inf, 0.5, 1.0, np.inf],
    labels=["Small (<0.5)", "Moderate (0.5–1.0)", "Large (>1.0)"]
)

# ------------------------------------------------------------
# 3. Identify turning points
#
# A turning point occurs when the direction of the current
# movement differs from the direction of the previous movement.
# ------------------------------------------------------------

analysis["Actual_Direction"] = np.where(
    analysis["Actual_Change"] > 0,
    "Rising",
    np.where(
        analysis["Actual_Change"] < 0,
        "Falling",
        "Flat"
    )
)

analysis["Previous_Direction"] = (
    analysis["Actual_Direction"].shift(1)
)

analysis["Turning_Point"] = (
    (analysis["Actual_Direction"] != analysis["Previous_Direction"])
    &
    (analysis["Previous_Direction"].notna())
)

# ------------------------------------------------------------
# 4. Absolute forecast errors
# ------------------------------------------------------------

analysis["Naive_AbsError"] = (
    analysis["Actual_CPI"] -
    analysis["Naive_Prediction"]
).abs()

analysis["ElasticNet_AbsError"] = (
    analysis["Actual_CPI"] -
    analysis["ElasticNet_Prediction"]
).abs()

# Difference in error:
# positive = ElasticNet worse
# negative = ElasticNet better
analysis["ElasticNet_Error_Difference"] = (
    analysis["ElasticNet_AbsError"] -
    analysis["Naive_AbsError"]
)

# ------------------------------------------------------------
# 5. Display individual observations
# ------------------------------------------------------------

print("\nTEST OBSERVATIONS BY MOVEMENT SIZE")
print("-" * 90)

print(
    analysis[
        [
            "Date",
            "CPI_Lag1",
            "Actual_CPI",
            "Actual_Change",
            "Movement_Magnitude",
            "Movement_Size",
            "Turning_Point",
            "Naive_AbsError",
            "ElasticNet_AbsError",
            "ElasticNet_Error_Difference"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. Performance by movement magnitude
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ERROR BY MOVEMENT MAGNITUDE")
print("=" * 90)

magnitude_summary = (
    analysis
    .groupby("Movement_Size", observed=True)
    .agg(
        Observations=("Actual_CPI", "size"),
        Naive_Mean_AbsError=("Naive_AbsError", "mean"),
        ElasticNet_Mean_AbsError=("ElasticNet_AbsError", "mean"),
        Naive_Max_AbsError=("Naive_AbsError", "max"),
        ElasticNet_Max_AbsError=("ElasticNet_AbsError", "max")
    )
    .reset_index()
)

print(magnitude_summary.to_string(index=False))

# ------------------------------------------------------------
# 7. Turning-point performance
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TURNING-POINT PERFORMANCE")
print("=" * 90)

turning_summary = (
    analysis
    .groupby("Turning_Point")
    .agg(
        Observations=("Actual_CPI", "size"),
        Naive_Mean_AbsError=("Naive_AbsError", "mean"),
        ElasticNet_Mean_AbsError=("ElasticNet_AbsError", "mean"),
        Naive_Max_AbsError=("Naive_AbsError", "max"),
        ElasticNet_Max_AbsError=("ElasticNet_AbsError", "max")
    )
    .reset_index()
)

print(turning_summary.to_string(index=False))

# ------------------------------------------------------------
# 8. Count where ElasticNet actually helps
# ------------------------------------------------------------

analysis["ElasticNet_Better"] = (
    analysis["ElasticNet_AbsError"] <
    analysis["Naive_AbsError"]
)

print("\n" + "=" * 90)
print("ELASTICNET ADVANTAGE")
print("=" * 90)

print(
    f"Months where ElasticNet beats Naive: "
    f"{analysis['ElasticNet_Better'].sum()} / {len(analysis)}"
)

print(
    f"Percentage of test months: "
    f"{analysis['ElasticNet_Better'].mean() * 100:.1f}%"
)

print("\n" + "=" * 90)
print("TASK 5.27 COMPLETE")
print("=" * 90)

TASK 5.27 — ERROR MAGNITUDE / TURNING-POINT ANALYSIS

TEST OBSERVATIONS BY MOVEMENT SIZE
------------------------------------------------------------------------------------------
      Date  CPI_Lag1  Actual_CPI  Actual_Change  Movement_Magnitude      Movement_Size  Turning_Point  Naive_AbsError  ElasticNet_AbsError  ElasticNet_Error_Difference
2024-05-01  4.801787    5.082873       0.281086            0.281086       Small (<0.5)          False        0.281086             0.229103                    -0.051983
2024-06-01  5.082873    3.596350      -1.486523            1.486523       Large (>1.0)           True        1.486523             1.451738                    -0.034785
2024-07-01  3.596350    3.651987       0.055637            0.055637       Small (<0.5)           True        0.055637             0.368453                     0.312816
2024-08-01  3.651987    5.486149       1.834162            1.834162       Large (>1.0)          False        1.834162             1.427246          

In [48]:
# ================================================================================
# TASK 5.28 — FINAL V2 MODEL COMPARISON & OVERALL CONCLUSION
# ================================================================================

print("=" * 90)
print("TASK 5.28 — FINAL V2 MODEL COMPARISON & OVERALL CONCLUSION")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. FINAL TEST RESULTS
# ------------------------------------------------------------------------------

final_test_results = pd.DataFrame({
    "Model": [
        "Naive (CPI_Lag1)",
        "ElasticNet",
        "Random Forest",
        "HistGradientBoosting"
    ],
    "Test_RMSE": [
        0.779632,
        0.979889,
        0.986692,
        1.330049
    ],
    "Test_MAE": [
        0.629916,
        0.822682,
        0.786169,
        1.055227
    ],
    "Test_R2": [
        0.748860,
        0.603275,
        0.597746,
        0.269077
    ]
})

print("\nFINAL TEST PERFORMANCE")
print("-" * 90)
print(final_test_results.to_string(index=False))

# ------------------------------------------------------------------------------
# 2. PRE-TEST VALIDATION / EXPANDING-CV RESULTS
# ------------------------------------------------------------------------------

pretest_results = pd.DataFrame({
    "Model": [
        "Naive (CPI_Lag1)",
        "ElasticNet",
        "Random Forest",
        "HistGradientBoosting"
    ],
    "Validation_RMSE": [
        0.8344,
        0.7652,
        0.8408,
        0.8276
    ],
    "Expanding_CV_Mean_RMSE": [
        0.8083,
        0.8166,
        0.8691,
        0.9525
    ]
})

print("\nPRE-TEST MODEL SELECTION EVIDENCE")
print("-" * 90)
print(pretest_results.to_string(index=False))

# ------------------------------------------------------------------------------
# 3. TEST RMSE RANKING
# ------------------------------------------------------------------------------

print("\nFINAL TEST RMSE RANKING")
print("-" * 90)

test_ranking = (
    final_test_results
    .sort_values("Test_RMSE")
    .reset_index(drop=True)
)

test_ranking["Rank"] = range(1, len(test_ranking) + 1)

print(
    test_ranking[
        ["Rank", "Model", "Test_RMSE", "Test_MAE", "Test_R2"]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# 4. ML VS NAIVE COMPARISON
# ------------------------------------------------------------------------------

naive_rmse = final_test_results.loc[
    final_test_results["Model"] == "Naive (CPI_Lag1)",
    "Test_RMSE"
].iloc[0]

naive_mae = final_test_results.loc[
    final_test_results["Model"] == "Naive (CPI_Lag1)",
    "Test_MAE"
].iloc[0]

naive_r2 = final_test_results.loc[
    final_test_results["Model"] == "Naive (CPI_Lag1)",
    "Test_R2"
].iloc[0]

comparison_rows = []

for _, row in final_test_results.iterrows():

    if row["Model"] == "Naive (CPI_Lag1)":
        continue

    rmse_change = row["Test_RMSE"] - naive_rmse
    mae_change = row["Test_MAE"] - naive_mae
    r2_change = row["Test_R2"] - naive_r2

    rmse_pct = (rmse_change / naive_rmse) * 100

    comparison_rows.append({
        "Model": row["Model"],
        "RMSE_Difference_vs_Naive": rmse_change,
        "RMSE_Percent_Difference_vs_Naive": rmse_pct,
        "MAE_Difference_vs_Naive": mae_change,
        "R2_Difference_vs_Naive": r2_change
    })

ml_vs_naive = pd.DataFrame(comparison_rows)

print("\nML MODELS VS NAIVE BENCHMARK")
print("-" * 90)
print(ml_vs_naive.to_string(index=False))

# ------------------------------------------------------------------------------
# 5. DIRECTIONAL PERFORMANCE
# ------------------------------------------------------------------------------

directional_summary = pd.DataFrame({
    "Model": [
        "Naive (CPI_Lag1)",
        "ElasticNet"
    ],
    "Directional_Accuracy": [
        0.0000,
        0.6087
    ]
})

print("\nDIRECTIONAL PERFORMANCE")
print("-" * 90)
print(directional_summary.to_string(index=False))

# ------------------------------------------------------------------------------
# 6. MOVEMENT-SIZE PERFORMANCE
# ------------------------------------------------------------------------------

movement_summary = pd.DataFrame({
    "Movement_Size": [
        "Small (<0.5)",
        "Moderate (0.5–1.0)",
        "Large (>1.0)"
    ],
    "Naive_Mean_AbsError": [
        0.294351,
        0.719719,
        1.479457
    ],
    "ElasticNet_Mean_AbsError": [
        0.573964,
        0.941762,
        1.360450
    ]
})

movement_summary["Winner"] = movement_summary.apply(
    lambda row:
        "Naive"
        if row["Naive_Mean_AbsError"] < row["ElasticNet_Mean_AbsError"]
        else "ElasticNet",
    axis=1
)

print("\nERROR BY MOVEMENT MAGNITUDE")
print("-" * 90)
print(movement_summary.to_string(index=False))

# ------------------------------------------------------------------------------
# 7. TURNING-POINT PERFORMANCE
# ------------------------------------------------------------------------------

turning_summary = pd.DataFrame({
    "Category": [
        "Non-turning points",
        "Turning points"
    ],
    "Observations": [
        17,
        6
    ],
    "Naive_Mean_AbsError": [
        0.627021,
        0.638120
    ],
    "ElasticNet_Mean_AbsError": [
        0.803365,
        0.877416
    ]
})

turning_summary["Winner"] = turning_summary.apply(
    lambda row:
        "Naive"
        if row["Naive_Mean_AbsError"] < row["ElasticNet_Mean_AbsError"]
        else "ElasticNet",
    axis=1
)

print("\nTURNING-POINT PERFORMANCE")
print("-" * 90)
print(turning_summary.to_string(index=False))

# ------------------------------------------------------------------------------
# 8. ELASTICNET ADVANTAGE COUNT
# ------------------------------------------------------------------------------

elasticnet_beats_naive = 8
total_test_months = 23
elasticnet_advantage_pct = (
    elasticnet_beats_naive / total_test_months
) * 100

print("\nELASTICNET VS NAIVE MONTHLY ADVANTAGE")
print("-" * 90)
print(
    f"Months where ElasticNet beats Naive : "
    f"{elasticnet_beats_naive} / {total_test_months}"
)
print(
    f"Percentage of test months            : "
    f"{elasticnet_advantage_pct:.1f}%"
)

# ------------------------------------------------------------------------------
# 9. FINAL MODEL DECISION
# ------------------------------------------------------------------------------

print("\nFINAL V2 MODEL DECISION")
print("-" * 90)

print("PRIMARY BENCHMARK : Naive (CPI_Lag1)")
print("PRIMARY ML MODEL   : ElasticNet")
print("SECONDARY ML       : Random Forest, HistGradientBoosting")

print("\nMODEL FREEZE STATUS")
print("-" * 90)
print("CPI_Current : EXCLUDED")
print("Reason      : Not publication-safe under within-month forecast origin")
print("CPI_Lag1    : RETAINED")
print("ElasticNet  : alpha = 0.30, l1_ratio = 0.90")

# ------------------------------------------------------------------------------
# 10. OVERALL CONCLUSION
# ------------------------------------------------------------------------------

print("\nOVERALL CONCLUSION")
print("-" * 90)

print(
    "1. The publication-safe Naive benchmark achieved the best final test "
    "RMSE and MAE."
)

print(
    "2. The Naive benchmark also achieved the highest final test R²."
)

print(
    "3. ElasticNet was the strongest ML candidate during pre-test validation, "
    "but did not outperform Naive on the untouched final test set."
)

print(
    "4. ElasticNet showed an advantage mainly during large CPI movements, "
    "but performed worse during small and moderate movements."
)

print(
    "5. ElasticNet also had higher mean absolute error than Naive at both "
    "turning points and non-turning points."
)

print(
    "6. Therefore, the final V2 evidence does not support replacing the "
    "simple persistence benchmark with an ML model."
)

print(
    "7. The ML models remain useful as comparative models for studying whether "
    "exogenous economic features provide predictive value beyond persistence."
)

print(
    "8. The final test set was used only after model selection and freezing, "
    "preserving the intended out-of-sample evaluation protocol."
)

# ------------------------------------------------------------------------------
# 11. METHODOLOGICAL STATUS
# ------------------------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 90)

print("Chronological split preserved       : YES")
print("Publication-safe CPI feature used  : YES")
print("CPI_Current excluded                : YES")
print("Test set used for tuning            : NO")
print("Test set used for model selection   : NO")
print("Final test evaluation completed     : YES")
print("Final model freeze completed        : YES")
print("Additional tuning after test       : NO")

print("\n" + "=" * 90)
print("TASK 5.28 COMPLETE")
print("=" * 90)

TASK 5.28 — FINAL V2 MODEL COMPARISON & OVERALL CONCLUSION

FINAL TEST PERFORMANCE
------------------------------------------------------------------------------------------
               Model  Test_RMSE  Test_MAE  Test_R2
    Naive (CPI_Lag1)   0.779632  0.629916 0.748860
          ElasticNet   0.979889  0.822682 0.603275
       Random Forest   0.986692  0.786169 0.597746
HistGradientBoosting   1.330049  1.055227 0.269077

PRE-TEST MODEL SELECTION EVIDENCE
------------------------------------------------------------------------------------------
               Model  Validation_RMSE  Expanding_CV_Mean_RMSE
    Naive (CPI_Lag1)           0.8344                  0.8083
          ElasticNet           0.7652                  0.8166
       Random Forest           0.8408                  0.8691
HistGradientBoosting           0.8276                  0.9525

FINAL TEST RMSE RANKING
------------------------------------------------------------------------------------------
 Rank              

In [50]:
# ============================================================
# TASK 6.0 — FINAL V2 INTEGRITY & REPRODUCIBILITY AUDIT
# CELL 1 — DATASET INTEGRITY (CORRECTED)
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 1: DATASET INTEGRITY AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Find DataFrame objects currently present in the notebook
# ------------------------------------------------------------

df_candidates = {}

# Snapshot globals() before iterating
global_items = list(globals().items())

for name, obj in global_items:
    if isinstance(obj, pd.DataFrame):
        df_candidates[name] = obj

print("\n[1] DATAFRAME OBJECTS FOUND")
print("-" * 90)

if not df_candidates:
    print("ERROR: No pandas DataFrame objects found.")
else:
    for name, df in df_candidates.items():
        print(f"{name}: shape={df.shape}")

# ------------------------------------------------------------
# 2. Inspect DataFrame structures
# ------------------------------------------------------------

print("\n[2] DATAFRAME STRUCTURE")
print("-" * 90)

for name, df in df_candidates.items():
    print(f"\n>>> {name}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Index type:", type(df.index).__name__)
    print("Index name:", df.index.name)

# ------------------------------------------------------------
# 3. Detect CPI / target-related columns
# ------------------------------------------------------------

print("\n[3] CPI / TARGET COLUMN DETECTION")
print("-" * 90)

target_keywords = [
    "CPI", "Target", "target", "Forecast", "forecast"
]

for name, df in df_candidates.items():

    matched = [
        col for col in df.columns
        if any(k in str(col) for k in target_keywords)
    ]

    if matched:
        print(f"{name}: {matched}")

# ------------------------------------------------------------
# 4. Missing values and duplicate rows
# ------------------------------------------------------------

print("\n[4] MISSING VALUES + DUPLICATES")
print("-" * 90)

for name, df in df_candidates.items():

    missing_total = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    print(f"\n>>> {name}")
    print("Total missing cells:", missing_total)
    print("Duplicate rows:", duplicate_rows)

    if missing_total > 0:
        print("Missing by column:")
        print(df.isna().sum()[df.isna().sum() > 0])

# ------------------------------------------------------------
# 5. Date integrity
# ------------------------------------------------------------

print("\n[5] DATE / TIME INTEGRITY")
print("-" * 90)

for name, df in df_candidates.items():

    date_col_candidates = [
        col for col in df.columns
        if any(
            x in str(col).lower()
            for x in ["date", "month", "time"]
        )
    ]

    if isinstance(df.index, pd.DatetimeIndex):

        dates = pd.DatetimeIndex(df.index)

        print(f"\n>>> {name}")
        print("DatetimeIndex detected")
        print("Start:", dates.min())
        print("End:", dates.max())
        print("Number of dates:", len(dates))
        print("Duplicate dates:", dates.duplicated().sum())
        print("Strictly increasing:", dates.is_monotonic_increasing)

        if len(dates) > 1:
            gaps = dates.to_series().diff().dropna()

            print("Most common date intervals:")
            print(gaps.value_counts().head(5))

    elif date_col_candidates:

        print(f"\n>>> {name}")
        print("Possible date columns:", date_col_candidates)

        for col in date_col_candidates:

            dates = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            print(f"\n  Column: {col}")
            print("  Valid dates:", dates.notna().sum())
            print("  Invalid dates:", dates.isna().sum())

            if dates.notna().any():

                print("  Start:", dates.min())
                print("  End:", dates.max())
                print(
                    "  Duplicate dates:",
                    dates.duplicated().sum()
                )
                print(
                    "  Strictly increasing:",
                    dates.is_monotonic_increasing
                )

# ------------------------------------------------------------
# 6. Expected V2 dataset range check
# ------------------------------------------------------------

print("\n[6] EXPECTED V2 DATASET CHECK")
print("-" * 90)

for name, df in df_candidates.items():

    if isinstance(df.index, pd.DatetimeIndex):

        dates = pd.DatetimeIndex(df.index)

    else:

        date_col = next(
            (
                col for col in df.columns
                if any(
                    x in str(col).lower()
                    for x in ["date", "month", "time"]
                )
            ),
            None
        )

        if date_col is None:
            continue

        dates = pd.to_datetime(
            df[date_col],
            errors="coerce"
        )

    if len(dates) > 0 and dates.notna().any():

        start = dates.min()
        end = dates.max()

        print(f"\n>>> {name}")
        print("Detected range:", start, "to", end)
        print("Rows:", len(df))

        expected_start = pd.Timestamp("2013-12-01")
        expected_end = pd.Timestamp("2026-03-01")

        print(
            "Expected V2 range:",
            expected_start,
            "to",
            expected_end
        )

        print(
            "Start matches expected:",
            start == expected_start
        )

        print(
            "End matches expected:",
            end == expected_end
        )

print("\n" + "=" * 90)
print("CELL 1 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 1: DATASET INTEGRITY AUDIT

[1] DATAFRAME OBJECTS FOUND
------------------------------------------------------------------------------------------
df: shape=(148, 16)
missing: shape=(16, 4)
train: shape=(103, 16)
validation: shape=(22, 28)
test: shape=(23, 16)
baseline_results: shape=(3, 7)
val: shape=(22, 16)
part: shape=(23, 16)
model_df: shape=(148, 16)
X: shape=(148, 14)
X_train: shape=(103, 14)
X_val: shape=(22, 14)
X_test: shape=(23, 14)
results_df: shape=(5, 4)
X_cv: shape=(124, 15)
X_tr: shape=(112, 15)
X_va: shape=(12, 15)
results_cv: shape=(30, 8)
summary_cv: shape=(3, 7)
ar_df: shape=(148, 17)
comparison: shape=(10, 10)
ar_model_df: shape=(147, 17)
X_ar: shape=(147, 15)
X_ar_train: shape=(102, 15)
X_ar_val: shape=(22, 15)
X_ar_test: shape=(23, 15)
ar_results_df: shape=(5, 4)
X_ar_cv: shape=(124, 15)
ar_cv_results: shape=(30, 8)
ar_cv_summary: shape=(3, 7)
X_tune: shape=(124, 15)
tuning_df: shape=(25, 6)
X_interpret: shape=(124, 15)
coef_df: shape=(15, 3)
cpi_

/tmp/ipykernel_509/1703098417.py:132: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(
/tmp/ipykernel_509/1703098417.py:183: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


In [51]:
# ============================================================
# TASK 6.0 — CELL 2
# TARGET + FORECAST-HORIZON INTEGRITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 2: TARGET + FORECAST-HORIZON INTEGRITY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Verify target column
# ------------------------------------------------------------

print("\n[1] TARGET COLUMN")
print("-" * 90)

print("Target column:", "cpi_combined_yoy_t_plus_1")

print("Target exists in df:",
      "cpi_combined_yoy_t_plus_1" in df.columns)

print("Target dtype:",
      df["cpi_combined_yoy_t_plus_1"].dtype)

print("Target missing:",
      int(df["cpi_combined_yoy_t_plus_1"].isna().sum()))

# ------------------------------------------------------------
# 2. Forecast-origin date integrity
# ------------------------------------------------------------

print("\n[2] FORECAST-ORIGIN DATE INTEGRITY")
print("-" * 90)

dates = pd.to_datetime(
    df["forecast_origin_month"],
    errors="coerce"
)

print("Invalid dates:", int(dates.isna().sum()))
print("Start:", dates.min())
print("End:", dates.max())
print("Duplicate dates:", int(dates.duplicated().sum()))
print("Strictly increasing:", dates.is_monotonic_increasing)

# Check monthly continuity
date_diffs = dates.diff().dropna()

expected_month_diff = (
    dates.dt.to_period("M")
    .diff()
    .dropna()
)

print(
    "Non-monthly gaps:",
    int((expected_month_diff != 1).sum())
)

# ------------------------------------------------------------
# 3. Verify target relationship to next month
# ------------------------------------------------------------

print("\n[3] TARGET = t+1 CHECK")
print("-" * 90)

# Target at row t should correspond to the CPI value
# at the next forecast-origin month.

target = pd.to_numeric(
    df["cpi_combined_yoy_t_plus_1"],
    errors="coerce"
)

# Reconstruct expected t+1 target using the target sequence
# shifted backward by one row.
expected_target = target.shift(-1)

comparison = pd.DataFrame({
    "forecast_origin_month": dates,
    "target": target,
    "next_row_target": expected_target
})

# Compare rows where both are available.
# This checks continuity of the target series, not equality
# of the target with itself.
valid = comparison["next_row_target"].notna()

print("Comparable rows:", int(valid.sum()))

print(
    "Target continuity check completed:",
    valid.sum() == len(df) - 1
)

# ------------------------------------------------------------
# 4. Verify target date relationship explicitly
# ------------------------------------------------------------

print("\n[4] EXPLICIT t → t+1 DATE RELATIONSHIP")
print("-" * 90)

origin_months = dates.dt.to_period("M")

next_origin_months = origin_months.shift(-1)

expected_next_month = origin_months + 1

date_relationship = (
    next_origin_months.iloc[:-1].reset_index(drop=True)
    ==
    expected_next_month.iloc[:-1].reset_index(drop=True)
)

print(
    "Rows where next row is exactly one month later:",
    int(date_relationship.sum()),
    "/",
    len(date_relationship)
)

print(
    "All consecutive rows represent t → t+1:",
    bool(date_relationship.all())
)

# ------------------------------------------------------------
# 5. Verify CPI_Lag1 construction if available
# ------------------------------------------------------------

print("\n[5] CPI_Lag1 INTEGRITY")
print("-" * 90)

if "CPI_Lag1" in safe_ar_df.columns:

    lag_df = safe_ar_df.copy()

    lag_dates = pd.to_datetime(
        lag_df["forecast_origin_month"]
    )

    lag_target = pd.to_numeric(
        lag_df["cpi_combined_yoy_t_plus_1"],
        errors="coerce"
    )

    lag_value = pd.to_numeric(
        lag_df["CPI_Lag1"],
        errors="coerce"
    )

    print("Rows:", len(lag_df))
    print("CPI_Lag1 missing:", int(lag_value.isna().sum()))

    print(
        "First forecast-origin:",
        lag_dates.min()
    )

    print(
        "Last forecast-origin:",
        lag_dates.max()
    )

    # CPI_Lag1 at t should correspond to the CPI target
    # associated with the previous forecast-origin.
    #
    # Therefore, compare CPI_Lag1[t] with
    # target[t-1].

    expected_lag = lag_target.shift(1)

    valid_lag = (
        lag_value.notna()
        & expected_lag.notna()
    )

    if valid_lag.any():

        lag_difference = (
            lag_value[valid_lag].reset_index(drop=True)
            -
            expected_lag[valid_lag].reset_index(drop=True)
        )

        print(
            "Comparable CPI_Lag1 rows:",
            int(valid_lag.sum())
        )

        print(
            "Maximum absolute difference:",
            float(np.abs(lag_difference).max())
        )

        print(
            "Mean absolute difference:",
            float(np.abs(lag_difference).mean())
        )

        print(
            "CPI_Lag1 matches previous target:",
            bool(
                np.allclose(
                    lag_value[valid_lag].values,
                    expected_lag[valid_lag].values,
                    equal_nan=False
                )
            )
        )

else:

    print("ERROR: safe_ar_df does not contain CPI_Lag1")

# ------------------------------------------------------------
# 6. Publication-safe feature check
# ------------------------------------------------------------

print("\n[6] PUBLICATION-SAFE FINAL FEATURE CHECK")
print("-" * 90)

safe_features = list(X_safe_ar.columns)

print("Final feature count:", len(safe_features))
print("Final features:")

for i, feature in enumerate(safe_features, start=1):
    print(f"{i:2d}. {feature}")

print(
    "\nCPI_Current present:",
    "CPI_Current" in safe_features
)

print(
    "CPI_Lag1 present:",
    "CPI_Lag1" in safe_features
)

print("\n" + "=" * 90)
print("CELL 2 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 2: TARGET + FORECAST-HORIZON INTEGRITY

[1] TARGET COLUMN
------------------------------------------------------------------------------------------
Target column: cpi_combined_yoy_t_plus_1
Target exists in df: False


KeyError: 'cpi_combined_yoy_t_plus_1'

In [52]:
# ============================================================
# TASK 6.0 — CELL 2A
# CURRENT NOTEBOOK-STATE / TARGET LOCATION CHECK
# ============================================================

import pandas as pd

print("=" * 90)
print("TASK 6.0 — CELL 2A: CURRENT NOTEBOOK STATE CHECK")
print("=" * 90)

# ------------------------------------------------------------
# 1. Inspect current df
# ------------------------------------------------------------

print("\n[1] CURRENT 'df' OBJECT")
print("-" * 90)

print("df type:", type(df).__name__)
print("df shape:", getattr(df, "shape", "N/A"))
print("df columns:")
print(list(df.columns) if isinstance(df, pd.DataFrame) else "NOT A DATAFRAME")

# ------------------------------------------------------------
# 2. Check all existing DataFrames for the authoritative target
# ------------------------------------------------------------

print("\n[2] DATAFRAMES CONTAINING THE V2 TARGET")
print("-" * 90)

target_col = "cpi_combined_yoy_t_plus_1"

global_items = list(globals().items())

target_dfs = []

for name, obj in global_items:
    if isinstance(obj, pd.DataFrame):
        if target_col in obj.columns:
            target_dfs.append(name)
            print(
                f"{name}: shape={obj.shape}, "
                f"target_present=True"
            )

if not target_dfs:
    print("NO DATAFRAME CURRENTLY CONTAINS THE V2 TARGET.")

# ------------------------------------------------------------
# 3. Check likely authoritative V2 objects
# ------------------------------------------------------------

print("\n[3] LIKELY V2 DATAFRAME CHECK")
print("-" * 90)

for name in [
    "df",
    "model_df",
    "safe_ar_df",
    "cpi_lag_df",
    "original_target"
]:
    obj = globals().get(name, None)

    if isinstance(obj, pd.DataFrame):

        print(f"\n>>> {name}")
        print("Shape:", obj.shape)
        print("Has forecast_origin_month:",
              "forecast_origin_month" in obj.columns)
        print("Has target:",
              target_col in obj.columns)
        print("Has CPI_Lag1:",
              "CPI_Lag1" in obj.columns)
        print("First columns:",
              list(obj.columns[:5]))

# ------------------------------------------------------------
# 4. Inspect target-related objects without changing them
# ------------------------------------------------------------

print("\n[4] TARGET-RELATED OBJECTS")
print("-" * 90)

for name in target_dfs:

    obj = globals()[name]

    print(f"\n>>> {name}")

    print("Rows:", len(obj))
    print(
        "Target missing:",
        int(obj[target_col].isna().sum())
    )

    if "forecast_origin_month" in obj.columns:

        dates = pd.to_datetime(
            obj["forecast_origin_month"],
            errors="coerce"
        )

        print("Date start:", dates.min())
        print("Date end:", dates.max())
        print(
            "Duplicate dates:",
            int(dates.duplicated().sum())
        )
        print(
            "Strictly increasing:",
            dates.is_monotonic_increasing
        )

# ------------------------------------------------------------
# 5. Final diagnosis
# ------------------------------------------------------------

print("\n[5] DIAGNOSIS")
print("-" * 90)

if target_col in df.columns:

    print(
        "STATUS: Current df contains the target. "
        "Cell 2 can proceed using df."
    )

elif target_dfs:

    print(
        "STATUS: Current df does NOT contain the target, "
        "but another existing dataframe does."
    )

    print(
        "Authoritative target-containing dataframe(s):",
        target_dfs
    )

else:

    print(
        "STATUS: Target not found in any current dataframe."
    )

print("\n" + "=" * 90)
print("CELL 2A COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 2A: CURRENT NOTEBOOK STATE CHECK

[1] CURRENT 'df' OBJECT
------------------------------------------------------------------------------------------
df type: DataFrame
df shape: (3, 4)
df columns:
['Movement_Size', 'Naive_Mean_AbsError', 'ElasticNet_Mean_AbsError', 'Winner']

[2] DATAFRAMES CONTAINING THE V2 TARGET
------------------------------------------------------------------------------------------
train: shape=(103, 16), target_present=True
validation: shape=(22, 28), target_present=True
test: shape=(23, 16), target_present=True
val: shape=(22, 16), target_present=True
part: shape=(23, 16), target_present=True
model_df: shape=(148, 16), target_present=True
ar_df: shape=(148, 17), target_present=True
ar_model_df: shape=(147, 17), target_present=True
timing_check: shape=(147, 5), target_present=True
valid_checks: shape=(147, 5), target_present=True
original_check: shape=(148, 3), target_present=True
ar_check: shape=(147, 5), target_present=True
original_target: sha

In [53]:
# ============================================================
# TASK 6.0 — CELL 2B
# FINAL TARGET + FORECAST-HORIZON INTEGRITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 2B: FINAL TARGET + FORECAST-HORIZON INTEGRITY")
print("=" * 90)

# ------------------------------------------------------------
# Use the publication-safe V2 dataset
# ------------------------------------------------------------

audit_df = safe_ar_df.copy()

target_col = "cpi_combined_yoy_t_plus_1"
date_col = "forecast_origin_month"

dates = pd.to_datetime(audit_df[date_col])
target = pd.to_numeric(audit_df[target_col], errors="coerce")
lag1 = pd.to_numeric(audit_df["CPI_Lag1"], errors="coerce")

# ------------------------------------------------------------
# 1. Basic target integrity
# ------------------------------------------------------------

print("\n[1] TARGET INTEGRITY")
print("-" * 90)

print("Dataset:", "safe_ar_df")
print("Rows:", len(audit_df))
print("Target column:", target_col)
print("Target exists:", target_col in audit_df.columns)
print("Target missing:", int(target.isna().sum()))
print("Target dtype:", audit_df[target_col].dtype)

# ------------------------------------------------------------
# 2. Forecast-origin integrity
# ------------------------------------------------------------

print("\n[2] FORECAST-ORIGIN INTEGRITY")
print("-" * 90)

print("Start:", dates.min())
print("End:", dates.max())
print("Duplicate dates:", int(dates.duplicated().sum()))
print("Strictly increasing:", dates.is_monotonic_increasing)

periods = dates.dt.to_period("M")

consecutive_months = (
    periods.iloc[1:].reset_index(drop=True)
    ==
    (periods.iloc[:-1].reset_index(drop=True) + 1)
)

print(
    "Consecutive monthly rows:",
    int(consecutive_months.sum()),
    "/",
    len(consecutive_months)
)

print(
    "All rows monthly-contiguous:",
    bool(consecutive_months.all())
)

# ------------------------------------------------------------
# 3. Explicit t -> t+1 target alignment
# ------------------------------------------------------------

print("\n[3] t -> t+1 TARGET ALIGNMENT")
print("-" * 90)

# For a dataset where each row represents:
# origin month t -> target month t+1,
# the target attached to row t should equal the
# underlying CPI value at the next chronological row.

next_target = target.shift(-1)

alignment = pd.DataFrame({
    "origin": dates,
    "target_t_plus_1": target,
    "next_row_target": next_target
})

valid = next_target.notna()

print("Comparable rows:", int(valid.sum()))

# This verifies that the chronological structure is valid.
# The target itself is intentionally NOT expected to equal
# the next row's target.
print(
    "Target column has complete t+1 values:",
    int(target.notna().sum()) == len(target)
)

# ------------------------------------------------------------
# 4. Verify CPI_Lag1 meaning
# ------------------------------------------------------------

print("\n[4] CPI_Lag1 ALIGNMENT")
print("-" * 90)

print("CPI_Lag1 exists:", "CPI_Lag1" in audit_df.columns)
print("CPI_Lag1 missing:", int(lag1.isna().sum()))

# In the publication-safe setup, CPI_Lag1 at forecast
# origin t should represent the CPI value available from
# the previous target month.

# Compare lag values against the previous target where
# the relationship is defined.
previous_target = target.shift(1)

lag_valid = lag1.notna() & previous_target.notna()

print(
    "Comparable CPI_Lag1 rows:",
    int(lag_valid.sum())
)

if lag_valid.any():

    difference = (
        lag1[lag_valid].reset_index(drop=True)
        -
        previous_target[lag_valid].reset_index(drop=True)
    )

    print(
        "Maximum |CPI_Lag1 - previous target|:",
        float(np.abs(difference).max())
    )

    print(
        "Mean |CPI_Lag1 - previous target|:",
        float(np.abs(difference).mean())
    )

    print(
        "CPI_Lag1 equals previous target:",
        bool(np.allclose(
            lag1[lag_valid].values,
            previous_target[lag_valid].values,
            equal_nan=False
        ))
    )

# ------------------------------------------------------------
# 5. Check first-row behavior
# ------------------------------------------------------------

print("\n[5] FIRST-ROW LAG BEHAVIOR")
print("-" * 90)

print("First origin:", dates.iloc[0])
print("First CPI_Lag1:", lag1.iloc[0])
print("First lag is missing:", pd.isna(lag1.iloc[0]))

# ------------------------------------------------------------
# 6. Publication-safe feature set
# ------------------------------------------------------------

print("\n[6] FINAL PUBLICATION-SAFE FEATURES")
print("-" * 90)

final_features = list(X_safe_ar.columns)

print("Feature count:", len(final_features))

for i, feature in enumerate(final_features, start=1):
    print(f"{i:2d}. {feature}")

print("\nCPI_Current included:",
      "CPI_Current" in final_features)

print("CPI_Lag1 included:",
      "CPI_Lag1" in final_features)

# ------------------------------------------------------------
# 7. Final preliminary verdict
# ------------------------------------------------------------

print("\n[7] PRELIMINARY VERDICT")
print("-" * 90)

checks = {
    "Target exists": target_col in audit_df.columns,
    "Target complete": target.notna().all(),
    "Dates unique": dates.duplicated().sum() == 0,
    "Dates increasing": dates.is_monotonic_increasing,
    "Monthly continuity": bool(consecutive_months.all()),
    "CPI_Lag1 exists": "CPI_Lag1" in audit_df.columns,
    "CPI_Current excluded": "CPI_Current" not in final_features,
    "CPI_Lag1 included": "CPI_Lag1" in final_features
}

for name, result in checks.items():
    print(f"{name}: {'PASS' if result else 'FAIL'}")

print("\n" + "=" * 90)
print("CELL 2B COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 2B: FINAL TARGET + FORECAST-HORIZON INTEGRITY

[1] TARGET INTEGRITY
------------------------------------------------------------------------------------------
Dataset: safe_ar_df
Rows: 147
Target column: cpi_combined_yoy_t_plus_1
Target exists: True
Target missing: 0
Target dtype: float64

[2] FORECAST-ORIGIN INTEGRITY
------------------------------------------------------------------------------------------
Start: 2014-01-01 00:00:00
End: 2026-03-01 00:00:00
Duplicate dates: 0
Strictly increasing: True
Consecutive monthly rows: 146 / 146
All rows monthly-contiguous: True

[3] t -> t+1 TARGET ALIGNMENT
------------------------------------------------------------------------------------------
Comparable rows: 146
Target column has complete t+1 values: True

[4] CPI_Lag1 ALIGNMENT
------------------------------------------------------------------------------------------
CPI_Lag1 exists: True
CPI_Lag1 missing: 0
Comparable CPI_Lag1 rows: 146
Maximum |CPI_Lag1 - previous ta

In [54]:
# ============================================================
# TASK 6.0 — CELL 3
# FEATURE + LEAKAGE INTEGRITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 3: FEATURE + LEAKAGE INTEGRITY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Final publication-safe feature set
# ------------------------------------------------------------

final_features = list(X_safe_ar.columns)

print("\n[1] FINAL FEATURE SET")
print("-" * 90)

print("Number of features:", len(final_features))

for i, feature in enumerate(final_features, start=1):
    print(f"{i:2d}. {feature}")

# ------------------------------------------------------------
# 2. Explicit forbidden-feature check
# ------------------------------------------------------------

print("\n[2] FORBIDDEN / UNSAFE FEATURE CHECK")
print("-" * 90)

forbidden_features = [
    "CPI_Current",
    "cpi_combined_yoy_t_plus_1",
    "Target",
    "target"
]

forbidden_present = [
    feature for feature in final_features
    if feature in forbidden_features
]

print("Forbidden features found:", forbidden_present)

print(
    "CPI_Current present:",
    "CPI_Current" in final_features
)

print(
    "Target present as feature:",
    "cpi_combined_yoy_t_plus_1" in final_features
)

# ------------------------------------------------------------
# 3. Feature provenance classification
# ------------------------------------------------------------

print("\n[3] FEATURE PROVENANCE CLASSIFICATION")
print("-" * 90)

feature_classes = {}

for feature in final_features:

    if feature == "CPI_Lag1":
        classification = "AUTOREGRESSIVE — PREVIOUS TARGET"

    elif "_t_minus_1" in feature or "_Lag1" in feature:
        classification = "LAGGED / PRIOR-PERIOD"

    elif "Rainfall_Lag1" in feature:
        classification = "LAGGED / PRIOR-PERIOD"

    elif feature in [
        "Petrol_MoM_Change_Pct",
        "Diesel_MoM_Change_Pct",
        "Brent_Log_MoM"
    ]:
        classification = "CURRENT-ORIGIN TRANSFORMATION"

    elif feature in [
        "Petrol_Monthly_Avg_Rs_per_Litre",
        "Diesel_Monthly_Avg_Rs_per_Litre",
        "Brent_Monthly_Avg_USD_per_Barrel",
        "Rainfall_Deviation_LPA_Pct",
        "Rainfall_Lag1_Available",
        "Inflation_Expectation_3M"
    ]:
        classification = "ORIGIN-MONTH EXOGENOUS"

    else:
        classification = "UNCLASSIFIED — REVIEW"

    feature_classes[feature] = classification
    print(f"{feature:<45} -> {classification}")

# ------------------------------------------------------------
# 4. Check target leakage through exact column overlap
# ------------------------------------------------------------

print("\n[4] TARGET / FEATURE COLUMN OVERLAP")
print("-" * 90)

target_col = "cpi_combined_yoy_t_plus_1"

print(
    "Target column:",
    target_col
)

print(
    "Target included in X_safe_ar:",
    target_col in X_safe_ar.columns
)

print(
    "X_safe_ar feature count:",
    X_safe_ar.shape[1]
)

# ------------------------------------------------------------
# 5. Check CPI_Lag1 against target chronology
# ------------------------------------------------------------

print("\n[5] CPI_Lag1 CHRONOLOGY CHECK")
print("-" * 90)

audit_df = safe_ar_df.copy()

dates = pd.to_datetime(
    audit_df["forecast_origin_month"]
)

target = pd.to_numeric(
    audit_df[target_col],
    errors="coerce"
)

lag1 = pd.to_numeric(
    audit_df["CPI_Lag1"],
    errors="coerce"
)

previous_target = target.shift(1)

valid = (
    lag1.notna()
    & previous_target.notna()
)

difference = (
    lag1[valid].reset_index(drop=True)
    -
    previous_target[valid].reset_index(drop=True)
)

print("Comparable observations:", int(valid.sum()))

print(
    "Maximum absolute difference:",
    float(np.abs(difference).max())
)

print(
    "CPI_Lag1 equals previous target:",
    bool(np.allclose(
        lag1[valid].values,
        previous_target[valid].values
    ))
)

# ------------------------------------------------------------
# 6. Check whether any final feature equals the target
# ------------------------------------------------------------

print("\n[6] DIRECT NUMERIC TARGET-COPY CHECK")
print("-" * 90)

direct_target_copies = []

for feature in final_features:

    feature_values = pd.to_numeric(
        audit_df[feature],
        errors="coerce"
    )

    comparison = (
        feature_values.reset_index(drop=True)
        ==
        target.reset_index(drop=True)
    )

    # Ignore rows where either side is missing
    comparable = (
        feature_values.notna().reset_index(drop=True)
        &
        target.notna().reset_index(drop=True)
    )

    if comparable.any():

        identical = bool(
            comparison[comparable].all()
        )

        if identical:
            direct_target_copies.append(feature)

print(
    "Features numerically identical to target:",
    direct_target_copies
)

# ------------------------------------------------------------
# 7. Check final X against safe_ar_df
# ------------------------------------------------------------

print("\n[7] FINAL X / SOURCE DATA CONSISTENCY")
print("-" * 90)

source_features = list(
    safe_ar_df[final_features].columns
)

print(
    "Feature order identical:",
    final_features == source_features
)

print(
    "X_safe_ar shape:",
    X_safe_ar.shape
)

print(
    "Source feature matrix shape:",
    safe_ar_df[final_features].shape
)

# Compare values
source_matrix = safe_ar_df[
    final_features
].reset_index(drop=True)

x_matrix = X_safe_ar.reset_index(drop=True)

try:

    values_match = np.allclose(
        x_matrix.astype(float).values,
        source_matrix.astype(float).values,
        equal_nan=True
    )

except Exception:

    values_match = False

print(
    "X_safe_ar values match source:",
    bool(values_match)
)

# ------------------------------------------------------------
# 8. Final leakage verdict
# ------------------------------------------------------------

print("\n[8] FINAL LEAKAGE VERDICT")
print("-" * 90)

checks = {
    "No CPI_Current": "CPI_Current" not in final_features,
    "No target column": target_col not in final_features,
    "CPI_Lag1 correctly aligned": bool(
        np.allclose(
            lag1[valid].values,
            previous_target[valid].values
        )
    ),
    "No direct target-copy feature": len(
        direct_target_copies
    ) == 0,
    "Final X matches source": bool(values_match),
    "All features classified": all(
        classification != "UNCLASSIFIED — REVIEW"
        for classification in feature_classes.values()
    )
}

for name, result in checks.items():
    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

print("\n" + "=" * 90)
print("CELL 3 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 3: FEATURE + LEAKAGE INTEGRITY

[1] FINAL FEATURE SET
------------------------------------------------------------------------------------------
Number of features: 15
 1. Petrol_Monthly_Avg_Rs_per_Litre
 2. Diesel_Monthly_Avg_Rs_per_Litre
 3. Petrol_MoM_Change_Pct
 4. Diesel_MoM_Change_Pct
 5. Brent_Monthly_Avg_USD_per_Barrel
 6. Brent_Log_MoM
 7. Rainfall_Deviation_LPA_Pct
 8. Rainfall_Deviation_LPA_Pct_Lag1
 9. Rainfall_Lag1_Available
10. WPI_All_Commodities_t_minus_1
11. WPI_MoM_Pct_t_minus_1
12. FAO_Food_Price_Index_t_minus_1
13. FAO_Food_Price_Index_MoM_t_minus_1
14. Inflation_Expectation_3M
15. CPI_Lag1

[2] FORBIDDEN / UNSAFE FEATURE CHECK
------------------------------------------------------------------------------------------
Forbidden features found: []
CPI_Current present: False
Target present as feature: False

[3] FEATURE PROVENANCE CLASSIFICATION
------------------------------------------------------------------------------------------
Petrol_Monthly_Avg

In [55]:
# ============================================================
# TASK 6.0 — CELL 4
# PUBLICATION-TIMING SAFETY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 4: PUBLICATION-TIMING SAFETY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Define final publication-safe features
# ------------------------------------------------------------

final_features = list(X_safe_ar.columns)

print("\n[1] FINAL FEATURE TIMING CLASSIFICATION")
print("-" * 90)

timing_map = {

    "Petrol_Monthly_Avg_Rs_per_Litre":
        "CURRENT ORIGIN MONTH",

    "Diesel_Monthly_Avg_Rs_per_Litre":
        "CURRENT ORIGIN MONTH",

    "Petrol_MoM_Change_Pct":
        "CURRENT ORIGIN MONTH",

    "Diesel_MoM_Change_Pct":
        "CURRENT ORIGIN MONTH",

    "Brent_Monthly_Avg_USD_per_Barrel":
        "CURRENT ORIGIN MONTH",

    "Brent_Log_MoM":
        "CURRENT ORIGIN MONTH",

    "Rainfall_Deviation_LPA_Pct":
        "CURRENT ORIGIN MONTH",

    "Rainfall_Deviation_LPA_Pct_Lag1":
        "PRIOR MONTH",

    "Rainfall_Lag1_Available":
        "PRIOR MONTH AVAILABILITY FLAG",

    "WPI_All_Commodities_t_minus_1":
        "PRIOR MONTH",

    "WPI_MoM_Pct_t_minus_1":
        "PRIOR MONTH",

    "FAO_Food_Price_Index_t_minus_1":
        "PRIOR MONTH",

    "FAO_Food_Price_Index_MoM_t_minus_1":
        "PRIOR MONTH",

    "Inflation_Expectation_3M":
        "ORIGIN-MONTH EXPECTATION",

    "CPI_Lag1":
        "PREVIOUS TARGET / PRIOR CPI"
}

for feature in final_features:

    print(
        f"{feature:<45} -> "
        f"{timing_map.get(feature, 'UNMAPPED')}"
    )

# ------------------------------------------------------------
# 2. Check that every final feature has a timing definition
# ------------------------------------------------------------

print("\n[2] TIMING MAP COMPLETENESS")
print("-" * 90)

unmapped = [
    feature for feature in final_features
    if feature not in timing_map
]

print("Unmapped features:", unmapped)

print(
    "All final features mapped:",
    len(unmapped) == 0
)

# ------------------------------------------------------------
# 3. Verify existing timing-check objects
# ------------------------------------------------------------

print("\n[3] EXISTING TIMING CHECKS")
print("-" * 90)

for name in [
    "timing_check",
    "valid_checks",
    "ar_check",
    "relationship_check"
]:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):

        print(f"\n>>> {name}")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))

        # Display useful boolean / difference columns
        bool_cols = [
            col for col in obj.columns
            if obj[col].dtype == bool
        ]

        if bool_cols:

            for col in bool_cols:

                print(
                    f"{col}:",
                    int(obj[col].sum()),
                    "/",
                    len(obj),
                    "TRUE"
                )

        numeric_difference_cols = [
            col for col in obj.columns
            if any(
                keyword in str(col).lower()
                for keyword in [
                    "difference",
                    "diff",
                    "error"
                ]
            )
        ]

        for col in numeric_difference_cols:

            series = pd.to_numeric(
                obj[col],
                errors="coerce"
            ).dropna()

            if len(series) > 0:

                print(
                    f"{col} max abs:",
                    float(np.abs(series).max())
                )

# ------------------------------------------------------------
# 4. Verify CPI timing relationship directly
# ------------------------------------------------------------

print("\n[4] CPI PUBLICATION-TIMING RELATIONSHIP")
print("-" * 90)

audit_df = safe_ar_df.copy()

dates = pd.to_datetime(
    audit_df["forecast_origin_month"]
)

target = pd.to_numeric(
    audit_df["cpi_combined_yoy_t_plus_1"],
    errors="coerce"
)

lag1 = pd.to_numeric(
    audit_df["CPI_Lag1"],
    errors="coerce"
)

previous_target = target.shift(1)

valid = (
    lag1.notna()
    & previous_target.notna()
)

lag_difference = (
    lag1[valid].reset_index(drop=True)
    -
    previous_target[valid].reset_index(drop=True)
)

print("Comparable observations:", int(valid.sum()))

print(
    "Maximum timing difference:",
    float(np.abs(lag_difference).max())
)

print(
    "CPI_Lag1 uses prior-period target:",
    bool(np.allclose(
        lag1[valid].values,
        previous_target[valid].values
    ))
)

# ------------------------------------------------------------
# 5. Verify final features don't contain explicit future
#    naming conventions
# ------------------------------------------------------------

print("\n[5] FUTURE-TIMING NAME CHECK")
print("-" * 90)

future_keywords = [
    "t_plus_1",
    "t+1",
    "future",
    "lead",
    "forward"
]

future_named_features = []

for feature in final_features:

    feature_lower = str(feature).lower()

    if any(
        keyword in feature_lower
        for keyword in future_keywords
    ):
        future_named_features.append(feature)

print(
    "Future-looking feature names:",
    future_named_features
)

# ------------------------------------------------------------
# 6. Check publication-safe exclusions
# ------------------------------------------------------------

print("\n[6] PUBLICATION-SAFE EXCLUSION CHECK")
print("-" * 90)

unsafe_candidates = [
    "CPI_Current",
    "CPI_t",
    "CPI_Target",
    "cpi_combined_yoy_t_plus_1"
]

present_unsafe = [
    feature for feature in final_features
    if feature in unsafe_candidates
]

print(
    "Explicitly unsafe candidates present:",
    present_unsafe
)

# ------------------------------------------------------------
# 7. Final timing verdict
# ------------------------------------------------------------

print("\n[7] FINAL PUBLICATION-TIMING VERDICT")
print("-" * 90)

checks = {

    "All features have timing definitions":
        len(unmapped) == 0,

    "No future-looking feature names":
        len(future_named_features) == 0,

    "No explicitly unsafe timing features":
        len(present_unsafe) == 0,

    "CPI_Lag1 uses prior target":
        bool(np.allclose(
            lag1[valid].values,
            previous_target[valid].values
        )),

    "Final feature count is 15":
        len(final_features) == 15
}

for name, result in checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

print("\n" + "=" * 90)
print("CELL 4 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 4: PUBLICATION-TIMING SAFETY

[1] FINAL FEATURE TIMING CLASSIFICATION
------------------------------------------------------------------------------------------
Petrol_Monthly_Avg_Rs_per_Litre               -> CURRENT ORIGIN MONTH
Diesel_Monthly_Avg_Rs_per_Litre               -> CURRENT ORIGIN MONTH
Petrol_MoM_Change_Pct                         -> CURRENT ORIGIN MONTH
Diesel_MoM_Change_Pct                         -> CURRENT ORIGIN MONTH
Brent_Monthly_Avg_USD_per_Barrel              -> CURRENT ORIGIN MONTH
Brent_Log_MoM                                 -> CURRENT ORIGIN MONTH
Rainfall_Deviation_LPA_Pct                    -> CURRENT ORIGIN MONTH
Rainfall_Deviation_LPA_Pct_Lag1               -> PRIOR MONTH
Rainfall_Lag1_Available                       -> PRIOR MONTH AVAILABILITY FLAG
WPI_All_Commodities_t_minus_1                 -> PRIOR MONTH
WPI_MoM_Pct_t_minus_1                         -> PRIOR MONTH
FAO_Food_Price_Index_t_minus_1                -> PRIOR MONTH
FAO_Food_P

In [59]:
# ============================================================
# TASK 6.0 — CELL 6
# EXPANDING-WINDOW CV INTEGRITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 6: EXPANDING-WINDOW CV INTEGRITY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Identify CV result objects
# ------------------------------------------------------------

print("\n[1] CV OBJECTS FOUND")
print("-" * 90)

for name in [
    "safe_cv_results_df",
    "safe_cv_summary",
    "fold_rmse"
]:
    obj = globals().get(name, None)

    if isinstance(obj, pd.DataFrame):
        print(f"{name}: shape={obj.shape}")
        print("Columns:", list(obj.columns))
    else:
        print(f"{name}: NOT FOUND / NOT A DATAFRAME")

# ------------------------------------------------------------
# 2. Inspect safe CV results
# ------------------------------------------------------------

print("\n[2] SAFE CV RESULTS")
print("-" * 90)

cv = safe_cv_results_df.copy()

print("Shape:", cv.shape)
print("Columns:", list(cv.columns))

print("\nComplete CV results:")
print(cv.to_string(index=False))

# ------------------------------------------------------------
# 3. Inspect CV summary
# ------------------------------------------------------------

print("\n[3] SAFE CV SUMMARY")
print("-" * 90)

if isinstance(
    globals().get("safe_cv_summary"),
    pd.DataFrame
):
    print(safe_cv_summary.to_string(index=False))
else:
    print("safe_cv_summary unavailable.")

# ------------------------------------------------------------
# 4. Inspect fold RMSE
# ------------------------------------------------------------

print("\n[4] FOLD RMSE")
print("-" * 90)

if isinstance(
    globals().get("fold_rmse"),
    pd.DataFrame
):
    print(fold_rmse.to_string(index=False))
else:
    print("fold_rmse unavailable.")

# ------------------------------------------------------------
# 5. Detect likely fold/date columns
# ------------------------------------------------------------

print("\n[5] FOLD / DATE COLUMN DETECTION")
print("-" * 90)

for col in cv.columns:
    print(
        f"{col:<35} dtype={cv[col].dtype}"
    )

# ------------------------------------------------------------
# 6. Test-period contamination check
# ------------------------------------------------------------

print("\n[6] TEST-PERIOD CONTAMINATION")
print("-" * 90)

test_start = pd.Timestamp("2024-05-01")
test_end = pd.Timestamp("2026-03-01")

test_date_hits = []

for col in cv.columns:

    col_lower = str(col).lower()

    if (
        "date" in col_lower
        or "month" in col_lower
        or "train" in col_lower
        or "valid" in col_lower
        or "val" in col_lower
    ):

        converted = pd.to_datetime(
            cv[col],
            errors="coerce"
        )

        if converted.notna().any():

            in_test = (
                (converted >= test_start)
                &
                (converted <= test_end)
            )

            if in_test.any():
                test_date_hits.append(
                    (col, int(in_test.sum()))
                )

print(
    "CV columns containing explicit test-period dates:",
    test_date_hits
)

print(
    "Explicit test-period contamination:",
    len(test_date_hits) > 0
)

# ------------------------------------------------------------
# 7. Try to identify train/validation boundary columns
# ------------------------------------------------------------

print("\n[7] FOLD CHRONOLOGY")
print("-" * 90)

train_start_col = None
train_end_col = None
val_start_col = None
val_end_col = None

for col in cv.columns:

    c = str(col).lower().replace(" ", "_")

    if "train" in c and "start" in c:
        train_start_col = col

    if "train" in c and "end" in c:
        train_end_col = col

    if (
        ("valid" in c or "val" in c)
        and "start" in c
    ):
        val_start_col = col

    if (
        ("valid" in c or "val" in c)
        and "end" in c
    ):
        val_end_col = col

print("Train start column:", train_start_col)
print("Train end column:", train_end_col)
print("Validation start column:", val_start_col)
print("Validation end column:", val_end_col)

chronology_verified = False
expanding_verified = False

if all([
    train_start_col is not None,
    train_end_col is not None,
    val_start_col is not None,
    val_end_col is not None
]):

    train_starts = pd.to_datetime(
        cv[train_start_col],
        errors="coerce"
    )

    train_ends = pd.to_datetime(
        cv[train_end_col],
        errors="coerce"
    )

    val_starts = pd.to_datetime(
        cv[val_start_col],
        errors="coerce"
    )

    val_ends = pd.to_datetime(
        cv[val_end_col],
        errors="coerce"
    )

    chronology = (
        train_ends < val_starts
    )

    chronology_verified = bool(
        chronology.all()
    )

    print(
        "Train end < validation start:",
        int(chronology.sum()),
        "/",
        len(cv)
    )

    print(
        "All folds chronologically valid:",
        chronology_verified
    )

    # Expanding-window condition
    if len(train_ends) > 1:

        train_end_values = train_ends.reset_index(
            drop=True
        )

        expanding = (
            train_end_values.iloc[1:].values
            >=
            train_end_values.iloc[:-1].values
        )

        expanding_verified = bool(
            expanding.all()
        )

        print(
            "Training-end dates:",
            train_end_values.tolist()
        )

        print(
            "Training window expands/non-decreases:",
            expanding_verified
        )

else:

    print(
        "Fold boundary columns could not be automatically identified."
    )

# ------------------------------------------------------------
# 8. Fold-number integrity
# ------------------------------------------------------------

print("\n[8] FOLD NUMBER / ORDER")
print("-" * 90)

fold_col = None

for candidate in ["fold", "Fold", "fold_id", "Fold_ID"]:

    if candidate in cv.columns:
        fold_col = candidate
        break

if fold_col is not None:

    fold_values = cv[fold_col]

    print("Fold column:", fold_col)
    print("Fold values:", fold_values.tolist())
    print("Unique folds:", fold_values.nunique())
    print("Rows:", len(cv))

else:

    print(
        "No explicit fold-number column detected."
    )

# ------------------------------------------------------------
# 9. Compare CV end date with frozen test start
# ------------------------------------------------------------

print("\n[9] CV / TEST BOUNDARY")
print("-" * 90)

if val_end_col is not None:

    cv_latest_validation = pd.to_datetime(
        cv[val_end_col],
        errors="coerce"
    ).max()

    print(
        "Latest CV validation date:",
        cv_latest_validation
    )

    print(
        "Frozen test start:",
        test_start
    )

    print(
        "CV validation ends before test:",
        cv_latest_validation < test_start
    )

else:

    print(
        "Could not identify CV validation-end column."
    )

# ------------------------------------------------------------
# 10. Final CV integrity verdict
# ------------------------------------------------------------

print("\n[10] FINAL CV INTEGRITY VERDICT")
print("-" * 90)

print(
    "CV results object exists:",
    isinstance(cv, pd.DataFrame)
)

print(
    "CV contains observations:",
    len(cv) > 0
)

print(
    "No explicit test-period dates:",
    len(test_date_hits) == 0
)

if chronology_verified:
    print(
        "Train precedes validation in every fold: PASS"
    )
else:
    print(
        "Train precedes validation in every fold: NOT VERIFIED"
    )

if expanding_verified:
    print(
        "Expanding training window: PASS"
    )
else:
    print(
        "Expanding training window: NOT VERIFIED"
    )

print("\n" + "=" * 90)
print("CELL 6 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 6: EXPANDING-WINDOW CV INTEGRITY

[1] CV OBJECTS FOUND
------------------------------------------------------------------------------------------
safe_cv_results_df: shape=(40, 7)
Columns: ['Fold', 'Validation_Start', 'Validation_End', 'Model', 'RMSE', 'MAE', 'R2']
safe_cv_summary: shape=(4, 7)
Columns: ['Model', 'Mean_RMSE', 'Median_RMSE', 'Std_RMSE', 'Mean_MAE', 'Mean_R2', 'Folds']
fold_rmse: shape=(10, 6)
Columns: ['Fold', 'ElasticNet', 'Naive', 'RMSE_Difference', 'ElasticNet_Improvement_Pct', 'Winner']

[2] SAFE CV RESULTS
------------------------------------------------------------------------------------------
Shape: (40, 7)
Columns: ['Fold', 'Validation_Start', 'Validation_End', 'Model', 'RMSE', 'MAE', 'R2']

Complete CV results:
 Fold Validation_Start Validation_End                Model     RMSE      MAE        R2
    1       2019-01-01     2019-06-01                Naive 0.281631 0.205699 -0.855198
    1       2019-01-01     2019-06-01           ElasticNet 0.30

In [60]:
# ============================================================
# TASK 6.0 — CELL 6A
# RECOVER EXPANDING-WINDOW CV FOLD STRUCTURE
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 6A: CV FOLD STRUCTURE RECOVERY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Find CV-related variables
# ------------------------------------------------------------

print("\n[1] CV-RELATED VARIABLES")
print("-" * 90)

keywords = [
    "cv",
    "fold",
    "split",
    "tscv",
    "time",
    "window",
    "train_idx",
    "val_idx"
]

global_items = list(globals().items())

found = []

for name, obj in global_items:

    name_lower = str(name).lower()

    if any(k in name_lower for k in keywords):

        if isinstance(obj, (pd.DataFrame, pd.Series, np.ndarray, list, tuple, dict)):

            found.append(name)

            if isinstance(obj, pd.DataFrame):
                description = f"DataFrame {obj.shape}"

            elif isinstance(obj, pd.Series):
                description = f"Series {obj.shape}"

            elif isinstance(obj, np.ndarray):
                description = f"ndarray {obj.shape}"

            elif isinstance(obj, dict):
                description = f"dict ({len(obj)} keys)"

            else:
                try:
                    description = f"{type(obj).__name__} (len={len(obj)})"
                except:
                    description = type(obj).__name__

            print(f"{name:<40} -> {description}")

# ------------------------------------------------------------
# 2. Inspect likely splitters
# ------------------------------------------------------------

print("\n[2] LIKELY CV SPLITTER OBJECTS")
print("-" * 90)

for name in found:

    obj = globals().get(name)

    class_name = type(obj).__name__

    if any(
        keyword in class_name.lower()
        for keyword in [
            "split",
            "fold",
            "timeseri",
            "cv"
        ]
    ):

        print(
            f"{name}: type={class_name}"
        )

        if hasattr(obj, "n_splits"):
            print(
                "  n_splits:",
                obj.n_splits
            )

        if hasattr(obj, "test_size"):
            print(
                "  test_size:",
                obj.test_size
            )

        if hasattr(obj, "max_train_size"):
            print(
                "  max_train_size:",
                obj.max_train_size
            )

# ------------------------------------------------------------
# 3. Inspect arrays/lists that may contain CV indices
# ------------------------------------------------------------

print("\n[3] INDEX / SPLIT STRUCTURES")
print("-" * 90)

for name in found:

    obj = globals().get(name)

    if isinstance(obj, (list, tuple)):

        if len(obj) > 0:

            first = obj[0]

            if isinstance(first, (tuple, list)) and len(first) == 2:

                print(
                    f"{name}: appears to contain "
                    f"{len(obj)} train/validation pairs"
                )

                print(
                    "  First element types:",
                    type(first[0]).__name__,
                    type(first[1]).__name__
                )

                try:
                    print(
                        "  First train length:",
                        len(first[0])
                    )

                    print(
                        "  First validation length:",
                        len(first[1])
                    )
                except:
                    pass

# ------------------------------------------------------------
# 4. Look specifically for TimeSeriesSplit
# ------------------------------------------------------------

print("\n[4] TIMESERIES SPLIT OBJECTS")
print("-" * 90)

for name, obj in global_items:

    class_name = type(obj).__name__

    if "TimeSeriesSplit" in class_name:

        print(
            f"FOUND: {name}"
        )

        print(
            "Type:",
            class_name
        )

        if hasattr(obj, "n_splits"):
            print(
                "n_splits:",
                obj.n_splits
            )

        if hasattr(obj, "test_size"):
            print(
                "test_size:",
                obj.test_size
            )

        if hasattr(obj, "max_train_size"):
            print(
                "max_train_size:",
                obj.max_train_size
            )

# ------------------------------------------------------------
# 5. Inspect CV results validation chronology
# ------------------------------------------------------------

print("\n[5] VALIDATION FOLD CHRONOLOGY")
print("-" * 90)

cv = safe_cv_results_df.copy()

fold_summary = (
    cv.groupby("Fold")
      .agg(
          Validation_Start=("Validation_Start", "min"),
          Validation_End=("Validation_End", "max"),
          Models=("Model", "count")
      )
      .reset_index()
)

print(
    fold_summary.to_string(index=False)
)

print("\nNumber of folds:", len(fold_summary))

# ------------------------------------------------------------
# 6. Check validation windows themselves
# ------------------------------------------------------------

print("\n[6] VALIDATION WINDOW INTEGRITY")
print("-" * 90)

validation_starts = pd.to_datetime(
    fold_summary["Validation_Start"]
)

validation_ends = pd.to_datetime(
    fold_summary["Validation_End"]
)

print(
    "Validation starts strictly increasing:",
    validation_starts.is_monotonic_increasing
)

print(
    "Validation ends strictly increasing:",
    validation_ends.is_monotonic_increasing
)

validation_lengths = (
    validation_ends.dt.to_period("M")
    -
    validation_starts.dt.to_period("M")
)

print(
    "Validation window lengths:",
    validation_lengths.tolist()
)

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n[7] CELL 6A STATUS")
print("-" * 90)

print(
    "CV result structure recovered:",
    True
)

print(
    "Validation chronology verified:",
    validation_starts.is_monotonic_increasing
    and validation_ends.is_monotonic_increasing
)

print(
    "Training-window expansion:",
    "REQUIRES TRAIN-BOUNDARY OBJECT/CODE"
)

print("\n" + "=" * 90)
print("CELL 6A COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 6A: CV FOLD STRUCTURE RECOVERY

[1] CV-RELATED VARIABLES
------------------------------------------------------------------------------------------
X_cv                                     -> DataFrame (124, 15)
y_cv                                     -> Series (124,)
fold_results                             -> list (len=30)
results_cv                               -> DataFrame (30, 8)
summary_cv                               -> DataFrame (3, 7)
X_ar_cv                                  -> DataFrame (124, 15)
y_ar_cv                                  -> Series (124,)
ar_fold_results                          -> list (len=30)
ar_cv_results                            -> DataFrame (30, 8)
ar_cv_summary                            -> DataFrame (3, 7)
fold_rmses                               -> list (len=10)
fold_maes                                -> list (len=10)
X_full_cv                                -> DataFrame (124, 15)
y_full_cv                                -> Series

In [61]:
# ============================================================
# TASK 6.0 — CELL 6B
# ACTUAL FINAL CV FOLD-BOUNDARY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 6B: ACTUAL FINAL CV FOLD BOUNDARIES")
print("=" * 90)

# ------------------------------------------------------------
# 1. Inspect final fold tables
# ------------------------------------------------------------

print("\n[1] FINAL CV FOLD TABLE OBJECTS")
print("-" * 90)

for name in ["exo_fold_df", "ar_fold_df", "fold_table"]:

    obj = globals().get(name, None)

    if isinstance(obj, pd.DataFrame):

        print(f"\n>>> {name}")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))
        print(obj.to_string(index=False))

    else:

        print(f"{name}: NOT FOUND / NOT A DATAFRAME")

# ------------------------------------------------------------
# 2. Inspect actual fold-list structures
# ------------------------------------------------------------

print("\n[2] FINAL CV FOLD LIST STRUCTURES")
print("-" * 90)

for name in ["exo_folds", "ar_folds"]:

    folds = globals().get(name, None)

    if isinstance(folds, list):

        print(f"\n>>> {name}")
        print("Number of folds:", len(folds))

        for i, fold in enumerate(folds[:10], start=1):

            print(
                f"\nFold {i}: type={type(fold).__name__}"
            )

            try:
                print(
                    "Length:",
                    len(fold)
                )
            except:
                pass

            if isinstance(fold, (tuple, list)):

                print(
                    "Elements:",
                    len(fold)
                )

                for j, element in enumerate(fold):

                    try:
                        print(
                            f"  element {j}: "
                            f"type={type(element).__name__}, "
                            f"length={len(element)}"
                        )
                    except:
                        print(
                            f"  element {j}: "
                            f"type={type(element).__name__}"
                        )

# ------------------------------------------------------------
# 3. Extract fold-table candidates
# ------------------------------------------------------------

print("\n[3] AUTOMATIC FOLD-BOUNDARY DETECTION")
print("-" * 90)

candidate_tables = []

for name in [
    "exo_fold_df",
    "ar_fold_df",
    "fold_table"
]:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):

        candidate_tables.append(
            (name, obj)
        )

for name, table in candidate_tables:

    print(f"\n>>> {name}")

    date_columns = []

    for col in table.columns:

        converted = pd.to_datetime(
            table[col],
            errors="coerce"
        )

        if converted.notna().sum() == len(table):

            date_columns.append(col)

    print(
        "Date-like columns:",
        date_columns
    )

    numeric_columns = [
        col for col in table.columns
        if pd.api.types.is_numeric_dtype(
            table[col]
        )
    ]

    print(
        "Numeric columns:",
        numeric_columns
    )

# ------------------------------------------------------------
# 4. Explicitly inspect fold_table
# ------------------------------------------------------------

print("\n[4] PRIMARY FOLD TABLE")
print("-" * 90)

if isinstance(
    globals().get("fold_table"),
    pd.DataFrame
):

    ft = fold_table.copy()

    print(
        ft.to_string(index=False)
    )

    print("\nColumn dtypes:")
    print(ft.dtypes)

else:

    print("fold_table unavailable.")

# ------------------------------------------------------------
# 5. Check whether fold boundaries expand
# ------------------------------------------------------------

print("\n[5] EXPANDING-WINDOW CHECK FROM FOLD TABLE")
print("-" * 90)

# Try to identify columns containing train/end/start information.

if isinstance(
    globals().get("fold_table"),
    pd.DataFrame
):

    ft = fold_table.copy()

    train_start_col = None
    train_end_col = None
    val_start_col = None
    val_end_col = None

    for col in ft.columns:

        c = str(col).lower().replace(
            " ", "_"
        )

        if (
            "train" in c
            and "start" in c
        ):
            train_start_col = col

        if (
            "train" in c
            and "end" in c
        ):
            train_end_col = col

        if (
            ("valid" in c or "val" in c)
            and "start" in c
        ):
            val_start_col = col

        if (
            ("valid" in c or "val" in c)
            and "end" in c
        ):
            val_end_col = col

    print(
        "Train start:",
        train_start_col
    )

    print(
        "Train end:",
        train_end_col
    )

    print(
        "Validation start:",
        val_start_col
    )

    print(
        "Validation end:",
        val_end_col
    )

    if train_end_col is not None:

        train_end = pd.to_datetime(
            ft[train_end_col],
            errors="coerce"
        )

        print(
            "\nTraining-end dates:",
            train_end.tolist()
        )

        if train_end.notna().all() and len(train_end) > 1:

            expanding = (
                train_end.iloc[1:].reset_index(
                    drop=True
                )
                >=
                train_end.iloc[:-1].reset_index(
                    drop=True
                )
            )

            print(
                "Training end non-decreasing:",
                bool(expanding.all())
            )

    if (
        train_end_col is not None
        and val_start_col is not None
    ):

        train_end = pd.to_datetime(
            ft[train_end_col],
            errors="coerce"
        )

        val_start = pd.to_datetime(
            ft[val_start_col],
            errors="coerce"
        )

        chronology = train_end < val_start

        print(
            "Train end < validation start:",
            int(chronology.sum()),
            "/",
            len(ft)
        )

        print(
            "All folds chronological:",
            bool(chronology.all())
        )

# ------------------------------------------------------------
# 6. Check final CV validation periods
# ------------------------------------------------------------

print("\n[6] FINAL CV VALIDATION WINDOWS")
print("-" * 90)

cv = safe_cv_results_df.copy()

validation_summary = (
    cv.groupby("Fold")
      .agg(
          Validation_Start=("Validation_Start", "min"),
          Validation_End=("Validation_End", "max"),
          Model_Count=("Model", "count")
      )
      .reset_index()
)

print(
    validation_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n[7] CELL 6B STATUS")
print("-" * 90)

print(
    "Final safe CV has 10 folds:",
    len(validation_summary) == 10
)

print(
    "Each fold has 4 models:",
    bool(
        (validation_summary["Model_Count"] == 4).all()
    )
)

print(
    "Final CV validation ends before test:",
    pd.to_datetime(
        validation_summary["Validation_End"]
    ).max()
    < pd.Timestamp("2024-05-01")
)

print("\n" + "=" * 90)
print("CELL 6B COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 6B: ACTUAL FINAL CV FOLD BOUNDARIES

[1] FINAL CV FOLD TABLE OBJECTS
------------------------------------------------------------------------------------------

>>> exo_fold_df
Shape: (10, 5)
Columns: ['Fold', 'Train_Start', 'Train_End', 'Validation_Start', 'Validation_End']
 Fold Train_Start  Train_End Validation_Start Validation_End
    1  2013-12-01 2018-11-01       2018-12-01     2019-05-01
    2  2013-12-01 2019-05-01       2019-06-01     2019-11-01
    3  2013-12-01 2019-11-01       2019-12-01     2020-05-01
    4  2013-12-01 2020-05-01       2020-06-01     2020-11-01
    5  2013-12-01 2020-11-01       2020-12-01     2021-05-01
    6  2013-12-01 2021-05-01       2021-06-01     2021-11-01
    7  2013-12-01 2021-11-01       2021-12-01     2022-05-01
    8  2013-12-01 2022-05-01       2022-06-01     2022-11-01
    9  2013-12-01 2022-11-01       2022-12-01     2023-05-01
   10  2013-12-01 2023-05-01       2023-06-01     2023-11-01

>>> ar_fold_df
Shape: (10, 5)
Column

/tmp/ipykernel_509/4005663228.py:117: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(


In [62]:
# ============================================================
# TASK 6.0 — CELL 7
# MODEL-FREEZE + TEST-PRESERVATION AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 7: MODEL-FREEZE + TEST-PRESERVATION AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Frozen model specification
# ------------------------------------------------------------

print("\n[1] FROZEN FINAL MODEL SPECIFICATION")
print("-" * 90)

print("Primary benchmark: Naive (CPI_Lag1)")
print("Primary ML model: ElasticNet")
print("Frozen alpha:", 0.30)
print("Frozen l1_ratio:", 0.90)
print("Secondary models: Random Forest, HistGradientBoosting")

# ------------------------------------------------------------
# 2. Inspect final model objects
# ------------------------------------------------------------

print("\n[2] FINAL MODEL OBJECTS")
print("-" * 90)

model_names = [
    "elastic_net_final",
    "elasticnet_final",
    "final_elasticnet",
    "rf_final",
    "random_forest_final",
    "hgb_final",
    "hist_gradient_final",
    "final_models",
    "models"
]

for name in model_names:

    obj = globals().get(name, None)

    if obj is not None:

        print(
            f"{name}: type={type(obj).__name__}"
        )

        if hasattr(obj, "alpha"):
            print("  alpha:", obj.alpha)

        if hasattr(obj, "l1_ratio"):
            print("  l1_ratio:", obj.l1_ratio)

        if hasattr(obj, "random_state"):
            print("  random_state:", obj.random_state)

# ------------------------------------------------------------
# 3. Search all model-like objects for ElasticNet parameters
# ------------------------------------------------------------

print("\n[3] ELASTICNET PARAMETER OBJECTS")
print("-" * 90)

global_items = list(globals().items())

elastic_objects = []

for name, obj in global_items:

    class_name = type(obj).__name__.lower()

    if (
        "elasticnet" in class_name
        or "elastic_net" in str(name).lower()
    ):

        if hasattr(obj, "alpha") and hasattr(
            obj, "l1_ratio"
        ):

            elastic_objects.append(
                (name, obj)
            )

            print(
                f"{name}: "
                f"alpha={obj.alpha}, "
                f"l1_ratio={obj.l1_ratio}"
            )

# ------------------------------------------------------------
# 4. Test-period variables
# ------------------------------------------------------------

print("\n[4] TEST-RELATED OBJECTS")
print("-" * 90)

for name, obj in global_items:

    name_lower = str(name).lower()

    if "test" in name_lower:

        if isinstance(obj, pd.DataFrame):

            print(
                f"{name}: DataFrame {obj.shape}"
            )

        elif isinstance(obj, pd.Series):

            print(
                f"{name}: Series {obj.shape}"
            )

        elif isinstance(obj, np.ndarray):

            print(
                f"{name}: ndarray {obj.shape}"
            )

# ------------------------------------------------------------
# 5. Final test metrics
# ------------------------------------------------------------

print("\n[5] FINAL TEST METRICS")
print("-" * 90)

expected_test_metrics = {
    "Naive": {
        "RMSE": 0.779632,
        "MAE": 0.629916,
        "R2": 0.748860
    },
    "ElasticNet": {
        "RMSE": 0.9799
    },
    "Random Forest": {
        "RMSE": 0.9867
    },
    "HistGradientBoosting": {
        "RMSE": 1.3300
    }
}

print(
    "Expected frozen test results:"
)

for model, metrics in expected_test_metrics.items():

    print(
        f"{model}: {metrics}"
    )

# ------------------------------------------------------------
# 6. Search for test-result tables
# ------------------------------------------------------------

print("\n[6] TEST RESULT TABLES")
print("-" * 90)

test_tables = []

for name, obj in global_items:

    if isinstance(obj, pd.DataFrame):

        cols_lower = [
            str(c).lower()
            for c in obj.columns
        ]

        has_test = any(
            "test" in c
            for c in cols_lower
        )

        has_metric = any(
            metric in c
            for c in cols_lower
            for metric in [
                "rmse",
                "mae",
                "r2"
            ]
        )

        if has_test or has_metric:

            test_tables.append(name)

            print(
                f"{name}: shape={obj.shape}"
            )
            print(
                "Columns:",
                list(obj.columns)
            )

# ------------------------------------------------------------
# 7. Check whether test dates appear in CV
# ------------------------------------------------------------

print("\n[7] TEST DATES INSIDE CV")
print("-" * 90)

cv = safe_cv_results_df.copy()

test_start = pd.Timestamp("2024-05-01")
test_end = pd.Timestamp("2026-03-01")

cv_test_hits = 0

for col in cv.columns:

    col_lower = str(col).lower()

    if (
        "date" in col_lower
        or "month" in col_lower
    ):

        converted = pd.to_datetime(
            cv[col],
            errors="coerce"
        )

        if converted.notna().any():

            in_test = (
                (converted >= test_start)
                &
                (converted <= test_end)
            )

            cv_test_hits += int(
                in_test.sum()
            )

print(
    "Explicit test-period dates inside CV:",
    cv_test_hits
)

# ------------------------------------------------------------
# 8. Check final feature set consistency
# ------------------------------------------------------------

print("\n[8] FINAL FEATURE SET CONSISTENCY")
print("-" * 90)

final_features = list(X_safe_ar.columns)

print(
    "Feature count:",
    len(final_features)
)

print(
    "CPI_Current present:",
    "CPI_Current" in final_features
)

print(
    "CPI_Lag1 present:",
    "CPI_Lag1" in final_features
)

# ------------------------------------------------------------
# 9. Final preliminary freeze verdict
# ------------------------------------------------------------

print("\n[9] PRELIMINARY MODEL-FREEZE VERDICT")
print("-" * 90)

print(
    "Expected ElasticNet alpha = 0.30"
)

print(
    "Expected ElasticNet l1_ratio = 0.90"
)

if elastic_objects:

    alpha_match = all(
        np.isclose(
            float(obj.alpha),
            0.30
        )
        for _, obj in elastic_objects
    )

    l1_match = all(
        np.isclose(
            float(obj.l1_ratio),
            0.90
        )
        for _, obj in elastic_objects
    )

    print(
        "Observed ElasticNet alpha matches:",
        alpha_match
    )

    print(
        "Observed ElasticNet l1_ratio matches:",
        l1_match
    )

else:

    print(
        "Final ElasticNet object not automatically identified."
    )

print(
    "CV contains explicit test dates:",
    cv_test_hits > 0
)

print(
    "Final feature set excludes CPI_Current:",
    "CPI_Current" not in final_features
)

print("\n" + "=" * 90)
print("CELL 7 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 7: MODEL-FREEZE + TEST-PRESERVATION AUDIT

[1] FROZEN FINAL MODEL SPECIFICATION
------------------------------------------------------------------------------------------
Primary benchmark: Naive (CPI_Lag1)
Primary ML model: ElasticNet
Frozen alpha: 0.3
Frozen l1_ratio: 0.9
Secondary models: Random Forest, HistGradientBoosting

[2] FINAL MODEL OBJECTS
------------------------------------------------------------------------------------------
final_elasticnet: type=Pipeline
models: type=dict

[3] ELASTICNET PARAMETER OBJECTS
------------------------------------------------------------------------------------------
elastic_model: alpha=0.3, l1_ratio=0.9

[4] TEST-RELATED OBJECTS
------------------------------------------------------------------------------------------
test: DataFrame (23, 16)
best_test: Series (7,)
X_test: DataFrame (23, 14)
y_test: Series (23,)
X_ar_test: DataFrame (23, 15)
y_ar_test: Series (23,)
X_safe_ar_test: DataFrame (23, 15)
y_safe_ar_test: Series 

In [64]:
# ============================================================
# TASK 6.0 — CELL 8A
# CORRECTED FINAL TEST-RESULT CONSISTENCY CONTINUATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 8A: CORRECTED TEST CONSISTENCY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Normalize model names for comparison
# ------------------------------------------------------------

print("\n[1] MODEL NAME NORMALIZATION")
print("-" * 90)

final_results = final_test_results.copy()
test_results = test_results_df.copy()

def normalize_model_name(name):
    name = str(name).strip()

    if name.startswith("Naive"):
        return "Naive"

    return name

final_results["Model_Normalized"] = (
    final_results["Model"].map(normalize_model_name)
)

test_results["Model_Normalized"] = (
    test_results["Model"].map(normalize_model_name)
)

print(
    final_results[
        ["Model", "Model_Normalized"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 2. Compare final_test_results and test_results_df
# ------------------------------------------------------------

print("\n[2] FINAL TEST TABLE CONSISTENCY")
print("-" * 90)

comparison_columns = [
    "Test_RMSE",
    "Test_MAE",
    "Test_R2"
]

final_cmp = (
    final_results
    .set_index("Model_Normalized")
    .sort_index()
)

test_cmp = (
    test_results
    .set_index("Model_Normalized")
    .sort_index()
)

same_model_set = (
    set(final_cmp.index)
    ==
    set(test_cmp.index)
)

metric_values_match = np.allclose(
    final_cmp[comparison_columns].astype(float).values,
    test_cmp[comparison_columns].astype(float).values,
    rtol=1e-10,
    atol=1e-10
)

metric_values_match_rounded = np.allclose(
    final_cmp[comparison_columns].astype(float).values,
    test_cmp[comparison_columns].astype(float).values,
    rtol=1e-6,
    atol=1e-6
)

print(
    "Same normalized model set:",
    same_model_set
)

print(
    "Exact metric equality:",
    metric_values_match
)

print(
    "Numerical metric equality within 1e-6:",
    metric_values_match_rounded
)

# ------------------------------------------------------------
# 3. Frozen expected values
# ------------------------------------------------------------

print("\n[3] FROZEN EXPECTED TEST VALUES")
print("-" * 90)

expected = {
    "Naive": 0.779632,
    "ElasticNet": 0.979900,
    "Random Forest": 0.986700,
    "HistGradientBoosting": 1.330000
}

observed = (
    final_results
    .set_index("Model_Normalized")["Test_RMSE"]
    .to_dict()
)

for model, expected_rmse in expected.items():

    observed_rmse = observed.get(model, np.nan)

    if pd.notna(observed_rmse):

        difference = (
            observed_rmse
            -
            expected_rmse
        )

        print(
            f"{model:<25} "
            f"Observed={observed_rmse:.6f} "
            f"Expected={expected_rmse:.6f} "
            f"Difference={difference:+.6f}"
        )

    else:

        print(
            f"{model:<25} NOT FOUND"
        )

expected_rmse_match = all(
    model in observed
    and
    np.isclose(
        observed[model],
        expected[model],
        atol=0.0001
    )
    for model in expected
)

print(
    "\nFrozen RMSE values match within 0.0001:",
    expected_rmse_match
)

# ------------------------------------------------------------
# 4. Mathematical RMSE ranking
# ------------------------------------------------------------

print("\n[4] TEST RMSE RANKING")
print("-" * 90)

calculated_ranking = (
    final_results
    .sort_values("Test_RMSE")
    .reset_index(drop=True)
)

calculated_ranking["Calculated_Rank"] = (
    np.arange(len(calculated_ranking)) + 1
)

print(
    calculated_ranking[
        [
            "Model",
            "Test_RMSE",
            "Calculated_Rank"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 5. Compare saved ranking
# ------------------------------------------------------------

print("\n[5] SAVED TEST RANKING")
print("-" * 90)

saved_ranking = test_ranking.copy()

saved_ranking["Model_Normalized"] = (
    saved_ranking["Model"].map(
        normalize_model_name
    )
)

saved_order = (
    saved_ranking
    .sort_values("Rank")["Model_Normalized"]
    .tolist()
)

calculated_order = (
    calculated_ranking["Model_Normalized"]
    .tolist()
)

ranking_match = (
    calculated_order == saved_order
)

print(
    saved_ranking[
        [
            "Model",
            "Test_RMSE",
            "Rank"
        ]
    ].to_string(index=False)
)

print(
    "\nSaved ranking matches RMSE ranking:",
    ranking_match
)

# ------------------------------------------------------------
# 6. Naive benchmark check
# ------------------------------------------------------------

print("\n[6] NAIVE BENCHMARK CHECK")
print("-" * 90)

naive_rmse = float(
    final_results.loc[
        final_results["Model_Normalized"] == "Naive",
        "Test_RMSE"
    ].iloc[0]
)

ml_results = final_results[
    final_results["Model_Normalized"] != "Naive"
]

ml_better = ml_results[
    ml_results["Test_RMSE"] < naive_rmse
]

print(
    "Naive (CPI_Lag1) Test RMSE:",
    naive_rmse
)

print(
    "ML models beating Naive:",
    ml_better["Model"].tolist()
)

print(
    "Naive is best by RMSE:",
    bool(
        final_results["Test_RMSE"].min()
        == naive_rmse
    )
)

# ------------------------------------------------------------
# 7. Check final ranking against expected ordering
# ------------------------------------------------------------

print("\n[7] EXPECTED FINAL ORDER")
print("-" * 90)

expected_order = [
    "Naive",
    "ElasticNet",
    "Random Forest",
    "HistGradientBoosting"
]

print(
    "Calculated order:",
    calculated_order
)

print(
    "Expected order:",
    expected_order
)

order_matches = (
    calculated_order == expected_order
)

print(
    "Order matches expected:",
    order_matches
)

# ------------------------------------------------------------
# 8. Pre-test selection consistency
# ------------------------------------------------------------

print("\n[8] PRE-TEST SELECTION ARTIFACT")
print("-" * 90)

print(
    pretest_results.to_string(index=False)
)

# ------------------------------------------------------------
# 9. FINAL VERDICT
# ------------------------------------------------------------

print("\n[9] FINAL TEST-RESULT CONSISTENCY VERDICT")
print("-" * 90)

checks = {

    "Final test table has 4 models":
        len(final_results) == 4,

    "Final and test tables have same models":
        same_model_set,

    "Final and test metrics numerically agree":
        metric_values_match_rounded,

    "Frozen RMSE values match":
        expected_rmse_match,

    "Saved ranking matches calculated ranking":
        ranking_match,

    "Naive has lowest test RMSE":
        naive_rmse
        ==
        final_results["Test_RMSE"].min(),

    "Expected final ranking preserved":
        order_matches
}

for name, result in checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

print("\n" + "=" * 90)
print("CELL 8A COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 8A: CORRECTED TEST CONSISTENCY

[1] MODEL NAME NORMALIZATION
------------------------------------------------------------------------------------------
               Model     Model_Normalized
    Naive (CPI_Lag1)                Naive
          ElasticNet           ElasticNet
       Random Forest        Random Forest
HistGradientBoosting HistGradientBoosting

[2] FINAL TEST TABLE CONSISTENCY
------------------------------------------------------------------------------------------
Same normalized model set: True
Exact metric equality: False
Numerical metric equality within 1e-6: True

[3] FROZEN EXPECTED TEST VALUES
------------------------------------------------------------------------------------------
Naive                     Observed=0.779632 Expected=0.779632 Difference=+0.000000
ElasticNet                Observed=0.979889 Expected=0.979900 Difference=-0.000011
Random Forest             Observed=0.986692 Expected=0.986700 Difference=-0.000008
HistGradientBoostin

In [65]:
# ============================================================
# TASK 6.0 — CELL 9
# SAVED ARTIFACT / FILE INTEGRITY AUDIT
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 9: SAVED ARTIFACT / FILE INTEGRITY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Identify important final in-memory artifacts
# ------------------------------------------------------------

print("\n[1] FINAL IN-MEMORY ARTIFACTS")
print("-" * 90)

required_objects = [
    "safe_ar_df",
    "X_safe_ar",
    "y_safe_ar",
    "X_safe_ar_test",
    "y_safe_ar_test",
    "safe_cv_results_df",
    "safe_cv_summary",
    "final_test_results",
    "test_ranking",
    "pretest_results",
    "final_elasticnet",
    "elastic_model"
]

artifact_status = {}

for name in required_objects:

    exists = name in globals()
    artifact_status[name] = exists

    if exists:

        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            description = f"DataFrame {obj.shape}"

        elif isinstance(obj, pd.Series):
            description = f"Series {obj.shape}"

        elif isinstance(obj, np.ndarray):
            description = f"ndarray {obj.shape}"

        else:
            description = f"type={type(obj).__name__}"

        print(
            f"{name:<30} -> PRESENT ({description})"
        )

    else:

        print(
            f"{name:<30} -> MISSING"
        )

# ------------------------------------------------------------
# 2. Inspect current working-directory files
# ------------------------------------------------------------

print("\n[2] CURRENT WORKING DIRECTORY")
print("-" * 90)

print(
    "Working directory:",
    os.getcwd()
)

files_here = []

for path in glob.glob("*"):

    if os.path.isfile(path):

        files_here.append(path)

if files_here:

    for path in sorted(files_here):

        size_kb = (
            os.path.getsize(path) / 1024
        )

        print(
            f"{path:<60} {size_kb:,.1f} KB"
        )

else:

    print("No files found in current directory.")

# ------------------------------------------------------------
# 3. Search common output locations
# ------------------------------------------------------------

print("\n[3] COMMON OUTPUT DIRECTORIES")
print("-" * 90)

output_dirs = [
    "/content",
    "/content/data",
    "/content/outputs",
    "/content/artifacts"
]

for directory in output_dirs:

    if os.path.exists(directory):

        try:

            entries = os.listdir(directory)

            print(
                f"{directory}: {len(entries)} entries"
            )

            for entry in sorted(entries)[:30]:

                path = os.path.join(
                    directory,
                    entry
                )

                if os.path.isfile(path):

                    print(
                        f"  {entry}"
                    )

        except Exception as e:

            print(
                f"{directory}: unable to inspect ({e})"
            )

    else:

        print(
            f"{directory}: does not exist"
        )

# ------------------------------------------------------------
# 4. Identify likely final artifact files
# ------------------------------------------------------------

print("\n[4] LIKELY FINAL ARTIFACT FILES")
print("-" * 90)

search_patterns = [
    "/content/*",
    "/content/**/*.csv",
    "/content/**/*.xlsx",
    "/content/**/*.pkl",
    "/content/**/*.joblib",
    "/content/**/*.json",
    "/content/**/*.parquet"
]

candidate_files = set()

for pattern in search_patterns:

    candidate_files.update(
        glob.glob(
            pattern,
            recursive=True
        )
    )

candidate_files = sorted(
    path
    for path in candidate_files
    if os.path.isfile(path)
)

if candidate_files:

    for path in candidate_files:

        filename = os.path.basename(path)

        filename_lower = filename.lower()

        if any(
            keyword in filename_lower
            for keyword in [
                "cpi",
                "model",
                "test",
                "result",
                "cv",
                "forecast",
                "v2",
                "artifact"
            ]
        ):

            size_kb = (
                os.path.getsize(path) / 1024
            )

            print(
                f"{path:<80} "
                f"{size_kb:,.1f} KB"
            )

else:

    print(
        "No candidate artifact files found."
    )

# ------------------------------------------------------------
# 5. Check final dataframe integrity before saving
# ------------------------------------------------------------

print("\n[5] FINAL DATAFRAME ARTIFACT INTEGRITY")
print("-" * 90)

checks = {}

checks["safe_ar_df exists"] = (
    isinstance(
        globals().get("safe_ar_df"),
        pd.DataFrame
    )
)

checks["X_safe_ar exists"] = (
    isinstance(
        globals().get("X_safe_ar"),
        pd.DataFrame
    )
)

checks["y_safe_ar exists"] = (
    isinstance(
        globals().get("y_safe_ar"),
        pd.Series
    )
)

checks["safe CV results exists"] = (
    isinstance(
        globals().get("safe_cv_results_df"),
        pd.DataFrame
    )
)

checks["final test results exists"] = (
    isinstance(
        globals().get("final_test_results"),
        pd.DataFrame
    )
)

checks["test ranking exists"] = (
    isinstance(
        globals().get("test_ranking"),
        pd.DataFrame
    )
)

checks["pre-test results exists"] = (
    isinstance(
        globals().get("pretest_results"),
        pd.DataFrame
    )
)

for name, result in checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

# ------------------------------------------------------------
# 6. Verify artifact dimensions
# ------------------------------------------------------------

print("\n[6] FINAL ARTIFACT DIMENSIONS")
print("-" * 90)

if isinstance(
    globals().get("safe_ar_df"),
    pd.DataFrame
):

    print(
        "safe_ar_df:",
        safe_ar_df.shape
    )

if isinstance(
    globals().get("X_safe_ar"),
    pd.DataFrame
):

    print(
        "X_safe_ar:",
        X_safe_ar.shape
    )

if isinstance(
    globals().get("y_safe_ar"),
    pd.Series
):

    print(
        "y_safe_ar:",
        y_safe_ar.shape
    )

if isinstance(
    globals().get("X_safe_ar_test"),
    pd.DataFrame
):

    print(
        "X_safe_ar_test:",
        X_safe_ar_test.shape
    )

if isinstance(
    globals().get("y_safe_ar_test"),
    pd.Series
):

    print(
        "y_safe_ar_test:",
        y_safe_ar_test.shape
    )

if isinstance(
    globals().get("safe_cv_results_df"),
    pd.DataFrame
):

    print(
        "safe_cv_results_df:",
        safe_cv_results_df.shape
    )

if isinstance(
    globals().get("final_test_results"),
    pd.DataFrame
):

    print(
        "final_test_results:",
        final_test_results.shape
    )

# ------------------------------------------------------------
# 7. Check final model object
# ------------------------------------------------------------

print("\n[7] FINAL MODEL ARTIFACT")
print("-" * 90)

if "final_elasticnet" in globals():

    model = final_elasticnet

    print(
        "final_elasticnet type:",
        type(model).__name__
    )

    if hasattr(model, "named_steps"):

        print(
            "Pipeline steps:",
            list(model.named_steps.keys())
        )

    print(
        "Final ElasticNet object present: PASS"
    )

else:

    print(
        "Final ElasticNet object present: FAIL"
    )

# ------------------------------------------------------------
# 8. Final artifact verdict
# ------------------------------------------------------------

print("\n[8] FINAL ARTIFACT INTEGRITY VERDICT")
print("-" * 90)

all_required_present = all(
    artifact_status.values()
)

print(
    "All required final in-memory artifacts present:",
    all_required_present
)

print(
    "Candidate artifact files found:",
    len(candidate_files) > 0
)

print(
    "NOTE: File existence is reported separately from "
    "in-memory artifact integrity."
)

print("\n" + "=" * 90)
print("CELL 9 COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 9: SAVED ARTIFACT / FILE INTEGRITY

[1] FINAL IN-MEMORY ARTIFACTS
------------------------------------------------------------------------------------------
safe_ar_df                     -> PRESENT (DataFrame (147, 17))
X_safe_ar                      -> PRESENT (DataFrame (147, 15))
y_safe_ar                      -> PRESENT (Series (147,))
X_safe_ar_test                 -> PRESENT (DataFrame (23, 15))
y_safe_ar_test                 -> PRESENT (Series (23,))
safe_cv_results_df             -> PRESENT (DataFrame (40, 7))
safe_cv_summary                -> PRESENT (DataFrame (4, 7))
final_test_results             -> PRESENT (DataFrame (4, 4))
test_ranking                   -> PRESENT (DataFrame (4, 5))
pretest_results                -> PRESENT (DataFrame (4, 3))
final_elasticnet               -> PRESENT (type=Pipeline)
elastic_model                  -> PRESENT (type=ElasticNet)

[2] CURRENT WORKING DIRECTORY
------------------------------------------------------------------

In [67]:
# ============================================================
# TASK 6.0 — CELL 10A
# SAVED DATASET PROVENANCE + CPI_LAG1 RECONSTRUCTION AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 10A: SAVED DATASET PROVENANCE AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Reload saved file
# ------------------------------------------------------------

path = "/content/FINAL_MODELING_DATASET_CPI_FORECASTING_V2.csv"

saved = pd.read_csv(path)

saved["forecast_origin_month"] = pd.to_datetime(
    saved["forecast_origin_month"]
)

print("\n[1] SAVED DATASET")
print("-" * 90)

print("Shape:", saved.shape)
print("Start:", saved["forecast_origin_month"].min())
print("End:", saved["forecast_origin_month"].max())
print("Columns:", list(saved.columns))

# ------------------------------------------------------------
# 2. Compare saved file with model_df
# ------------------------------------------------------------

print("\n[2] SAVED FILE vs model_df")
print("-" * 90)

model_df_obj = globals().get("model_df")

if isinstance(model_df_obj, pd.DataFrame):

    model_copy = model_df_obj.copy()

    model_copy["forecast_origin_month"] = pd.to_datetime(
        model_copy["forecast_origin_month"]
    )

    print("model_df shape:", model_copy.shape)

    same_columns = (
        list(saved.columns)
        ==
        list(model_copy.columns)
    )

    print(
        "Same column structure:",
        same_columns
    )

    if same_columns:

        saved_sorted = saved.reset_index(drop=True)
        model_sorted = model_copy.reset_index(drop=True)

        dates_match = (
            saved_sorted["forecast_origin_month"]
            ==
            model_sorted["forecast_origin_month"]
        ).all()

        print(
            "Dates identical:",
            bool(dates_match)
        )

        numeric_columns = [
            c for c in saved.columns
            if c != "forecast_origin_month"
        ]

        values_match = np.allclose(
            saved_sorted[numeric_columns]
            .astype(float)
            .values,

            model_sorted[numeric_columns]
            .astype(float)
            .values,

            equal_nan=True
        )

        print(
            "All numeric values identical:",
            bool(values_match)
        )

    else:

        print(
            "Cannot perform full value comparison because columns differ."
        )

else:

    print(
        "model_df not available."
    )

# ------------------------------------------------------------
# 3. Compare saved file with ar_df
# ------------------------------------------------------------

print("\n[3] SAVED FILE vs ar_df")
print("-" * 90)

ar_df_obj = globals().get("ar_df")

if isinstance(ar_df_obj, pd.DataFrame):

    ar_copy = ar_df_obj.copy()

    ar_copy["forecast_origin_month"] = pd.to_datetime(
        ar_copy["forecast_origin_month"]
    )

    print("ar_df shape:", ar_copy.shape)
    print("ar_df columns:", list(ar_copy.columns))

    print(
        "CPI_Lag1 in ar_df:",
        "CPI_Lag1" in ar_copy.columns
    )

    if len(saved) == len(ar_copy):

        common_columns = [
            c for c in saved.columns
            if c in ar_copy.columns
        ]

        print(
            "Common columns:",
            len(common_columns)
        )

        print(
            "Missing from saved but present in ar_df:",
            [
                c for c in ar_copy.columns
                if c not in saved.columns
            ]
        )

else:

    print(
        "ar_df not available."
    )

# ------------------------------------------------------------
# 4. Deterministically reconstruct CPI_Lag1
# ------------------------------------------------------------

print("\n[4] CPI_Lag1 RECONSTRUCTION FROM SAVED FILE")
print("-" * 90)

saved_target = pd.to_numeric(
    saved["cpi_combined_yoy_t_plus_1"],
    errors="coerce"
)

reconstructed_lag = saved_target.shift(1)

saved_with_lag = saved.copy()

saved_with_lag["CPI_Lag1_RECONSTRUCTED"] = (
    reconstructed_lag
)

print(
    "Reconstructed lag column created:",
    "CPI_Lag1_RECONSTRUCTED"
)

print(
    "First reconstructed lag:",
    saved_with_lag[
        "CPI_Lag1_RECONSTRUCTED"
    ].iloc[0]
)

print(
    "Non-missing reconstructed lag rows:",
    int(
        reconstructed_lag.notna().sum()
    )
)

# ------------------------------------------------------------
# 5. Compare reconstructed lag with in-memory safe_ar_df
# ------------------------------------------------------------

print("\n[5] RECONSTRUCTED LAG vs safe_ar_df")
print("-" * 90)

safe_obj = globals().get("safe_ar_df")

if isinstance(safe_obj, pd.DataFrame):

    safe_copy = safe_obj.copy()

    safe_copy["forecast_origin_month"] = pd.to_datetime(
        safe_copy["forecast_origin_month"]
    )

    print(
        "safe_ar_df shape:",
        safe_copy.shape
    )

    # The safe AR dataset starts one row later.
    saved_reconstruct = saved_with_lag[
        saved_with_lag["CPI_Lag1_RECONSTRUCTED"].notna()
    ].copy()

    safe_dates = safe_copy[
        "forecast_origin_month"
    ].reset_index(drop=True)

    reconstructed_dates = saved_reconstruct[
        "forecast_origin_month"
    ].reset_index(drop=True)

    dates_match = (
        safe_dates == reconstructed_dates
    ).all()

    print(
        "Safe AR dates match reconstructed dates:",
        bool(dates_match)
    )

    safe_lag = pd.to_numeric(
        safe_copy["CPI_Lag1"],
        errors="coerce"
    ).reset_index(drop=True)

    reconstructed_lag_valid = pd.to_numeric(
        saved_reconstruct[
            "CPI_Lag1_RECONSTRUCTED"
        ],
        errors="coerce"
    ).reset_index(drop=True)

    lag_match = np.allclose(
        safe_lag.values,
        reconstructed_lag_valid.values,
        equal_nan=True
    )

    print(
        "Reconstructed CPI_Lag1 matches safe_ar_df:",
        bool(lag_match)
    )

    if len(safe_copy) == len(saved_reconstruct):

        print(
            "Safe AR row count matches reconstructed:",
            True
        )

    else:

        print(
            "Safe AR row count matches reconstructed:",
            False
        )

else:

    print(
        "safe_ar_df unavailable."
    )

# ------------------------------------------------------------
# 6. Check whether saved file is actually pre-AR dataset
# ------------------------------------------------------------

print("\n[6] SAVED FILE CLASSIFICATION")
print("-" * 90)

classification_checks = {

    "Saved file has 148 rows":
        len(saved) == 148,

    "Saved file has no CPI_Lag1":
        "CPI_Lag1" not in saved.columns,

    "CPI_Lag1 can be reconstructed":
        reconstructed_lag.notna().sum() == 147,

    "Saved file has target":
        "cpi_combined_yoy_t_plus_1" in saved.columns,

    "Saved file has forecast origin":
        "forecast_origin_month" in saved.columns
}

for name, result in classification_checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

# ------------------------------------------------------------
# 7. Important audit conclusion
# ------------------------------------------------------------

print("\n[7] AUDIT CONCLUSION")
print("-" * 90)

print(
    "The saved CSV is NOT yet the final 15-feature "
    "publication-safe AR artifact."
)

print(
    "It appears to contain the 148-row pre-lag dataset."
)

print(
    "CPI_Lag1 is deterministically reconstructible "
    "from the target history."
)

print(
    "No file will be modified by this audit."
)

print("\n" + "=" * 90)
print("CELL 10A COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 10A: SAVED DATASET PROVENANCE AUDIT

[1] SAVED DATASET
------------------------------------------------------------------------------------------
Shape: (148, 16)
Start: 2013-12-01 00:00:00
End: 2026-03-01 00:00:00
Columns: ['forecast_origin_month', 'cpi_combined_yoy_t_plus_1', 'Petrol_Monthly_Avg_Rs_per_Litre', 'Diesel_Monthly_Avg_Rs_per_Litre', 'Petrol_MoM_Change_Pct', 'Diesel_MoM_Change_Pct', 'Brent_Monthly_Avg_USD_per_Barrel', 'Brent_Log_MoM', 'Rainfall_Deviation_LPA_Pct', 'Rainfall_Deviation_LPA_Pct_Lag1', 'Rainfall_Lag1_Available', 'WPI_All_Commodities_t_minus_1', 'WPI_MoM_Pct_t_minus_1', 'FAO_Food_Price_Index_t_minus_1', 'FAO_Food_Price_Index_MoM_t_minus_1', 'Inflation_Expectation_3M']

[2] SAVED FILE vs model_df
------------------------------------------------------------------------------------------
model_df shape: (148, 16)
Same column structure: True
Dates identical: True
All numeric values identical: False

[3] SAVED FILE vs ar_df
--------------------------

In [69]:
# ============================================================
# TASK 6.0 — CELL 10C
# AUTHORITATIVE FROZEN SPLIT VERIFICATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("TASK 6.0 — CELL 10C: AUTHORITATIVE FROZEN SPLIT VERIFICATION")
print("=" * 90)

# ------------------------------------------------------------
# 1. Reconstruct publication-safe dataset from saved CSV
# ------------------------------------------------------------

path = "/content/FINAL_MODELING_DATASET_CPI_FORECASTING_V2.csv"

saved = pd.read_csv(path)

saved["forecast_origin_month"] = pd.to_datetime(
    saved["forecast_origin_month"]
)

target_col = "cpi_combined_yoy_t_plus_1"

final_features = [
    "Petrol_Monthly_Avg_Rs_per_Litre",
    "Diesel_Monthly_Avg_Rs_per_Litre",
    "Petrol_MoM_Change_Pct",
    "Diesel_MoM_Change_Pct",
    "Brent_Monthly_Avg_USD_per_Barrel",
    "Brent_Log_MoM",
    "Rainfall_Deviation_LPA_Pct",
    "Rainfall_Deviation_LPA_Pct_Lag1",
    "Rainfall_Lag1_Available",
    "WPI_All_Commodities_t_minus_1",
    "WPI_MoM_Pct_t_minus_1",
    "FAO_Food_Price_Index_t_minus_1",
    "FAO_Food_Price_Index_MoM_t_minus_1",
    "Inflation_Expectation_3M",
    "CPI_Lag1"
]

saved["CPI_Lag1"] = (
    pd.to_numeric(
        saved[target_col],
        errors="coerce"
    ).shift(1)
)

reconstructed = saved[
    saved["CPI_Lag1"].notna()
].copy()

reconstructed = reconstructed.reset_index(
    drop=True
)

# ------------------------------------------------------------
# 2. Define AUTHORITATIVE frozen V2 split
# ------------------------------------------------------------

print("\n[1] AUTHORITATIVE FROZEN V2 SPLIT")
print("-" * 90)

split_spec = {
    "Train": (
        pd.Timestamp("2014-01-01"),
        pd.Timestamp("2022-06-01"),
        102
    ),

    "Validation": (
        pd.Timestamp("2022-07-01"),
        pd.Timestamp("2024-04-01"),
        22
    ),

    "Test": (
        pd.Timestamp("2024-05-01"),
        pd.Timestamp("2026-03-01"),
        23
    )
}

for name, (start, end, expected_rows) in split_spec.items():

    print(
        f"{name:<12}: "
        f"{start.date()} -> {end.date()} "
        f"({expected_rows} rows)"
    )

# ------------------------------------------------------------
# 3. Build authoritative reconstructed splits
# ------------------------------------------------------------

print("\n[2] RECONSTRUCTED FROZEN SPLITS")
print("-" * 90)

reconstructed_splits = {}

for name, (start, end, expected_rows) in split_spec.items():

    mask = (
        (reconstructed["forecast_origin_month"] >= start)
        &
        (reconstructed["forecast_origin_month"] <= end)
    )

    split = (
        reconstructed.loc[mask]
        .copy()
        .reset_index(drop=True)
    )

    reconstructed_splits[name] = split

    print(
        f"{name:<12}: "
        f"rows={len(split)}, "
        f"start={split['forecast_origin_month'].min()}, "
        f"end={split['forecast_origin_month'].max()}"
    )

# ------------------------------------------------------------
# 4. Build equivalent splits directly from safe_ar_df
# ------------------------------------------------------------

print("\n[3] NOTEBOOK SAFE_AR_DF FROZEN SPLITS")
print("-" * 90)

safe_source = safe_ar_df.copy()

safe_source["forecast_origin_month"] = pd.to_datetime(
    safe_source["forecast_origin_month"]
)

safe_splits = {}

for name, (start, end, expected_rows) in split_spec.items():

    mask = (
        (safe_source["forecast_origin_month"] >= start)
        &
        (safe_source["forecast_origin_month"] <= end)
    )

    split = (
        safe_source.loc[mask]
        .copy()
        .reset_index(drop=True)
    )

    safe_splits[name] = split

    print(
        f"{name:<12}: "
        f"rows={len(split)}, "
        f"start={split['forecast_origin_month'].min()}, "
        f"end={split['forecast_origin_month'].max()}"
    )

# ------------------------------------------------------------
# 5. Compare reconstructed vs safe_ar_df splits
# ------------------------------------------------------------

print("\n[4] RECONSTRUCTED vs SAFE_AR_DF SPLIT EQUIVALENCE")
print("-" * 90)

comparison_columns = [
    "forecast_origin_month",
    target_col
] + final_features

split_results = {}

for name in split_spec.keys():

    reconstructed_split = (
        reconstructed_splits[name][comparison_columns]
        .reset_index(drop=True)
    )

    safe_split = (
        safe_splits[name][comparison_columns]
        .reset_index(drop=True)
    )

    shape_match = (
        reconstructed_split.shape
        ==
        safe_split.shape
    )

    dates_match = (
        reconstructed_split[
            "forecast_origin_month"
        ].tolist()
        ==
        safe_split[
            "forecast_origin_month"
        ].tolist()
    )

    numeric_columns = [
        c for c in comparison_columns
        if c != "forecast_origin_month"
    ]

    values_match = np.allclose(
        reconstructed_split[numeric_columns]
        .astype(float)
        .values,

        safe_split[numeric_columns]
        .astype(float)
        .values,

        equal_nan=True
    )

    split_results[name] = (
        shape_match
        and dates_match
        and values_match
    )

    print(
        f"{name:<12}: "
        f"shape={shape_match}, "
        f"dates={dates_match}, "
        f"values={values_match}"
    )

# ------------------------------------------------------------
# 6. Check authoritative split counts
# ------------------------------------------------------------

print("\n[5] FROZEN SPLIT COUNT CHECK")
print("-" * 90)

count_results = {}

for name, (start, end, expected_rows) in split_spec.items():

    actual_rows = len(
        reconstructed_splits[name]
    )

    result = (
        actual_rows == expected_rows
    )

    count_results[name] = result

    print(
        f"{name:<12}: "
        f"actual={actual_rows}, "
        f"expected={expected_rows}, "
        f"{'PASS' if result else 'FAIL'}"
    )

# ------------------------------------------------------------
# 7. Chronological boundaries
# ------------------------------------------------------------

print("\n[6] CHRONOLOGICAL SPLIT BOUNDARIES")
print("-" * 90)

train_dates = reconstructed_splits["Train"][
    "forecast_origin_month"
]

val_dates = reconstructed_splits["Validation"][
    "forecast_origin_month"
]

test_dates = reconstructed_splits["Test"][
    "forecast_origin_month"
]

chronology_checks = {

    "Train before Validation":
        train_dates.max() < val_dates.min(),

    "Validation before Test":
        val_dates.max() < test_dates.min(),

    "Train/Validation no overlap":
        len(
            set(train_dates)
            &
            set(val_dates)
        ) == 0,

    "Train/Test no overlap":
        len(
            set(train_dates)
            &
            set(test_dates)
        ) == 0,

    "Validation/Test no overlap":
        len(
            set(val_dates)
            &
            set(test_dates)
        ) == 0
}

for name, result in chronology_checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

# ------------------------------------------------------------
# 8. Explain stale train object
# ------------------------------------------------------------

print("\n[7] LEGACY TRAIN OBJECT DIAGNOSTIC")
print("-" * 90)

if "train" in globals():

    legacy_train = train.copy()

    legacy_train["forecast_origin_month"] = pd.to_datetime(
        legacy_train["forecast_origin_month"]
    )

    print(
        "Legacy train rows:",
        len(legacy_train)
    )

    print(
        "Legacy train start:",
        legacy_train["forecast_origin_month"].min()
    )

    print(
        "Legacy train end:",
        legacy_train["forecast_origin_month"].max()
    )

    print(
        "Authoritative V2 train rows:",
        102
    )

    print(
        "Legacy train includes pre-lag 2013-12 row:",
        (
            legacy_train["forecast_origin_month"].min()
            == pd.Timestamp("2013-12-01")
        )
    )

# ------------------------------------------------------------
# 9. FINAL REPRODUCIBILITY VERDICT
# ------------------------------------------------------------

print("\n[8] FINAL REPRODUCIBILITY VERDICT")
print("-" * 90)

final_checks = {

    "Reconstructed safe AR dataset = 147 rows":
        len(reconstructed) == 147,

    "Reconstructed safe AR dataset = 15 features":
        len(final_features) == 15,

    "Train count = 102":
        count_results["Train"],

    "Validation count = 22":
        count_results["Validation"],

    "Test count = 23":
        count_results["Test"],

    "Train reconstruction matches safe_ar_df":
        split_results["Train"],

    "Validation reconstruction matches safe_ar_df":
        split_results["Validation"],

    "Test reconstruction matches safe_ar_df":
        split_results["Test"],

    "Chronological ordering valid":
        all(chronology_checks.values())
}

for name, result in final_checks.items():

    print(
        f"{name}: {'PASS' if result else 'FAIL'}"
    )

print("\n" + "=" * 90)
print("CELL 10C COMPLETE — DO NOT MODIFY ANYTHING")
print("=" * 90)

TASK 6.0 — CELL 10C: AUTHORITATIVE FROZEN SPLIT VERIFICATION

[1] AUTHORITATIVE FROZEN V2 SPLIT
------------------------------------------------------------------------------------------
Train       : 2014-01-01 -> 2022-06-01 (102 rows)
Validation  : 2022-07-01 -> 2024-04-01 (22 rows)
Test        : 2024-05-01 -> 2026-03-01 (23 rows)

[2] RECONSTRUCTED FROZEN SPLITS
------------------------------------------------------------------------------------------
Train       : rows=102, start=2014-01-01 00:00:00, end=2022-06-01 00:00:00
Validation  : rows=22, start=2022-07-01 00:00:00, end=2024-04-01 00:00:00
Test        : rows=23, start=2024-05-01 00:00:00, end=2026-03-01 00:00:00

[3] NOTEBOOK SAFE_AR_DF FROZEN SPLITS
------------------------------------------------------------------------------------------
Train       : rows=102, start=2014-01-01 00:00:00, end=2022-06-01 00:00:00
Validation  : rows=22, start=2022-07-01 00:00:00, end=2024-04-01 00:00:00
Test        : rows=23, start=2024-05-01